# TimeLapse Processing

Phase-plane shift estimation applied to real migrated timelapse data.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal         import hilbert as sp_hilbert
from scipy.signal.windows import tukey
 
# FDTD cell size: all forward simulations use 1 mm cells.
# Displacements not on a mm boundary are rounded by gprMax to the nearest cell.
fdtd_cell_m = 0.001
def fdtd_true(nom):
    """Round nominal displacement(s) to nearest FDTD cell (1 mm)."""
    return np.round(np.asarray(nom, dtype=float) / fdtd_cell_m) * fdtd_cell_m

In [ ]:
# ── Standard figure auto-save (protocol: .wiki/FIGURES_PROTOCOL.md) ─────────
import sys, pathlib
if str(pathlib.Path.cwd()) not in sys.path:
    sys.path.insert(0, str(pathlib.Path.cwd()))
from helper_functions.figures import setup_autosave
setup_autosave(study="TimeLapse_Processing", prefix="TLP_")


In [ ]:

def estimate_shift_2d(base, mon, dz_g, dx_g, kz_cent, kz_pos_only=False, force_dz_zero=False):
    """
    Fit a 2D phase plane phi(kz,kx) = kz*dz + kx*dx + phi_0 to the cross-spectrum.
    Returns (dz_est, dx_est, phi_0, XS, kz_ax, kx_ax).

    kz_pos_only=True   — restrict WLS to KZ > 0 (use with analytic-signal inputs).
    force_dz_zero=True — fit only (dx, phi_0); KZ column dropped and dz_est returned
                         as 0.0. Use when vertical movement is known to be absent:
                         removes ill-conditioning from narrowband wavelet where phi_0
                         and kz*dz are nearly indistinguishable on the one-sided spectrum.
    """
    Nz, Nx = base.shape

    kz_ax = np.fft.fftfreq(Nz, d=dz_g) * 2 * np.pi
    kx_ax = np.fft.fftfreq(Nx, d=dx_g) * 2 * np.pi
    KZ, KX = np.meshgrid(kz_ax, kx_ax, indexing='ij')

    taper = np.outer(tukey(Nz, alpha=0.15), tukey(Nx, alpha=0.15))

    base_fft = np.fft.fft2(base * taper)
    mon_fft  = np.fft.fft2(mon  * taper)
    XS       = base_fft * np.conj(mon_fft)

    w   = np.abs(XS)
    phi = np.angle(XS)

    band = (np.abs(KZ) < 1.4 * kz_cent) & (np.abs(KX) < 1.4 * kz_cent)
    mask = (w > 0.10 * w.max()) & band & ((np.abs(KZ) + np.abs(KX)) > 0)
    if kz_pos_only:
        mask &= (KZ > 0)

    W = w[mask]
    if force_dz_zero:
        # 2-parameter fit: phi = kx*dx + phi_0  (dz = 0 by physics)
        A = np.column_stack([KX[mask], np.ones(mask.sum())])
        c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
        return 0.0, c[0], c[1], XS, kz_ax, kx_ax
    else:
        # 3-parameter fit: phi = kz*dz + kx*dx + phi_0
        A = np.column_stack([KZ[mask], KX[mask], np.ones(mask.sum())])
        c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
        return c[0], c[1], c[2], XS, kz_ax, kx_ax


# Section 0 - Load & Plot Raw Results

In [ ]:
STUDY_ROOT = Path(r'C:\Users\Administrator\OneDrive\Thesis\TimeLapse_Notebooks\timelapse_study')

d = np.load(str(STUDY_ROOT / 'static_results.npz'), allow_pickle=False)

data_static       = d['data_static']        # (8, n_t, n_traces)
scenarios         = list(d['scenarios'])    # ['Baseline', '2lambda', ...]
separation_lambda = d['separation_lambda']  # [0, 2, 1, 0.5, 0.25, 0.125, 0.0625, 0.03125]
x_traces          = d['x_traces']
time_ns           = d['time_ns']
dt                = float(d['dt'])
x_s1              = d['x_s1']
x_s2              = float(d['x_s2'])
z_scatterer       = float(d['z_scatterer'])
z_top             = float(d['z_top'])

print(f"Loaded: {STUDY_ROOT / 'static_results.npz'}")
print(f"  data_static : {data_static.shape}")
print(f"  scenarios   : {scenarios}")
print(f"\nTime axis:  dt={dt*1e9:.4f} ns,  t_max={time_ns[-1]:.2f} ns")

# ── Plot raw (non-migrated) B-scans for all scenarios ─────────────────────────
extent_bscan = [x_traces[0], x_traces[-1], time_ns[-1], 0]
n_scen       = len(scenarios)   # 8

fig, axes = plt.subplots(2, 4, figsize=(24, 9),
                         sharex=True, sharey=True, facecolor='w')
axes_flat = axes.flatten()

for i, (ax, lbl) in enumerate(zip(axes_flat, scenarios)):
    img  = data_static[i]
    vmax = np.nanpercentile(np.abs(img), 100)
    ax.imshow(img, aspect='auto', extent=extent_bscan,
              cmap='seismic', vmin=-vmax, vmax=vmax, interpolation='nearest')
    ax.axvline(x_s1[i], color='green', lw=0.8, ls='--', alpha=0.8)
    ax.axvline(x_s2,    color='green', lw=0.8, ls='--', alpha=0.8)
    sep   = separation_lambda[i]
    title = lbl if sep == 0 else f"{lbl}  ({sep}λ)"
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('x [m]', fontsize=8)
    if i % 4 == 0:
        ax.set_ylabel('Time [ns]', fontsize=8)

fig.suptitle('Raw (Background-Subtracted) B-Scans — All Scenarios', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 1 — Load & Plot Migrated Results

In [ ]:
STUDY_ROOT = Path(r'C:\Users\Administrator\OneDrive\Thesis\TimeLapse_Notebooks\timelapse_study')

d = np.load(str(STUDY_ROOT / 'migrated_results.npz'), allow_pickle=False)

kirchhoff         = d['kirchhoff']           # (7, n_z, n_x)
gazdag            = d['gazdag']              # (7, n_z, n_x)
backprop          = d['backprop']            # (7, n_z, n_x)
scenarios         = list(d['scenarios'])     # ['Baseline', '2lambda', ...]
separation_lambda = d['separation_lambda']   # [0, 2, 1, 0.5, 0.25, 0.125, 0.0625]
x_traces          = d['x_traces']
z_img             = d['z_img']
x_s1              = d['x_s1']
x_s2              = float(d['x_s2'])
z_scatterer       = float(d['z_scatterer'])
z_top             = float(d['z_top'])

# Physics parameters
v_ice = 0.168    # m/ns
f_c   = 1.5      # GHz
lam   = v_ice / f_c

dz_mig = float(z_img[1]    - z_img[0])
dx_mig = float(x_traces[1] - x_traces[0])
kz_c   = 2 * np.pi / lam

print(f"Loaded: {STUDY_ROOT / 'migrated_results.npz'}")
print(f"  kirchhoff : {kirchhoff.shape}")
print(f"  gazdag    : {gazdag.shape}")
print(f"  backprop  : {backprop.shape}")
print(f"  scenarios : {scenarios}")
print(f"\nGrid spacing:  dz={dz_mig*1e3:.2f} mm,  dx={dx_mig*1e3:.2f} mm")
print(f"lambda = {lam*1e3:.1f} mm,  kz_c = {kz_c:.2f} rad/m")

In [ ]:
extent_mig = [x_traces[0], x_traces[-1], z_img[-1], z_img[0]]
n_scen     = len(scenarios)   # 8

method_data = [
    ('Kirchhoff', kirchhoff),
    ('Gazdag',    gazdag),
    ('Back-prop', backprop),
]

for method_name, data in method_data:
    fig, axes = plt.subplots(2, 4, figsize=(24, 9),
                             sharex=True, sharey=True, facecolor='w')
    axes_flat = axes.flatten()

    for i, (ax, lbl) in enumerate(zip(axes_flat, scenarios)):
        img  = data[i]
        vmax = np.nanpercentile(np.abs(img), 100)
        ax.imshow(img, aspect='auto', extent=extent_mig,
                  cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='upper')
        ax.axhline(z_top,        color='k',      lw=0.8, ls='--', alpha=0.6)
        ax.axhline(z_scatterer,  color='tomato',  lw=0.8, ls='--', alpha=0.6)
        ax.axvline(x_s1[i],      color='tomato',  lw=0.8, ls=':',  alpha=0.8)
        sep = separation_lambda[i]
        title = lbl if sep == 0 else f"{lbl}  ({sep}λ)"
        ax.set_title(title, fontsize=9)
        ax.set_xlabel('x [m]', fontsize=8)
        if i % 4 == 0:
            ax.set_ylabel('z [m]', fontsize=8)


    fig.suptitle(f'{method_name} Migration — All Scenarios', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## Section 2 — Phase Plane Shift Estimation

Apply `estimate_shift_2d` to each of the 7 timelapse scenarios (Monitor) vs the Baseline,
for **Kirchhoff**, **Gazdag**, and **Back-prop** migrated images.

Before calling `estimate_shift_2d`, crop each image pair to a ±2.5λ window around the
data-found scatterer apex (located from the Hilbert-envelope difference peak, then refined
on the baseline envelope). This removes the large zero-shift background that otherwise
biases the phase-plane fit toward half the true displacement.

In [ ]:
for method_name, base_stack in [('Kirchhoff', kirchhoff), ('Gazdag', gazdag), ('Back-prop', backprop)]:

    base_img = base_stack[0]
    # Baseline envelope computed once per method
    env_base = np.abs(sp_hilbert(np.nan_to_num(base_img), axis=0))

    mon_cases = [
        (scenarios[i], 0.0, separation_lambda[i] * lam, base_stack[i])
        for i in range(1, len(scenarios))
    ]

    n_cases = len(mon_cases)
    fig, axes = plt.subplots(n_cases, 4, figsize=(20, 3.5 * n_cases))
    fig.suptitle(
        f"Phase-plane shift estimation — {method_name}  |  Baseline vs each scenario",
        fontsize=12, fontweight='bold', y=1.01
    )

    for row, (name, true_dz, true_dx, mon_img) in enumerate(mon_cases):

        # ── Locate scatterer apex from difference envelope ─────────────────────
        env_mon  = np.abs(sp_hilbert(np.nan_to_num(mon_img), axis=0))
        diff_env = np.abs(env_mon - env_base)

        # Rough location: peak of difference envelope
        iz_d, ix_d = np.unravel_index(np.argmax(diff_env), diff_env.shape)
        x_rough    = x_traces[ix_d]

        # Refine: baseline envelope apex within ±3λ of rough location
        hw_search = 3 * lam
        ix_lo_s   = np.searchsorted(x_traces, x_rough - hw_search)
        ix_hi_s   = np.searchsorted(x_traces, x_rough + hw_search)
        _, ix_loc = np.unravel_index(
            np.argmax(env_base[:, ix_lo_s:ix_hi_s]),
            env_base[:, ix_lo_s:ix_hi_s].shape
        )
        x_apex = x_traces[ix_lo_s + ix_loc]

        # ── Crop ±2.5λ around apex ────────────────────────────────────────────
        x_crop_hw = 2.5 * lam
        ix_lo     = np.searchsorted(x_traces, x_apex - x_crop_hw)
        ix_hi     = np.searchsorted(x_traces, x_apex + x_crop_hw)
        x_crop    = x_traces[ix_lo:ix_hi]

        base_crop = base_img[:, ix_lo:ix_hi]
        mon_crop  = mon_img[:, ix_lo:ix_hi]

        # ── Estimate shift (real images, Hermitian XS) ────────────────────────
        dz_est, dx_est, _, XS, kz_ax, kx_ax = estimate_shift_2d(
            base_crop, mon_crop, dz_mig, dx_mig, kz_c
        )

        XS_s     = np.fft.fftshift(XS)
        kz_s     = np.fft.fftshift(kz_ax)
        kx_s     = np.fft.fftshift(kx_ax)
        energy   = np.abs(XS_s)
        phi_show = np.where(energy > 0.05 * energy.max(),
                            np.degrees(np.angle(XS_s)), np.nan)

        klim        = 1.5 * kz_c
        extent_crop = [x_crop[0], x_crop[-1], z_img[-1], z_img[0]]

        # Col 0: difference image over cropped region
        ax0  = axes[row, 0]
        diff = mon_crop - base_crop
        vmax = np.nanmax(np.abs(diff))
        ax0.imshow(diff, aspect='auto', extent=extent_crop,
                   cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='upper')
        ax0.axvline(x_apex, color='k', lw=1.2, ls='-', label='apex (data)')
        ax0.set_title(
            f"{name}  (dx_true={true_dx*1e3:.1f} mm)\n"
            f"Difference (mon − base)  |  apex @ x={x_apex*100:.1f} cm",
            fontsize=9
        )
        ax0.set_xlabel('x [m]'); ax0.set_ylabel('z [m]')

        # Col 1: cross-spectrum phase
        ax1 = axes[row, 1]
        im1 = ax1.pcolormesh(kx_s, kz_s, phi_show,
                             cmap='RdBu_r', vmin=-180, vmax=180, shading='auto')
        for sgn in [-1, 1]:
            ax1.axhline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.5)
            ax1.axvline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.5)
        ax1.set_title("Cross-spectrum phase [deg]\ndashed = fit band", fontsize=9)
        ax1.set_xlabel('kx [rad/m]'); ax1.set_ylabel('kz [rad/m]')
        ax1.set_xlim(-klim * 2.5, klim * 2.5)
        ax1.set_ylim(-klim * 2.5, klim * 2.5)
        plt.colorbar(im1, ax=ax1, fraction=0.046)

        # Col 2: cross-spectrum energy |XS|
        ax2 = axes[row, 2]
        im2 = ax2.pcolormesh(kx_s, kz_s, energy,
                             cmap='inferno', shading='auto')
        for sgn in [-1, 1]:
            ax2.axhline(sgn * klim, color='w', lw=0.8, ls='--', alpha=0.5)
            ax2.axvline(sgn * klim, color='w', lw=0.8, ls='--', alpha=0.5)
        ax2.set_title("Cross-spectrum energy |XS|\ndashed = fit band", fontsize=9)
        ax2.set_xlabel('kx [rad/m]'); ax2.set_ylabel('kz [rad/m]')
        ax2.set_xlim(-klim * 2.5, klim * 2.5)
        ax2.set_ylim(-klim * 2.5, klim * 2.5)
        plt.colorbar(im2, ax=ax2, fraction=0.046)

        # Col 3: numerical summary
        ax3 = axes[row, 3]
        ax3.axis('off')
        txt = (
            f"True:  dz = {true_dz  * 1e3:+7.3f} mm\n"
            f"       dx = {true_dx  * 1e3:+7.3f} mm\n\n"
            f"Est:   dz = {dz_est   * 1e3:+7.3f} mm\n"
            f"       dx = {dx_est   * 1e3:+7.3f} mm\n\n"
            f"Err:   dz = {(dz_est - true_dz) * 1e3:+.4f} mm\n"
            f"       dx = {(dx_est - true_dx) * 1e3:+.4f} mm\n\n"
            f"Apex @ x = {x_apex*100:.2f} cm"
        )
        ax3.text(0.05, 0.92, txt, transform=ax3.transAxes,
                 fontsize=10, va='top', family='monospace',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.tight_layout()
    plt.show()


## Section 2b — NoisyTimeLapse: Load Migrated Results

Load the migrated (non-difference) images for the horizontal-shift study with
Laplace-distributed noise added to each B-scan before migration. Only **Kirchhoff**
and **Gazdag** are available for the noisy dataset (no back-propagation was run).

`estimate_shift_2d_cleaning` (defined below) replaces the magnitude-weighted L2
phase-plane fit used in Section 2 for this noisy case. Benchmarking against the
real noisy data (see conversation/dev notes) ruled out several alternatives before
landing on this design:

- **Median filtering + Huber regression + cross-spectrum smoothing** (the first attempt)
  was no better than plain least squares, and the cross-spectrum boxcar smoothing
  actively *hurt* — the crop is narrow enough that smoothing blends genuinely
  different phase-plane slopes across adjacent wavenumber bins rather than just
  cancelling noise.
- **RANSAC plane fitting** was a clear win for ¼λ–1λ shifts but didn't fix the
  fundamental phase-plane fragility at smaller shifts.
- **Generalized cross-correlation (GCC)** — correlate base/monitor directly in the
  space domain via `real(ifft2(XS))` and locate the peak, instead of fitting a
  plane to phase — turned out to be the most accurate for shifts ≳¼λ. (Note:
  numpy's `ifft` kernel sign puts the peak at `lag = -true_shift` relative to the
  phase-plane convention used elsewhere in this notebook; the lag axis here is
  pre-negated to correct for that — verified analytically and against a synthetic
  shifted signal.)
- A **Hilbert-envelope coarse-to-fine peak search** (find the envelope peak broadly,
  then refine with parabolic sub-sample interpolation on the raw correlation nearby)
  was tried to stop the peak from locking onto a wavelength-spaced sidelobe under
  noise — it didn't rescue the smallest shifts (⅛λ and below), because at that
  point the *global* correlation surface has no reliably-strongest coherent lobe at
  all, not just a "right lobe, wrong cycle" problem.
- **Widening the crop window** pulled small shifts into a more plausible range, but
  consistently broke the ¼λ–½λ cases (the wider crop pulls in the static reference
  scatterer and other non-moving structure, adding spurious correlation). No single
  crop width works for both regimes.

Given that, the final design uses GCC as the primary estimator and falls back to
the original `estimate_shift_2d` phase-plane fit when the GCC peak looks unreliable.
The trigger is a peak-to-sidelobe ratio computed from the correlation envelope
(`PSR < 8`, calibrated on this dataset) rather than the output magnitude — magnitude
alone could *not* separate good and bad cases, since a failed small-shift estimate
can land on the same order of magnitude as a genuine large-shift one. The fallback
does **not** recover small-shift accuracy; it only avoids GCC's worst failure mode
(errors of multiple wavelengths) in exchange for the lstsq fallback's own smaller,
but still substantial, small-shift error.

**Update — the real bottleneck wasn't the estimator, it was the crop.** Benchmarking
`estimate_shift_2d_cleaning` against every scenario revealed that ⅛λ, ¹⁄₁₆λ and ¹⁄₃₂λ
were catastrophically wrong (errors of tens of mm) no matter which estimator or
B-scan-domain cleaning was used. The actual cause: the apex-finding step (locating
the scatterer from the per-scenario base-vs-monitor difference envelope, `diff_env`)
was picking a location over a metre away from the true scatterer for those three
scenarios — every downstream method was being handed a crop centred on noise, not
on the scatterer. Things that did **not** fix this:

- Using the baseline's own envelope peak (`env_base` alone, ignoring the difference)
  instead of `diff_env` — this reliably locks onto the *static reference* scatterer
  (`x_s2`) instead, which reflects more strongly than the moving scatterer of interest.
- A normalised/relative difference score (`diff_env / env_base`) — amplifies noise
  wherever `env_base` is small, made the previously-good scenarios fail too.
- Gaussian-smoothing `diff_env` before the peak search, or masking out the location
  of the dominant (likely static) reflector — both partially helped some scenarios
  while breaking others; no single setting fixed all three.

**What worked:** stack `diff_env` across *all 7* monitor scenarios (median, not mean,
for robustness) before taking the argmax. The genuine signal — the moving scatterer —
produces a difference bump near the *same* location across every scenario, while
noise-driven false peaks are scenario-specific and don't reinforce across the stack.
This isn't a use of ground truth: it only uses the fact that all 7 monitor scans are
already available when processing a time-lapse dataset. One shared apex location is
now computed per migration method (Kirchhoff/Gazdag) and reused for every scenario's
crop, instead of redoing the fragile per-scenario localization seven times.

In [ ]:
STUDY_ROOT_NOISY = Path(r'C:\Users\Administrator\OneDrive\Thesis\TimeLapse_Notebooks\timelapse_study')

dn = np.load(str(STUDY_ROOT_NOISY / 'migrated_results_noisy.npz'), allow_pickle=False)

noisy_kirchhoff         = dn['kirchhoff']            # (8, n_z, n_x)
# 'gazdag' is occasionally absent (e.g. file overwritten mid-resave by a
# concurrent kernel session) — tolerate that rather than hard-failing, since
# Section 2c/6c already exclude noisy Gazdag from analysis regardless.
noisy_gazdag            = dn['gazdag'] if 'gazdag' in dn.files else None
noisy_scenarios         = list(dn['scenarios'])      # ['Baseline', '2λ', ...]
noisy_separation_lambda = dn['separation_lambda']    # [0, 2, 1, 0.5, 0.25, 0.125, 0.0625, 0.03125]
noisy_x_traces          = dn['x_traces']
noisy_z_img             = dn['z_img']
noisy_x_s1              = dn['x_s1']
noisy_x_s2              = float(dn['x_s2'])
noisy_z_scatterer       = float(dn['z_scatterer'])
noisy_z_top             = float(dn['z_top'])
noisy_noise_level       = float(dn['noise_level'])

# Grid spacing and wavenumber (same ice physics as TimeLapse)
noisy_dz_mig = float(noisy_z_img[1]    - noisy_z_img[0])
noisy_dx_mig = float(noisy_x_traces[1] - noisy_x_traces[0])
noisy_kz_c   = kz_c   # 2π / lam_ice — identical migration grid

print(f"Loaded: {STUDY_ROOT_NOISY / 'migrated_results_noisy.npz'}")
print(f"  kirchhoff   : {noisy_kirchhoff.shape}")
print(f"  gazdag      : {noisy_gazdag.shape if noisy_gazdag is not None else 'MISSING from file'}")
print(f"  scenarios   : {noisy_scenarios}")
print(f"  noise_level : {noisy_noise_level}")
print(f"\nGrid spacing:  dz={noisy_dz_mig*1e3:.2f} mm,  dx={noisy_dx_mig*1e3:.2f} mm")


In [ ]:
# ─── Robust phase-plane estimator for Laplace-noise-corrupted B-scans ────────
from scipy.signal import medfilt2d
from scipy.signal.windows import tukey


def estimate_shift_2d_cleaning(base, mon, dz_g, dx_g, kz_cent,
                                medfilt_size=3, coarse_search_lam=3.0,
                                fine_search_lam=0.5, psr_threshold=8.0):
    """
    Robust shift estimator for Laplace-noise-corrupted B-scans.

    Primary method: generalized cross-correlation (GCC). Correlate the
    (median-filtered, tapered) base/monitor crops directly in the space
    domain via R = real(ifft2(XS)), then locate the shift as the peak of R
    instead of fitting a plane to phase. This was empirically the most
    accurate approach for shifts >~ 1/4 of the carrier wavelength (see
    Section 2b markdown for the comparison against Huber/RANSAC/etc.).

    Peak search is two-stage to resist locking onto a wavelength-spaced
    sidelobe under noise: a coarse location from the Hilbert envelope of R
    (suppresses the carrier-frequency ripple) within ±coarse_search_lam*lam,
    then parabolic sub-sample refinement on the raw R within
    ±fine_search_lam*lam of that coarse location.

    NOTE: numpy's ifft kernel sign puts the raw correlation peak at
    lag = -true_shift relative to the phase-plane convention used by
    estimate_shift_2d; the lag axes below are pre-negated to correct for
    this (verified analytically + against a synthetic shifted signal).

    Fallback: GCC is unreliable for shifts below ~1/4 wavelength (the
    correlation surface has no reliably-strongest coherent peak once the
    true shift is comparable to the noise floor). Output magnitude alone
    cannot detect this — a failed small-shift estimate can coincidentally
    match the magnitude of a genuine large-shift one. Instead this uses a
    peak-to-sidelobe ratio (PSR) of the envelope: PSR = (peak - background
    mean) / background std, computed over the coarse search window. Good
    estimates (this dataset) have PSR ~10; failed ones have PSR ~4.5-6.
    Below `psr_threshold`, falls back to the original estimate_shift_2d
    phase-plane fit — this does NOT recover small-shift accuracy, it only
    avoids GCC's worst failure mode (errors of multiple wavelengths).
    """
    Nz, Nx = base.shape
    lam_val = 2 * np.pi / kz_cent

    base_clean = medfilt2d(base.astype(np.float64), kernel_size=medfilt_size)
    mon_clean  = medfilt2d(mon.astype(np.float64),  kernel_size=medfilt_size)

    taper = np.outer(tukey(Nz, alpha=0.15), tukey(Nx, alpha=0.15))
    base_fft = np.fft.fft2(base_clean * taper)
    mon_fft  = np.fft.fft2(mon_clean  * taper)
    XS       = base_fft * np.conj(mon_fft)

    kz_ax = np.fft.fftfreq(Nz, d=dz_g) * 2 * np.pi
    kx_ax = np.fft.fftfreq(Nx, d=dx_g) * 2 * np.pi

    # ── GCC peak search (sign-corrected; see docstring) ────────────────────
    R       = np.real(np.fft.ifft2(XS))
    R_shift = np.fft.fftshift(R)
    lag_z = -(np.arange(Nz) - Nz // 2) * dz_g
    lag_x = -(np.arange(Nx) - Nx // 2) * dx_g
    oz, ox  = np.argsort(lag_z), np.argsort(lag_x)
    lag_z, lag_x = lag_z[oz], lag_x[ox]
    R_shift = R_shift[np.ix_(oz, ox)]

    env   = np.abs(sp_hilbert(R_shift, axis=1))
    hw_c  = coarse_search_lam * lam_val
    zmask_c = np.abs(lag_z) <= hw_c
    xmask_c = np.abs(lag_x) <= hw_c
    sub_env = env[np.ix_(zmask_c, xmask_c)]
    iz_c, ix_c   = np.unravel_index(np.argmax(sub_env), sub_env.shape)
    lag_z_c, lag_x_c = lag_z[zmask_c][iz_c], lag_x[xmask_c][ix_c]
    peak_env = sub_env[iz_c, ix_c]

    background = np.delete(sub_env.ravel(), np.argmax(sub_env.ravel()))
    psr = (peak_env - background.mean()) / (background.std() + 1e-12)

    hw_f  = fine_search_lam * lam_val
    zmask_f = np.abs(lag_z - lag_z_c) <= hw_f
    xmask_f = np.abs(lag_x - lag_x_c) <= hw_f
    sub_R   = R_shift[np.ix_(zmask_f, xmask_f)]
    lag_z_f, lag_x_f = lag_z[zmask_f], lag_x[xmask_f]
    iz0, ix0 = np.unravel_index(np.argmax(np.abs(sub_R)), sub_R.shape)

    def _parabolic(vals, idx, axis_vals):
        if idx <= 0 or idx >= len(axis_vals) - 1:
            return axis_vals[idx]
        y0, y1, y2 = vals[idx - 1], vals[idx], vals[idx + 1]
        denom = y0 - 2 * y1 + y2
        if denom == 0:
            return axis_vals[idx]
        delta = 0.5 * (y0 - y2) / denom
        d = axis_vals[1] - axis_vals[0]
        return axis_vals[idx] + delta * d

    dz_gcc = _parabolic(sub_R[:, ix0], iz0, lag_z_f)
    dx_gcc = _parabolic(sub_R[iz0, :], ix0, lag_x_f)

    # ── PSR-gated fallback to the phase-plane lstsq fit ────────────────────
    if psr < psr_threshold:
        dz_est, dx_est, phi_0, _, _, _ = estimate_shift_2d(base, mon, dz_g, dx_g, kz_cent)
        return dz_est, dx_est, phi_0, XS, kz_ax, kx_ax

    return dz_gcc, dx_gcc, 0.0, XS, kz_ax, kx_ax

## Section 2c — NoisyTimeLapse: Phase-Plane Shift Estimation

Apply `estimate_shift_2d_cleaning` (GCC peak search with a phase-plane lstsq
fallback gated on peak-to-sidelobe ratio — see Section 2b) to the noisy migrated
images. The crop is centred on a single apex location per method, found from the
*median-stacked* difference envelope across all 7 monitor scenarios (see Section 2b)
rather than per-scenario — the per-scenario version was the actual cause of every
small-shift failure, not the estimator. The crop window is now ±2.5λ in **both**
x and z around that shared apex (previously only x was cropped, leaving the full
0–0.8 m depth range to dilute the GCC/lstsq fit).

**Kirchhoff: low error.** With the x+z crop, error stays small across all 7
scenarios — the shared-apex localization correctly lands on the moving scatterer
(diff_env_consensus peak at x≈1.65 m, near x_s1), so cropping tightly around it
sharpens the fit.

**Gazdag: large error — confirmed root cause.** Re-enabling Gazdag here (after
re-running its noisy migration) still gives large errors, and the cause is now
understood, not just suspected: Gazdag's noisy migrated images carry **vertical
streaking artifacts concentrated at shallow depth**, with amplitude up to ~60×
the genuine scatterer signal (peak amplitude ≈1.0 near x≈3.04 m, z≈0.02 m, vs.
≈0.016 at the true scatterer x≈1.66 m, z≈0.676 m — visually confirmed by clipping
the colour scale to 5% of peak, which reveals the real hyperbola underneath the
streaks). This is consistent with FK/phase-shift migration's known sensitivity to
noise near small vertical wavenumber (kz≈0): the code already zeros fully
evanescent bins, but near-evanescent low-kz noise still gets strongly amplified
on the way through the phase-shift operator.

This explains every earlier observation:
- **Present in every scenario** (it's a structural property of the operator), but
  **amplitude varies scenario-to-scenario** (≈0.20–0.33 at that column across the
  7 monitor scenarios) — depends on how much noise energy each independent random
  draw happens to have in those problematic low-kz components.
- **Median-stacking across scenarios doesn't cancel it**, because it's systematic
  (same operator weak spot every time), not independent random noise.
- **Survives the z ≥ z_top restriction**: confirmed directly — restricting the
  search to z ≥ z_top still lands on x≈3.03–3.04 m, just at the *bottom* of the
  allowed window (z=0.8 m) instead of the top (z=0.02 m). The streaks run through
  most of the depth column, not just near-surface, so cropping depth alone can't
  separate them from the real signal.

**Left active deliberately.** Gazdag is kept in the loop (rather than commented
out) so its degraded numbers stay visible here as a record of the issue. A real
fix means damping/regularizing the phase-shift operator near kz≈0 (alongside the
existing evanescent-zeroing) in `helper_functions/migration.py`'s
`gazdag_migration`, then re-running the noisy migration in
`TimeLapse_Playground.ipynb` — that's out of scope for this notebook, which only
consumes the already-migrated images.


In [ ]:
for method_name, base_stack in [
        ('Kirchhoff', noisy_kirchhoff),
        ('Gazdag', noisy_gazdag),  # disabled — structural migration artifact, see Section 2b/2c markdown
        ('Back-prop', backprop)
]:

    base_img = base_stack[0]
    # Baseline envelope computed once per method
    env_base = np.abs(sp_hilbert(np.nan_to_num(base_img), axis=0))

    mon_cases = [
        (noisy_scenarios[i], 0.0, noisy_separation_lambda[i] * lam, base_stack[i])
        for i in range(1, len(noisy_scenarios))
    ]

    # ── Shared apex: median-stacked difference envelope across ALL scenarios ──
    # The genuine signal (moving scatterer) bumps diff_env at the same location
    # in every scenario; noise-driven false peaks are scenario-specific and don't
    # reinforce across the stack. Per-scenario diff_env argmax (the old approach)
    # was the actual cause of every small-shift failure — see Section 2b.
    diff_env_stack = np.stack([
        np.abs(np.abs(sp_hilbert(np.nan_to_num(mon_img), axis=0)) - env_base)
        for _, _, _, mon_img in mon_cases
    ], axis=0)
    diff_env_consensus = np.median(diff_env_stack, axis=0)

    iz_d, ix_d = np.unravel_index(np.argmax(diff_env_consensus), diff_env_consensus.shape)
    x_rough = noisy_x_traces[ix_d]

    hw_search       = 3 * lam
    ix_lo_s         = np.searchsorted(noisy_x_traces, x_rough - hw_search)
    ix_hi_s         = np.searchsorted(noisy_x_traces, x_rough + hw_search)
    iz_loc, ix_loc  = np.unravel_index(
        np.argmax(env_base[:, ix_lo_s:ix_hi_s]),
        env_base[:, ix_lo_s:ix_hi_s].shape
    )
    x_apex = noisy_x_traces[ix_lo_s + ix_loc]
    z_apex = noisy_z_img[iz_loc]

    # ── Crop ±2.5λ around the shared apex in BOTH x and z — same window for
    # every scenario (previously only x was cropped; z_img kept its full
    # 0-0.8 m range, which dilutes the GCC/lstsq fit with irrelevant depth).
    x_crop_hw = 2.5 * lam
    z_crop_hw = 2.5 * lam
    ix_lo  = np.searchsorted(noisy_x_traces, x_apex - x_crop_hw)
    ix_hi  = np.searchsorted(noisy_x_traces, x_apex + x_crop_hw)
    iz_lo  = np.searchsorted(noisy_z_img,    z_apex - z_crop_hw)
    iz_hi  = np.searchsorted(noisy_z_img,    z_apex + z_crop_hw)
    x_crop = noisy_x_traces[ix_lo:ix_hi]
    z_crop = noisy_z_img[iz_lo:iz_hi]
    base_crop = base_img[iz_lo:iz_hi, ix_lo:ix_hi]

    n_cases = len(mon_cases)
    fig, axes = plt.subplots(n_cases, 4, figsize=(20, 3.5 * n_cases))
    fig.suptitle(
        f"NoisyTimeLapse — Robust shift estimation (GCC + lstsq fallback) — {method_name}  |  Baseline vs each scenario"
        f"  (noise_level={noisy_noise_level}, shared apex @ x={x_apex*100:.1f} cm, z={z_apex*100:.1f} cm)",
        fontsize=12, fontweight='bold', y=1.01
    )

    for row, (name, true_dz, true_dx, mon_img) in enumerate(mon_cases):

        mon_crop = mon_img[iz_lo:iz_hi, ix_lo:ix_hi]

        # ── Estimate shift (real images, Hermitian XS) ────────────────────────
        dz_est, dx_est, _, XS, kz_ax, kx_ax = estimate_shift_2d_cleaning(
            base_crop, mon_crop, noisy_dz_mig, noisy_dx_mig, noisy_kz_c
        )

        XS_s     = np.fft.fftshift(XS)
        kz_s     = np.fft.fftshift(kz_ax)
        kx_s     = np.fft.fftshift(kx_ax)
        energy   = np.abs(XS_s)
        phi_show = np.where(energy > 0.05 * energy.max(),
                            np.degrees(np.angle(XS_s)), np.nan)

        klim        = 1.5 * noisy_kz_c
        extent_crop = [x_crop[0], x_crop[-1], z_crop[-1], z_crop[0]]

        # Col 0: difference image over cropped region
        ax0  = axes[row, 0]
        diff = mon_crop - base_crop
        vmax = np.nanmax(np.abs(diff))
        ax0.imshow(diff, aspect='auto', extent=extent_crop,
                   cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='upper')
        ax0.axvline(x_apex, color='k', lw=1.2, ls='-', label='shared apex')
        ax0.set_title(
            f"{name}  (dx_true={true_dx*1e3:.1f} mm)\n"
            f"Difference (mon − base)  |  shared apex @ x={x_apex*100:.1f} cm, z={z_apex*100:.1f} cm",
            fontsize=9
        )
        ax0.set_xlabel('x [m]'); ax0.set_ylabel('z [m]')

        # Col 1: cross-spectrum phase
        ax1 = axes[row, 1]
        im1 = ax1.pcolormesh(kx_s, kz_s, phi_show,
                             cmap='RdBu_r', vmin=-180, vmax=180, shading='auto')
        for sgn in [-1, 1]:
            ax1.axhline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.5)
            ax1.axvline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.5)
        ax1.set_title("Cross-spectrum phase [deg]\ndashed = fit band", fontsize=9)
        ax1.set_xlabel('kx [rad/m]'); ax1.set_ylabel('kz [rad/m]')
        ax1.set_xlim(-klim * 2.5, klim * 2.5)
        ax1.set_ylim(-klim * 2.5, klim * 2.5)
        plt.colorbar(im1, ax=ax1, fraction=0.046)

        # Col 2: cross-spectrum energy |XS|
        ax2 = axes[row, 2]
        im2 = ax2.pcolormesh(kx_s, kz_s, energy,
                             cmap='inferno', shading='auto')
        for sgn in [-1, 1]:
            ax2.axhline(sgn * klim, color='w', lw=0.8, ls='--', alpha=0.5)
            ax2.axvline(sgn * klim, color='w', lw=0.8, ls='--', alpha=0.5)
        ax2.set_title("Cross-spectrum energy |XS|\ndashed = fit band", fontsize=9)
        ax2.set_xlabel('kx [rad/m]'); ax2.set_ylabel('kz [rad/m]')
        ax2.set_xlim(-klim * 2.5, klim * 2.5)
        ax2.set_ylim(-klim * 2.5, klim * 2.5)
        plt.colorbar(im2, ax=ax2, fraction=0.046)

        # Col 3: numerical summary
        ax3 = axes[row, 3]
        ax3.axis('off')
        txt = (
            f"True:  dz = {true_dz  * 1e3:+7.3f} mm\n"
            f"       dx = {true_dx  * 1e3:+7.3f} mm\n\n"
            f"Est:   dz = {dz_est   * 1e3:+7.3f} mm\n"
            f"       dx = {dx_est   * 1e3:+7.3f} mm\n\n"
            f"Err:   dz = {(dz_est - true_dz) * 1e3:+.4f} mm\n"
            f"       dx = {(dx_est - true_dx) * 1e3:+.4f} mm\n\n"
            f"Shared apex @ x = {x_apex*100:.2f} cm\n"
            f"               z = {z_apex*100:.2f} cm"
        )
        ax3.text(0.05, 0.92, txt, transform=ax3.transAxes,
                 fontsize=10, va='top', family='monospace',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.tight_layout()
    plt.show()

## Section 2d — VerticalTimeLapse: Load Migrated Results

Load the migrated (non-difference) images for the vertical scatterer-displacement study —
the scatterer is shifted in depth (z) rather than laterally (x).

In [ ]:
VT_ROOT = Path(r'C:\Users\Administrator\OneDrive\Thesis\TimeLapse_Notebooks\vertical_timelapse_study')

vt = np.load(str(VT_ROOT / 'vertical_migrated_results.npz'), allow_pickle=False)

vt_kirchhoff      = vt['kirchhoff']           # (7, n_z, n_x)
vt_gazdag         = vt['gazdag']              # (7, n_z, n_x)
vt_backprop       = vt['backprop']            # (7, n_z, n_x)
vt_scenarios      = list(vt['scenarios'])     # ['Baseline', '1λ', ...]
vt_shift_lambda   = vt['shift_lambda']        # [0, 1, 0.5, 0.25, 0.125, 0.0625, 0.03125]
vt_y_scatterers   = vt['y_scatterers']
vt_z_depths       = vt['z_depths']
vt_x_traces       = vt['x_traces']
vt_z_img          = vt['z_img']
vt_x_scatterer    = float(vt['x_scatterer'])
vt_y_baseline     = float(vt['y_baseline'])
vt_z_scatterer    = float(vt['z_scatterer'])
vt_z_top          = float(vt['z_top'])

# Grid spacing and wavenumber (same ice physics as TimeLapse)
vt_dz_mig = float(vt_z_img[1]    - vt_z_img[0])
vt_dx_mig = float(vt_x_traces[1] - vt_x_traces[0])
vt_kz_c   = kz_c   # 2π / lam_ice — identical migration grid

print(f"Loaded: {VT_ROOT / 'vertical_migrated_results.npz'}")
print(f"  kirchhoff : {vt_kirchhoff.shape}")
print(f"  gazdag    : {vt_gazdag.shape}")
print(f"  backprop  : {vt_backprop.shape}")
print(f"  scenarios : {vt_scenarios}")
print(f"Grid spacing:  dz={vt_dz_mig*1e3:.2f} mm,  dx={vt_dx_mig*1e3:.2f} mm")

## Section 2e — VerticalTimeLapse: Phase-Plane Shift Estimation

Apply `estimate_shift_2d` to each of the 6 vertical-shift scenarios (Monitor) vs the Baseline,
for **Kirchhoff**, **Gazdag**, and **Back-prop** migrated images. Here the scatterer moves only
in depth (Δz = shift_lambda·λ, Δx = 0) — the mirror image of Section 2's lateral-shift case.

Same procedure as Section 2: locate the apex from the Hilbert-envelope difference peak, refine
on the baseline envelope, then crop ±2.5λ in x around the apex before the phase-plane fit.

In [ ]:
for method_name, base_stack in [('Kirchhoff', vt_kirchhoff), ('Gazdag', vt_gazdag), ('Back-prop', vt_backprop)]:

    base_img = base_stack[0]
    # Baseline envelope computed once per method
    env_base = np.abs(sp_hilbert(np.nan_to_num(base_img), axis=0))

    mon_cases = [
        (vt_scenarios[i], vt_shift_lambda[i] * lam, 0.0, base_stack[i])
        for i in range(1, len(vt_scenarios))
    ]

    n_cases = len(mon_cases)
    fig, axes = plt.subplots(n_cases, 4, figsize=(20, 3.5 * n_cases))
    fig.suptitle(
        f"VerticalTimeLapse — Phase-plane shift estimation — {method_name}  |  Baseline vs each scenario",
        fontsize=12, fontweight='bold', y=1.01
    )

    for row, (name, true_dz, true_dx, mon_img) in enumerate(mon_cases):

        # ── Locate scatterer apex from difference envelope ─────────────────────
        env_mon  = np.abs(sp_hilbert(np.nan_to_num(mon_img), axis=0))
        diff_env = np.abs(env_mon - env_base)

        # Rough location: peak of difference envelope
        iz_d, ix_d = np.unravel_index(np.argmax(diff_env), diff_env.shape)
        x_rough    = vt_x_traces[ix_d]

        # Refine: baseline envelope apex within ±3λ of rough location
        hw_search = 3 * lam
        ix_lo_s   = np.searchsorted(vt_x_traces, x_rough - hw_search)
        ix_hi_s   = np.searchsorted(vt_x_traces, x_rough + hw_search)
        _, ix_loc = np.unravel_index(
            np.argmax(env_base[:, ix_lo_s:ix_hi_s]),
            env_base[:, ix_lo_s:ix_hi_s].shape
        )
        x_apex = vt_x_traces[ix_lo_s + ix_loc]

        # ── Crop ±2.5λ around apex ────────────────────────────────────────────
        x_crop_hw = 2.5 * lam
        ix_lo     = np.searchsorted(vt_x_traces, x_apex - x_crop_hw)
        ix_hi     = np.searchsorted(vt_x_traces, x_apex + x_crop_hw)
        x_crop    = vt_x_traces[ix_lo:ix_hi]

        base_crop = base_img[:, ix_lo:ix_hi]
        mon_crop  = mon_img[:, ix_lo:ix_hi]

        # ── Estimate shift (real images, Hermitian XS) ────────────────────────
        dz_est, dx_est, _, XS, kz_ax, kx_ax = estimate_shift_2d(
            base_crop, mon_crop, vt_dz_mig, vt_dx_mig, vt_kz_c
        )

        XS_s     = np.fft.fftshift(XS)
        kz_s     = np.fft.fftshift(kz_ax)
        kx_s     = np.fft.fftshift(kx_ax)
        energy   = np.abs(XS_s)
        phi_show = np.where(energy > 0.05 * energy.max(),
                            np.degrees(np.angle(XS_s)), np.nan)

        klim        = 1.5 * vt_kz_c
        extent_crop = [x_crop[0], x_crop[-1], vt_z_img[-1], vt_z_img[0]]

        # Col 0: difference image over cropped region
        ax0  = axes[row, 0]
        diff = mon_crop - base_crop
        vmax = np.nanmax(np.abs(diff))
        ax0.imshow(diff, aspect='auto', extent=extent_crop,
                   cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='upper')
        ax0.axvline(x_apex, color='k', lw=1.2, ls='-', label='apex (data)')
        ax0.set_title(
            f"{name}  (dz_true={true_dz*1e3:.1f} mm)\n"
            f"Difference (mon − base)  |  apex @ x={x_apex*100:.1f} cm",
            fontsize=9
        )
        ax0.set_xlabel('x [m]'); ax0.set_ylabel('z [m]')

        # Col 1: cross-spectrum phase
        ax1 = axes[row, 1]
        im1 = ax1.pcolormesh(kx_s, kz_s, phi_show,
                             cmap='RdBu_r', vmin=-180, vmax=180, shading='auto')
        for sgn in [-1, 1]:
            ax1.axhline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.5)
            ax1.axvline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.5)
        ax1.set_title("Cross-spectrum phase [deg]\ndashed = fit band", fontsize=9)
        ax1.set_xlabel('kx [rad/m]'); ax1.set_ylabel('kz [rad/m]')
        ax1.set_xlim(-klim * 2.5, klim * 2.5)
        ax1.set_ylim(-klim * 2.5, klim * 2.5)
        plt.colorbar(im1, ax=ax1, fraction=0.046)

        # Col 2: cross-spectrum energy |XS|
        ax2 = axes[row, 2]
        im2 = ax2.pcolormesh(kx_s, kz_s, energy,
                             cmap='inferno', shading='auto')
        for sgn in [-1, 1]:
            ax2.axhline(sgn * klim, color='w', lw=0.8, ls='--', alpha=0.5)
            ax2.axvline(sgn * klim, color='w', lw=0.8, ls='--', alpha=0.5)
        ax2.set_title("Cross-spectrum energy |XS|\ndashed = fit band", fontsize=9)
        ax2.set_xlabel('kx [rad/m]'); ax2.set_ylabel('kz [rad/m]')
        ax2.set_xlim(-klim * 2.5, klim * 2.5)
        ax2.set_ylim(-klim * 2.5, klim * 2.5)
        plt.colorbar(im2, ax=ax2, fraction=0.046)

        # Col 3: numerical summary
        ax3 = axes[row, 3]
        ax3.axis('off')
        txt = (
            f"True:  dz = {true_dz  * 1e3:+7.3f} mm\n"
            f"       dx = {true_dx  * 1e3:+7.3f} mm\n\n"
            f"Est:   dz = {dz_est   * 1e3:+7.3f} mm\n"
            f"       dx = {dx_est   * 1e3:+7.3f} mm\n\n"
            f"Err:   dz = {(dz_est - true_dz) * 1e3:+.4f} mm\n"
            f"       dx = {(dx_est - true_dx) * 1e3:+.4f} mm\n\n"
            f"Apex @ x = {x_apex*100:.2f} cm"
        )
        ax3.text(0.05, 0.92, txt, transform=ax3.transAxes,
                 fontsize=10, va='top', family='monospace',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.tight_layout()
    plt.show()

## Section 2f — DiagonalTimeLapse: Load Migrated Results

Load the migrated (non-difference) images for the diagonal scatterer-displacement study
(`Diagonal_TimeLapse_Playground.ipynb`) — the scatterer moves laterally **and** vertically
at once (Δx to the right, Δz downward, in a fixed 2:1 ratio across all 5 shift scenarios).

`diagonal_migrated_results.npz` does not store `x_traces`/`z_img` (unlike the
Vertical/Lateral/FluidFlow result files) — the survey/migration grid is identical
to the other studies (same `domain_x`, `n_traces`, `trace_step`, `rx_offset`), only the
depth grid spans 0–0.9 m over 180 samples (matching VerticalTimeLapse's grid) instead of
TimeLapse's 0–0.8 m / 160 samples, so it is recomputed here rather than reloaded.


In [ ]:
DG_ROOT = Path(r'C:\Users\Administrator\OneDrive\Thesis\TimeLapse_Notebooks\diagonal_timelapse_study')

dg = np.load(str(DG_ROOT / 'diagonal_migrated_results.npz'), allow_pickle=False)

dg_kirchhoff      = dg['kirchhoff']           # (6, n_z, n_x)
dg_gazdag         = dg['gazdag']              # (6, n_z, n_x)
dg_backprop       = dg['backprop']            # (6, n_z, n_x)
dg_scenarios      = list(dg['scenarios'])     # ['Baseline', 'Scenario 1 (1λx, ½λy)', ...]
dg_shift_lambda_x = dg['shift_lambda_x']      # [0, 1, 0.5, 0.25, 0.125, 0.0625]
dg_shift_lambda_y = dg['shift_lambda_y']      # [0, 0.5, 0.25, 0.125, 0.0625, 0.03125]
dg_x_scatterers   = dg['x_scatterers']
dg_y_scatterers   = dg['y_scatterers']
dg_z_depths       = dg['z_depths']
dg_x_baseline     = float(dg['x_baseline'])
dg_y_baseline     = float(dg['y_baseline'])
dg_z_scatterer    = float(dg['z_scatterer'])
dg_z_top          = float(dg['z_top'])

# Survey/migration grid — same physics as Section 1/2d, but Diagonal used the
# VerticalTimeLapse depth range (0-0.9 m, 180 samples), not TimeLapse's (0-0.8 m, 160).
dg_domain_x   = 4.0
dg_n_traces   = 380
dg_trace_step = 0.01
dg_rx_offset  = 0.1
dg_x_traces   = (0.1 + dg_rx_offset / 2) + np.arange(dg_n_traces) * dg_trace_step
dg_z_img      = np.linspace(0.0, 0.9, 180)

dg_dz_mig = float(dg_z_img[1]    - dg_z_img[0])
dg_dx_mig = float(dg_x_traces[1] - dg_x_traces[0])
dg_kz_c   = kz_c   # 2π / lam_ice — identical migration grid

print(f"Loaded: {DG_ROOT / 'diagonal_migrated_results.npz'}")
print(f"  kirchhoff : {dg_kirchhoff.shape}")
print(f"  gazdag    : {dg_gazdag.shape}")
print(f"  backprop  : {dg_backprop.shape}")
print(f"  scenarios : {dg_scenarios}")
print(f"Grid spacing:  dz={dg_dz_mig*1e3:.2f} mm,  dx={dg_dx_mig*1e3:.2f} mm")


## Section 2g — DiagonalTimeLapse: Phase-Plane Shift Estimation

Apply `estimate_shift_2d` to each of the 5 diagonal-shift scenarios (Monitor) vs the
Baseline, for **Kirchhoff**, **Gazdag**, and **Back-prop** migrated images. Unlike
Section 2 (Δx only) or Section 2e (Δz only), here both true shifts are non-zero
(Δx = shift_lambda_x·λ, Δz = shift_lambda_y·λ) — `estimate_shift_2d` already fits a
full 2-parameter (kz, kx) phase plane, so this is the most direct test of the
combined fit.

Same procedure as Sections 2/2e: locate the apex from the Hilbert-envelope
difference peak, refine on the baseline envelope, then crop ±2.5λ in x around the
apex before the phase-plane fit.


In [ ]:
for method_name, base_stack in [('Kirchhoff', dg_kirchhoff), ('Gazdag', dg_gazdag), ('Back-prop', dg_backprop)]:

    base_img = base_stack[0]
    # Baseline envelope computed once per method
    env_base = np.abs(sp_hilbert(np.nan_to_num(base_img), axis=0))

    mon_cases = [
        (dg_scenarios[i], dg_shift_lambda_y[i] * lam, dg_shift_lambda_x[i] * lam, base_stack[i])
        for i in range(1, len(dg_scenarios))
    ]

    n_cases = len(mon_cases)
    fig, axes = plt.subplots(n_cases, 4, figsize=(20, 3.5 * n_cases))
    fig.suptitle(
        f"DiagonalTimeLapse — Phase-plane shift estimation — {method_name}  |  Baseline vs each scenario",
        fontsize=12, fontweight='bold', y=1.01
    )

    for row, (name, true_dz, true_dx, mon_img) in enumerate(mon_cases):

        # ── Locate scatterer apex from difference envelope ─────────────────────
        env_mon  = np.abs(sp_hilbert(np.nan_to_num(mon_img), axis=0))
        diff_env = np.abs(env_mon - env_base)

        # Rough location: peak of difference envelope
        iz_d, ix_d = np.unravel_index(np.argmax(diff_env), diff_env.shape)
        x_rough    = dg_x_traces[ix_d]

        # Refine: baseline envelope apex within ±3λ of rough location
        hw_search = 3 * lam
        ix_lo_s   = np.searchsorted(dg_x_traces, x_rough - hw_search)
        ix_hi_s   = np.searchsorted(dg_x_traces, x_rough + hw_search)
        _, ix_loc = np.unravel_index(
            np.argmax(env_base[:, ix_lo_s:ix_hi_s]),
            env_base[:, ix_lo_s:ix_hi_s].shape
        )
        x_apex = dg_x_traces[ix_lo_s + ix_loc]

        # ── Crop ±2.5λ around apex ────────────────────────────────────────────
        x_crop_hw = 2.5 * lam
        ix_lo     = np.searchsorted(dg_x_traces, x_apex - x_crop_hw)
        ix_hi     = np.searchsorted(dg_x_traces, x_apex + x_crop_hw)
        x_crop    = dg_x_traces[ix_lo:ix_hi]

        base_crop = base_img[:, ix_lo:ix_hi]
        mon_crop  = mon_img[:, ix_lo:ix_hi]

        # ── Estimate shift (real images, Hermitian XS) ────────────────────────
        dz_est, dx_est, _, XS, kz_ax, kx_ax = estimate_shift_2d(
            base_crop, mon_crop, dg_dz_mig, dg_dx_mig, dg_kz_c
        )

        XS_s     = np.fft.fftshift(XS)
        kz_s     = np.fft.fftshift(kz_ax)
        kx_s     = np.fft.fftshift(kx_ax)
        energy   = np.abs(XS_s)
        phi_show = np.where(energy > 0.05 * energy.max(),
                            np.degrees(np.angle(XS_s)), np.nan)

        klim        = 1.5 * dg_kz_c
        extent_crop = [x_crop[0], x_crop[-1], dg_z_img[-1], dg_z_img[0]]

        # Col 0: difference image over cropped region
        ax0  = axes[row, 0]
        diff = mon_crop - base_crop
        vmax = np.nanmax(np.abs(diff))
        ax0.imshow(diff, aspect='auto', extent=extent_crop,
                   cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='upper')
        ax0.axvline(x_apex, color='k', lw=1.2, ls='-', label='apex (data)')
        ax0.set_title(
            f"{name}  (dz_true={true_dz*1e3:.1f} mm, dx_true={true_dx*1e3:.1f} mm)\n"
            f"Difference (mon − base)  |  apex @ x={x_apex*100:.1f} cm",
            fontsize=9
        )
        ax0.set_xlabel('x [m]'); ax0.set_ylabel('z [m]')

        # Col 1: cross-spectrum phase
        ax1 = axes[row, 1]
        im1 = ax1.pcolormesh(kx_s, kz_s, phi_show,
                             cmap='RdBu_r', vmin=-180, vmax=180, shading='auto')
        for sgn in [-1, 1]:
            ax1.axhline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.5)
            ax1.axvline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.5)
        ax1.set_title("Cross-spectrum phase [deg]\ndashed = fit band", fontsize=9)
        ax1.set_xlabel('kx [rad/m]'); ax1.set_ylabel('kz [rad/m]')
        ax1.set_xlim(-klim * 2.5, klim * 2.5)
        ax1.set_ylim(-klim * 2.5, klim * 2.5)
        plt.colorbar(im1, ax=ax1, fraction=0.046)

        # Col 2: cross-spectrum energy |XS|
        ax2 = axes[row, 2]
        im2 = ax2.pcolormesh(kx_s, kz_s, energy,
                             cmap='inferno', shading='auto')
        for sgn in [-1, 1]:
            ax2.axhline(sgn * klim, color='w', lw=0.8, ls='--', alpha=0.5)
            ax2.axvline(sgn * klim, color='w', lw=0.8, ls='--', alpha=0.5)
        ax2.set_title("Cross-spectrum energy |XS|\ndashed = fit band", fontsize=9)
        ax2.set_xlabel('kx [rad/m]'); ax2.set_ylabel('kz [rad/m]')
        ax2.set_xlim(-klim * 2.5, klim * 2.5)
        ax2.set_ylim(-klim * 2.5, klim * 2.5)
        plt.colorbar(im2, ax=ax2, fraction=0.046)

        # Col 3: numerical summary
        ax3 = axes[row, 3]
        ax3.axis('off')
        txt = (
            f"True:  dz = {true_dz  * 1e3:+7.3f} mm\n"
            f"       dx = {true_dx  * 1e3:+7.3f} mm\n\n"
            f"Est:   dz = {dz_est   * 1e3:+7.3f} mm\n"
            f"       dx = {dx_est   * 1e3:+7.3f} mm\n\n"
            f"Err:   dz = {(dz_est - true_dz) * 1e3:+.4f} mm\n"
            f"       dx = {(dx_est - true_dx) * 1e3:+.4f} mm\n\n"
            f"Apex @ x = {x_apex*100:.2f} cm"
        )
        ax3.text(0.05, 0.92, txt, transform=ax3.transAxes,
                 fontsize=10, va='top', family='monospace',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.tight_layout()
    plt.show()


## Section 2h — DiagonalTimeLapse (Noisy): Load Migrated Results

Load the migrated (non-difference) images for the diagonal study with Laplace-distributed
noise added to each B-scan before migration. Kirchhoff and Gazdag are post-processing on
already-loaded arrays and are always populated; Back-prop requires a separate gprMax
forward run per scenario (`write_backprop_files(..., sign_bit=True)` in
`Diagonal_TimeLapse_Playground.ipynb`) and will be all-NaN for any scenario whose
`.in` file hasn't been run through gprMax yet — `np.nan_to_num` in Section 2i below
already tolerates this the same way Section 2b/2d do.


In [ ]:
dgn = np.load(str(DG_ROOT / 'migrated_results_noisy.npz'), allow_pickle=False)

dgn_kirchhoff      = dgn['kirchhoff']           # (6, n_z, n_x)
dgn_gazdag         = dgn['gazdag']              # (6, n_z, n_x)
dgn_backprop       = dgn['backprop']            # (6, n_z, n_x) — NaN where not yet run
dgn_scenarios      = list(dgn['scenarios'])     # ['Baseline', 'Scenario 1 (1λx, ½λy)', ...]
dgn_shift_lambda_x = dgn['shift_lambda_x']
dgn_shift_lambda_y = dgn['shift_lambda_y']
dgn_x_scatterers   = dgn['x_scatterers']
dgn_y_scatterers   = dgn['y_scatterers']
dgn_z_depths       = dgn['z_depths']
dgn_x_baseline     = float(dgn['x_baseline'])
dgn_y_baseline     = float(dgn['y_baseline'])
dgn_z_scatterer    = float(dgn['z_scatterer'])
dgn_z_top          = float(dgn['z_top'])
dgn_noise_level    = float(dgn['noise_level'])

# Same survey/migration grid as Section 2f (not stored in the npz — see note there).
dgn_x_traces = dg_x_traces
dgn_z_img    = dg_z_img
dgn_dz_mig   = dg_dz_mig
dgn_dx_mig   = dg_dx_mig
dgn_kz_c     = kz_c

print(f"Loaded: {DG_ROOT / 'migrated_results_noisy.npz'}")
print(f"  kirchhoff   : {dgn_kirchhoff.shape}")
print(f"  gazdag      : {dgn_gazdag.shape}")
print(f"  backprop    : {dgn_backprop.shape}  ({int(np.isnan(dgn_backprop).all(axis=(1,2)).sum())} scenario(s) all-NaN — not yet run)")
print(f"  scenarios   : {dgn_scenarios}")
print(f"  noise_level : {dgn_noise_level}")
print(f"Grid spacing:  dz={dgn_dz_mig*1e3:.2f} mm,  dx={dgn_dx_mig*1e3:.2f} mm")


## Section 2i — DiagonalTimeLapse (Noisy): Phase-Plane Shift Estimation

Apply the plain `estimate_shift_2d` (same estimator as Section 2g, **not** the
noise-robust `estimate_shift_2d_cleaning` from Section 2b/2c) to the noisy diagonal
migrated images, using the same per-scenario apex-finding and ±2.5λ x-crop as
Section 2g. Any scenario whose Back-prop plane is all-NaN (gprMax not yet run for
that noisy `.in` file) will show as a flat/empty difference panel and a near-zero,
meaningless shift estimate — expected, not a bug; re-run after generating the
missing snapshots.


In [ ]:
for method_name, base_stack in [('Kirchhoff', dgn_kirchhoff), ('Gazdag', dgn_gazdag), ('Back-prop', dgn_backprop)]:

    base_img = base_stack[0]
    # Baseline envelope computed once per method
    env_base = np.abs(sp_hilbert(np.nan_to_num(base_img), axis=0))

    mon_cases = [
        (dgn_scenarios[i], dgn_shift_lambda_y[i] * lam, dgn_shift_lambda_x[i] * lam, base_stack[i])
        for i in range(1, len(dgn_scenarios))
    ]

    n_cases = len(mon_cases)
    fig, axes = plt.subplots(n_cases, 4, figsize=(20, 3.5 * n_cases))
    fig.suptitle(
        f"DiagonalTimeLapse (Noisy) — Phase-plane shift estimation — {method_name}  |  Baseline vs each scenario"
        f"  (noise_level={dgn_noise_level})",
        fontsize=12, fontweight='bold', y=1.01
    )

    for row, (name, true_dz, true_dx, mon_img) in enumerate(mon_cases):

        # ── Locate scatterer apex from difference envelope ─────────────────────
        env_mon  = np.abs(sp_hilbert(np.nan_to_num(mon_img), axis=0))
        diff_env = np.abs(env_mon - env_base)

        # Rough location: peak of difference envelope
        iz_d, ix_d = np.unravel_index(np.argmax(diff_env), diff_env.shape)
        x_rough    = dgn_x_traces[ix_d]

        # Refine: baseline envelope apex within ±3λ of rough location
        hw_search = 3 * lam
        ix_lo_s   = np.searchsorted(dgn_x_traces, x_rough - hw_search)
        ix_hi_s   = np.searchsorted(dgn_x_traces, x_rough + hw_search)
        _, ix_loc = np.unravel_index(
            np.argmax(env_base[:, ix_lo_s:ix_hi_s]),
            env_base[:, ix_lo_s:ix_hi_s].shape
        )
        x_apex = dgn_x_traces[ix_lo_s + ix_loc]

        # ── Crop ±2.5λ around apex ────────────────────────────────────────────
        x_crop_hw = 2.5 * lam
        ix_lo     = np.searchsorted(dgn_x_traces, x_apex - x_crop_hw)
        ix_hi     = np.searchsorted(dgn_x_traces, x_apex + x_crop_hw)
        x_crop    = dgn_x_traces[ix_lo:ix_hi]

        base_crop = np.nan_to_num(base_img[:, ix_lo:ix_hi])
        mon_crop  = np.nan_to_num(mon_img[:, ix_lo:ix_hi])

        # ── Estimate shift (real images, Hermitian XS) ────────────────────────
        dz_est, dx_est, _, XS, kz_ax, kx_ax = estimate_shift_2d(
            base_crop, mon_crop, dgn_dz_mig, dgn_dx_mig, dgn_kz_c
        )

        XS_s     = np.fft.fftshift(XS)
        kz_s     = np.fft.fftshift(kz_ax)
        kx_s     = np.fft.fftshift(kx_ax)
        energy   = np.abs(XS_s)
        phi_show = np.where(energy > 0.05 * energy.max(),
                            np.degrees(np.angle(XS_s)), np.nan)

        klim        = 1.5 * dgn_kz_c
        extent_crop = [x_crop[0], x_crop[-1], dgn_z_img[-1], dgn_z_img[0]]

        # Col 0: difference image over cropped region
        ax0  = axes[row, 0]
        diff = mon_crop - base_crop
        vmax = np.nanmax(np.abs(diff)) or 1.0
        ax0.imshow(diff, aspect='auto', extent=extent_crop,
                   cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='upper')
        ax0.axvline(x_apex, color='k', lw=1.2, ls='-', label='apex (data)')
        ax0.set_title(
            f"{name}  (dz_true={true_dz*1e3:.1f} mm, dx_true={true_dx*1e3:.1f} mm)\n"
            f"Difference (mon − base)  |  apex @ x={x_apex*100:.1f} cm",
            fontsize=9
        )
        ax0.set_xlabel('x [m]'); ax0.set_ylabel('z [m]')

        # Col 1: cross-spectrum phase
        ax1 = axes[row, 1]
        im1 = ax1.pcolormesh(kx_s, kz_s, phi_show,
                             cmap='RdBu_r', vmin=-180, vmax=180, shading='auto')
        for sgn in [-1, 1]:
            ax1.axhline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.5)
            ax1.axvline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.5)
        ax1.set_title("Cross-spectrum phase [deg]\ndashed = fit band", fontsize=9)
        ax1.set_xlabel('kx [rad/m]'); ax1.set_ylabel('kz [rad/m]')
        ax1.set_xlim(-klim * 2.5, klim * 2.5)
        ax1.set_ylim(-klim * 2.5, klim * 2.5)
        plt.colorbar(im1, ax=ax1, fraction=0.046)

        # Col 2: cross-spectrum energy |XS|
        ax2 = axes[row, 2]
        im2 = ax2.pcolormesh(kx_s, kz_s, energy,
                             cmap='inferno', shading='auto')
        for sgn in [-1, 1]:
            ax2.axhline(sgn * klim, color='w', lw=0.8, ls='--', alpha=0.5)
            ax2.axvline(sgn * klim, color='w', lw=0.8, ls='--', alpha=0.5)
        ax2.set_title("Cross-spectrum energy |XS|\ndashed = fit band", fontsize=9)
        ax2.set_xlabel('kx [rad/m]'); ax2.set_ylabel('kz [rad/m]')
        ax2.set_xlim(-klim * 2.5, klim * 2.5)
        ax2.set_ylim(-klim * 2.5, klim * 2.5)
        plt.colorbar(im2, ax=ax2, fraction=0.046)

        # Col 3: numerical summary
        ax3 = axes[row, 3]
        ax3.axis('off')
        txt = (
            f"True:  dz = {true_dz  * 1e3:+7.3f} mm\n"
            f"       dx = {true_dx  * 1e3:+7.3f} mm\n\n"
            f"Est:   dz = {dz_est   * 1e3:+7.3f} mm\n"
            f"       dx = {dx_est   * 1e3:+7.3f} mm\n\n"
            f"Err:   dz = {(dz_est - true_dz) * 1e3:+.4f} mm\n"
            f"       dx = {(dx_est - true_dx) * 1e3:+.4f} mm\n\n"
            f"Apex @ x = {x_apex*100:.2f} cm"
        )
        ax3.text(0.05, 0.92, txt, transform=ax3.transAxes,
                 fontsize=10, va='top', family='monospace',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.tight_layout()
    plt.show()


## Section 2j — VerticalTimeLapse (Noisy): Load Migrated Results

Load the migrated (non-difference) images for the vertical scatterer-displacement study
with Laplace-distributed noise added to each B-scan before migration (see
`Vertical_TimeLapse_Playground.ipynb`, "Application to Noisy Data"). Kirchhoff and Gazdag
are post-processing on already-loaded arrays and are always populated; Back-prop requires
a separate gprMax forward run per scenario (`write_backprop_files(..., sign_bit=True)`,
with the dispersion-limited low-pass filter and edge-source exclusion applied) and will be
all-NaN for any scenario whose `.in` file hasn't been run through gprMax yet —
`np.nan_to_num` in Section 2k below already tolerates this the same way Section 2i does.

In [ ]:
vtn = np.load(str(VT_ROOT / 'vertical_migrated_results_noisy.npz'), allow_pickle=False)

vtn_kirchhoff    = vtn['kirchhoff']           # (7, n_z, n_x)
vtn_gazdag       = vtn['gazdag']              # (7, n_z, n_x)
vtn_backprop     = vtn['backprop']            # (7, n_z, n_x) — NaN where not yet run
vtn_scenarios    = list(vtn['scenarios'])     # ['Baseline', '1λ', ...]
vtn_shift_lambda = vtn['shift_lambda']        # [0, 1, 0.5, 0.25, 0.125, 0.0625, 0.03125]
vtn_y_scatterers = vtn['y_scatterers']
vtn_z_depths     = vtn['z_depths']
vtn_x_traces     = vtn['x_traces']
vtn_z_img        = vtn['z_img']
vtn_x_scatterer  = float(vtn['x_scatterer'])
vtn_y_baseline   = float(vtn['y_baseline'])
vtn_z_scatterer  = float(vtn['z_scatterer'])
vtn_z_top        = float(vtn['z_top'])
vtn_noise_level  = float(vtn['noise_level'])

# Same migration grid as Section 2d (identical ice physics).
vtn_dz_mig = float(vtn_z_img[1]    - vtn_z_img[0])
vtn_dx_mig = float(vtn_x_traces[1] - vtn_x_traces[0])
vtn_kz_c   = kz_c

print(f"Loaded: {VT_ROOT / 'vertical_migrated_results_noisy.npz'}")
print(f"  kirchhoff   : {vtn_kirchhoff.shape}")
print(f"  gazdag      : {vtn_gazdag.shape}")
print(f"  backprop    : {vtn_backprop.shape}  ({int(np.isnan(vtn_backprop).all(axis=(1,2)).sum())} scenario(s) all-NaN — not yet run)")
print(f"  scenarios   : {vtn_scenarios}")
print(f"  noise_level : {vtn_noise_level}")
print(f"Grid spacing:  dz={vtn_dz_mig*1e3:.2f} mm,  dx={vtn_dx_mig*1e3:.2f} mm")


## Section 2k — VerticalTimeLapse (Noisy): Phase-Plane Shift Estimation

Apply the plain `estimate_shift_2d` (same estimator as Section 2e, **not** the
noise-robust `estimate_shift_2d_cleaning` from Section 2b/2c) to the noisy vertical
migrated images, using the same per-scenario apex-finding and ±2.5λ x-crop as
Section 2e. Any scenario whose Back-prop plane is all-NaN (gprMax not yet run for
that noisy `.in` file) will show as a flat/empty difference panel and a near-zero,
meaningless shift estimate — expected, not a bug; re-run after generating the
missing snapshots.

In [ ]:
for method_name, base_stack in [('Kirchhoff', vtn_kirchhoff), ('Gazdag', vtn_gazdag), ('Back-prop', vtn_backprop)]:

    base_img = base_stack[0]
    # Baseline envelope computed once per method
    env_base = np.abs(sp_hilbert(np.nan_to_num(base_img), axis=0))

    mon_cases = [
        (vtn_scenarios[i], vtn_shift_lambda[i] * lam, 0.0, base_stack[i])
        for i in range(1, len(vtn_scenarios))
    ]

    n_cases = len(mon_cases)
    fig, axes = plt.subplots(n_cases, 4, figsize=(20, 3.5 * n_cases))
    fig.suptitle(
        f"VerticalTimeLapse (Noisy) — Phase-plane shift estimation — {method_name}  |  Baseline vs each scenario"
        f"  (noise_level={vtn_noise_level})",
        fontsize=12, fontweight='bold', y=1.01
    )

    for row, (name, true_dz, true_dx, mon_img) in enumerate(mon_cases):

        # ── Locate scatterer apex from difference envelope ─────────────────────
        env_mon  = np.abs(sp_hilbert(np.nan_to_num(mon_img), axis=0))
        diff_env = np.abs(env_mon - env_base)

        # Rough location: peak of difference envelope
        iz_d, ix_d = np.unravel_index(np.argmax(diff_env), diff_env.shape)
        x_rough    = vtn_x_traces[ix_d]

        # Refine: baseline envelope apex within ±3λ of rough location
        hw_search = 3 * lam
        ix_lo_s   = np.searchsorted(vtn_x_traces, x_rough - hw_search)
        ix_hi_s   = np.searchsorted(vtn_x_traces, x_rough + hw_search)
        _, ix_loc = np.unravel_index(
            np.argmax(env_base[:, ix_lo_s:ix_hi_s]),
            env_base[:, ix_lo_s:ix_hi_s].shape
        )
        x_apex = vtn_x_traces[ix_lo_s + ix_loc]

        # ── Crop ±2.5λ around apex ────────────────────────────────────────────
        x_crop_hw = 2.5 * lam
        ix_lo     = np.searchsorted(vtn_x_traces, x_apex - x_crop_hw)
        ix_hi     = np.searchsorted(vtn_x_traces, x_apex + x_crop_hw)
        x_crop    = vtn_x_traces[ix_lo:ix_hi]

        base_crop = np.nan_to_num(base_img[:, ix_lo:ix_hi])
        mon_crop  = np.nan_to_num(mon_img[:, ix_lo:ix_hi])

        # ── Estimate shift (real images, Hermitian XS) ────────────────────────
        dz_est, dx_est, _, XS, kz_ax, kx_ax = estimate_shift_2d(
            base_crop, mon_crop, vtn_dz_mig, vtn_dx_mig, vtn_kz_c
        )

        XS_s     = np.fft.fftshift(XS)
        kz_s     = np.fft.fftshift(kz_ax)
        kx_s     = np.fft.fftshift(kx_ax)
        energy   = np.abs(XS_s)
        phi_show = np.where(energy > 0.05 * energy.max(),
                            np.degrees(np.angle(XS_s)), np.nan)

        klim        = 1.5 * vtn_kz_c
        extent_crop = [x_crop[0], x_crop[-1], vtn_z_img[-1], vtn_z_img[0]]

        # Col 0: difference image over cropped region
        ax0  = axes[row, 0]
        diff = mon_crop - base_crop
        vmax = np.nanmax(np.abs(diff)) or 1.0
        ax0.imshow(diff, aspect='auto', extent=extent_crop,
                   cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='upper')
        ax0.axvline(x_apex, color='k', lw=1.2, ls='-', label='apex (data)')
        ax0.set_title(
            f"{name}  (dz_true={true_dz*1e3:.1f} mm)\n"
            f"Difference (mon − base)  |  apex @ x={x_apex*100:.1f} cm",
            fontsize=9
        )
        ax0.set_xlabel('x [m]'); ax0.set_ylabel('z [m]')

        # Col 1: cross-spectrum phase
        ax1 = axes[row, 1]
        im1 = ax1.pcolormesh(kx_s, kz_s, phi_show,
                             cmap='RdBu_r', vmin=-180, vmax=180, shading='auto')
        for sgn in [-1, 1]:
            ax1.axhline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.5)
            ax1.axvline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.5)
        ax1.set_title("Cross-spectrum phase [deg]\ndashed = fit band", fontsize=9)
        ax1.set_xlabel('kx [rad/m]'); ax1.set_ylabel('kz [rad/m]')
        ax1.set_xlim(-klim * 2.5, klim * 2.5)
        ax1.set_ylim(-klim * 2.5, klim * 2.5)
        plt.colorbar(im1, ax=ax1, fraction=0.046)

        # Col 2: cross-spectrum energy |XS|
        ax2 = axes[row, 2]
        im2 = ax2.pcolormesh(kx_s, kz_s, energy,
                             cmap='inferno', shading='auto')
        for sgn in [-1, 1]:
            ax2.axhline(sgn * klim, color='w', lw=0.8, ls='--', alpha=0.5)
            ax2.axvline(sgn * klim, color='w', lw=0.8, ls='--', alpha=0.5)
        ax2.set_title("Cross-spectrum energy |XS|\ndashed = fit band", fontsize=9)
        ax2.set_xlabel('kx [rad/m]'); ax2.set_ylabel('kz [rad/m]')
        ax2.set_xlim(-klim * 2.5, klim * 2.5)
        ax2.set_ylim(-klim * 2.5, klim * 2.5)
        plt.colorbar(im2, ax=ax2, fraction=0.046)

        # Col 3: numerical summary
        ax3 = axes[row, 3]
        ax3.axis('off')
        txt = (
            f"True:  dz = {true_dz  * 1e3:+7.3f} mm\n"
            f"       dx = {true_dx  * 1e3:+7.3f} mm\n\n"
            f"Est:   dz = {dz_est   * 1e3:+7.3f} mm\n"
            f"       dx = {dx_est   * 1e3:+7.3f} mm\n\n"
            f"Err:   dz = {(dz_est - true_dz) * 1e3:+.4f} mm\n"
            f"       dx = {(dx_est - true_dx) * 1e3:+.4f} mm\n\n"
            f"Apex @ x = {x_apex*100:.2f} cm"
        )
        ax3.text(0.05, 0.92, txt, transform=ax3.transAxes,
                 fontsize=10, va='top', family='monospace',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.tight_layout()
    plt.show()


## Section 3 — FluidFlow: Lateral Fluid-Front Displacement

Estimate how far the fluid front moved (Δx) for each monitor scenario.

**Approach:**
1. Find x₀ (baseline fluid front) from the smallest-shift scenario's diff-envelope peak.
2. Crop ±2.5λ around x₀ so the focused edge diffraction dominates the cross-spectrum.
3. Real base vs real mon: Hermitian XS forces φ₀ = 0, WLS fits a pure kx slope → dx_raw.

**Centroid-shift correction (factor 2):**

The cross-spectrum measures the centroid shift of the changed water block, not the leading edge.
The newly filled region runs from x₀ to x₀ + Δx, so its centroid is at x₀ + Δx/2:

$$\Delta x_\text{raw} = \frac{\Delta x_\text{true}}{2} \implies \Delta x_\text{true} = 2\,\Delta x_\text{raw}$$

The 0.5 mm residual for the ¹⁄₃₂λ case is a grid-quantisation artefact (Δx = 3.5 mm < dx_mig = 10 mm).


In [ ]:
FF_ROOT = Path(r'C:\Users\Administrator\OneDrive\Thesis\TimeLapse_Notebooks\fluidflow_study')

ff = np.load(str(FF_ROOT / 'migrated_results.npz'), allow_pickle=False)

ff_kirchhoff         = ff['kirchhoff']           # (n_scen, n_z, n_x)
ff_gazdag            = ff['gazdag']
ff_backprop          = ff['backprop']
ff_scenarios         = list(ff['scenarios'])
ff_separation_lambda = ff['separation_lambda']   # in units of lam (ice wavelength)
ff_x_traces          = ff['x_traces']
ff_z_img             = ff['z_img']
ff_z_top             = float(ff['z_top'])

# Grid spacing and wavenumber (same ice physics as TimeLapse)
ff_dz_mig = float(ff_z_img[1]     - ff_z_img[0])
ff_dx_mig = float(ff_x_traces[1]  - ff_x_traces[0])
ff_kz_c   = kz_c   # 2π / lam_ice — identical migration grid

print(f"Loaded: {FF_ROOT / 'migrated_results.npz'}")
print(f"  kirchhoff : {ff_kirchhoff.shape}")
print(f"  gazdag    : {ff_gazdag.shape}")
print(f"  backprop  : {ff_backprop.shape}")
print(f"  scenarios : {ff_scenarios}")
print(f"\nGrid spacing:  dz={ff_dz_mig*1e3:.2f} mm,  dx={ff_dx_mig*1e3:.2f} mm")

In [ ]:
for method_name, base_stack in [('Kirchhoff', ff_kirchhoff), ('Gazdag', ff_gazdag), ('Back-prop', ff_backprop)]:

    base_img = base_stack[0]
    env_base = np.abs(sp_hilbert(np.nan_to_num(base_img), axis=0))

    # ── Locate baseline fluid front ────────────────────────────────────────────
    # Use the smallest-shift scenario's diff-envelope peak (≈ x₀ + λ/32 ≈ x₀).
    # This is robust against migration artifacts that can shift the global env_base
    # argmax to the wrong lateral position.
    env_ref      = np.abs(sp_hilbert(np.nan_to_num(base_stack[-1]), axis=0))
    diff_env_ref = np.abs(env_ref - env_base)
    _, ix_ref    = np.unravel_index(np.argmax(diff_env_ref), diff_env_ref.shape)
    x_apex_base  = ff_x_traces[ix_ref]

    mon_cases = [
        (ff_scenarios[i], 0.0, fdtd_true(ff_separation_lambda[i] * lam), base_stack[i])
        for i in range(1, len(ff_scenarios))
    ]

    n_cases = len(mon_cases)
    fig, axes = plt.subplots(n_cases, 4, figsize=(20, 3.5 * n_cases))
    fig.suptitle(
        f"FluidFlow — {method_name}  |  base vs mon (real)  |  crop ±2.5λ @ baseline front",
        fontsize=11, fontweight='bold', y=1.01
    )

    # Back-propagation resolves the reflector directly; Kirchhoff/Gazdag
    # produce a smeared-centroid that sits at half the true front shift.
    ff_scale = 1.0 if method_name == 'Back-prop' else 2.0

    for row, (name, true_dz, true_dx, mon_img) in enumerate(mon_cases):

        # ── Crop ±2.5λ around the BASELINE fluid front ────────────────────────
        x_crop_hw = 2.5 * lam
        ix_lo     = np.searchsorted(ff_x_traces, x_apex_base - x_crop_hw)
        ix_hi     = np.searchsorted(ff_x_traces, x_apex_base + x_crop_hw)
        x_crop    = ff_x_traces[ix_lo:ix_hi]

        base_crop = base_img[:, ix_lo:ix_hi]
        mon_crop  = mon_img[:,  ix_lo:ix_hi]

        # ── Phase-plane shift estimate (real images, Hermitian → phi_0 = 0) ───
        dz_est, dx_est, _, XS, kz_ax, kx_ax = estimate_shift_2d(
            base_crop, mon_crop, ff_dz_mig, ff_dx_mig, ff_kz_c
        )
        
        dx_front_inferred = ff_scale * dx_est
        x_front_predicted = x_apex_base + dx_front_inferred

        # ── Visualisation Updates ─────────────────────────────────────────────
        XS_s     = np.fft.fftshift(XS)
        kz_s     = np.fft.fftshift(kz_ax)
        kx_s     = np.fft.fftshift(kx_ax)
        energy   = np.abs(XS_s)
        phi_show = np.where(energy > 0.01 * energy.max(),
                            np.degrees(np.angle(XS_s)), np.nan)
        klim        = 1.5 * ff_kz_c
        extent_crop = [x_crop[0], x_crop[-1], ff_z_img[-1], ff_z_img[0]]

        # Col 0: difference image with the new Inferred Front marker
        ax0   = axes[row, 0]
        diff_ = mon_crop - base_crop
        vmax  = np.nanmax(np.abs(diff_)) or 1.0
        ax0.imshow(diff_, aspect='auto', extent=extent_crop,
                   cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='upper')
        ax0.axvline(x_apex_base,           color='k', lw=1.2, ls='--', alpha=0.9, label='x₀ (base front)')
        ax0.axvline(x_apex_base + true_dx, color='r', lw=1.0, ls=':',  alpha=0.9, label='True Front')
        
        # GREEN dashed line tracks your scaled inference
        ax0.axvline(x_front_predicted,     color='g', lw=1.2, ls='--', alpha=0.9, label=f'Inferred Front ({ff_scale:.0f}×)')
        
        ax0.axhline(ff_z_top, color='grey', lw=0.8, ls='--', alpha=0.6)
        ax0.set_title(
            f"{name}  (dx_true = {true_dx*1e3:.1f} mm)\n"
            f"mon − base  |  crop centred @ x={x_apex_base*100:.1f} cm",
            fontsize=9
        )
        ax0.set_xlabel('x [m]'); ax0.set_ylabel('z [m]')
        ax0.legend(fontsize=7, loc='lower right')

        # Col 1: cross-spectrum phase  (should be a kx-slope = dx)
        ax1 = axes[row, 1]
        im1 = ax1.pcolormesh(kx_s, kz_s, phi_show,
                             cmap='RdBu_r', vmin=-180, vmax=180, shading='auto')
        for sgn in [-1, 1]:
            ax1.axhline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.5)
            ax1.axvline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.5)
        ax1.set_title("XS phase [deg]\n(kx tilt = dx_est)", fontsize=9)
        ax1.set_xlabel('kx [rad/m]'); ax1.set_ylabel('kz [rad/m]')
        ax1.set_xlim(-klim * 2.5, klim * 2.5); ax1.set_ylim(-klim * 2.5, klim * 2.5)
        plt.colorbar(im1, ax=ax1, fraction=0.046)

        # Col 2: cross-spectrum energy
        ax2 = axes[row, 2]
        im2 = ax2.pcolormesh(kx_s, kz_s, energy, cmap='inferno', shading='auto')
        for sgn in [-1, 1]:
            ax2.axhline(sgn * klim, color='w', lw=0.8, ls='--', alpha=0.5)
            ax2.axvline(sgn * klim, color='w', lw=0.8, ls='--', alpha=0.5)
        ax2.set_title("XS energy |XS|", fontsize=9)
        ax2.set_xlabel('kx [rad/m]'); ax2.set_ylabel('kz [rad/m]')
        ax2.set_xlim(-klim * 2.5, klim * 2.5); ax2.set_ylim(-klim * 2.5, klim * 2.5)
        plt.colorbar(im2, ax=ax2, fraction=0.046)

        # Col 3: Numerical summary showing the corrected scaling
        ax3 = axes[row, 3]
        ax3.axis('off')
        txt = (
            f"True Front Δx: {true_dx * 1e3:+7.3f} mm\n\n"
            f"Raw Est Δx:    {dx_est  * 1e3:+7.3f} mm (Centroid)\n"
            f"Inferred Δx:   {dx_front_inferred * 1e3:+7.3f} mm ({ff_scale:.0f} × Est)\n\n"
            f"Inference Err: {(dx_front_inferred - true_dx) * 1e3:+.4f} mm\n"
            f"Vertical Drift: {dz_est  * 1e3:+.4f} mm (true 0)\n\n"
            f"x₀ (base)   @ {x_apex_base*100:.2f} cm\n"
            f"Front (inf) @ {x_front_predicted*100:.2f} cm"
        )
        ax3.text(0.05, 0.95, txt, transform=ax3.transAxes,
                 fontsize=10, va='top', family='monospace',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.tight_layout()
    plt.show()


## Section 3a — FluidFlow (Noisy): Load & Phase-Plane Shift Estimation

Noisy counterpart of Section 3, generated in `FluidFlow_Playground.ipynb` with
the same Laplace noise model used for the other noisy datasets in this study
(`migrated_results_noisy.npz`, same grid/scenarios as the clean run). Repeats
Section 3's baseline-front-centroid + 2× correction unchanged — same migration
grid, same `estimate_shift_2d`, just on the noisy base/monitor pair.

**Gazdag's known noisy-migration artifact** (documented in Section 2b/2c for
the lateral dataset) shows up here too: its noisy estimates collapse to ≈0,
which is visible below as a flat phase map with no clear kx tilt.

In [ ]:
ff_noisy = np.load(str(FF_ROOT / 'migrated_results_noisy.npz'), allow_pickle=False)

ff_noisy_kirchhoff         = ff_noisy['kirchhoff']
ff_noisy_gazdag            = ff_noisy['gazdag']
ff_noisy_backprop          = ff_noisy['backprop']
ff_noisy_scenarios         = list(ff_noisy['scenarios'])
ff_noisy_separation_lambda = ff_noisy['separation_lambda']
ff_noisy_noise_level       = float(ff_noisy['noise_level'])

assert ff_noisy_scenarios == ff_scenarios, "FluidFlow clean/noisy scenario order mismatch"
# Same migration grid as the clean run (verified: x_traces/z_img identical) --
# reuse ff_x_traces/ff_z_img/ff_dz_mig/ff_dx_mig/ff_kz_c/ff_z_top throughout.

print(f"Loaded: {FF_ROOT / 'migrated_results_noisy.npz'}")
print(f"  kirchhoff   : {ff_noisy_kirchhoff.shape}")
print(f"  gazdag      : {ff_noisy_gazdag.shape}")
print(f"  backprop    : {ff_noisy_backprop.shape}")
print(f"  scenarios   : {ff_noisy_scenarios}")
print(f"  noise_level : {ff_noisy_noise_level}")

In [ ]:
for method_name, base_stack in [('Kirchhoff', ff_noisy_kirchhoff), ('Gazdag', ff_noisy_gazdag), ('Back-prop', ff_noisy_backprop)]:

    base_img = np.nan_to_num(base_stack[0])
    env_base = np.abs(sp_hilbert(base_img, axis=0))

    # ── Locate baseline fluid front (same approach as Section 3) ───────────────
    env_ref      = np.abs(sp_hilbert(np.nan_to_num(base_stack[-1]), axis=0))
    diff_env_ref = np.abs(env_ref - env_base)
    _, ix_ref    = np.unravel_index(np.argmax(diff_env_ref), diff_env_ref.shape)
    x_apex_base  = ff_x_traces[ix_ref]

    mon_cases = [
        (ff_noisy_scenarios[i], 0.0, fdtd_true(ff_noisy_separation_lambda[i] * lam), base_stack[i])
        for i in range(1, len(ff_noisy_scenarios))
    ]

    n_cases = len(mon_cases)
    fig, axes = plt.subplots(n_cases, 4, figsize=(20, 3.5 * n_cases))
    fig.suptitle(
        f"FluidFlow (Noisy, noise_level={ff_noisy_noise_level}) — {method_name}  |  "
        f"base vs mon (real)  |  crop ±2.5λ @ baseline front",
        fontsize=11, fontweight='bold', y=1.01
    )

    ff_scale = 1.0 if method_name == 'Back-prop' else 2.0

    for row, (name, true_dz, true_dx, mon_img) in enumerate(mon_cases):

        # ── Crop ±2.5λ around the BASELINE fluid front ────────────────────────
        x_crop_hw = 2.5 * lam
        ix_lo     = np.searchsorted(ff_x_traces, x_apex_base - x_crop_hw)
        ix_hi     = np.searchsorted(ff_x_traces, x_apex_base + x_crop_hw)
        x_crop    = ff_x_traces[ix_lo:ix_hi]

        base_crop = base_img[:, ix_lo:ix_hi]
        mon_crop  = np.nan_to_num(mon_img)[:, ix_lo:ix_hi]

        # ── Phase-plane shift estimate (real images, Hermitian → phi_0 = 0) ───
        dz_est, dx_est, _, XS, kz_ax, kx_ax = estimate_shift_2d(
            base_crop, mon_crop, ff_dz_mig, ff_dx_mig, ff_kz_c
        )

        dx_front_inferred = ff_scale * dx_est
        x_front_predicted = x_apex_base + dx_front_inferred

        XS_s     = np.fft.fftshift(XS)
        kz_s     = np.fft.fftshift(kz_ax)
        kx_s     = np.fft.fftshift(kx_ax)
        energy   = np.abs(XS_s)
        phi_show = np.where(energy > 0.01 * energy.max(),
                            np.degrees(np.angle(XS_s)), np.nan)
        klim        = 1.5 * ff_kz_c
        extent_crop = [x_crop[0], x_crop[-1], ff_z_img[-1], ff_z_img[0]]

        # Col 0: difference image with the inferred-front marker
        ax0   = axes[row, 0]
        diff_ = mon_crop - base_crop
        vmax  = np.nanmax(np.abs(diff_)) or 1.0
        ax0.imshow(diff_, aspect='auto', extent=extent_crop,
                   cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='upper')
        ax0.axvline(x_apex_base,           color='k', lw=1.2, ls='--', alpha=0.9, label='x₀ (base front)')
        ax0.axvline(x_apex_base + true_dx, color='r', lw=1.0, ls=':',  alpha=0.9, label='True Front')
        ax0.axvline(x_front_predicted,     color='g', lw=1.2, ls='--', alpha=0.9, label=f'Inferred Front ({ff_scale:.0f}×)')
        ax0.axhline(ff_z_top, color='grey', lw=0.8, ls='--', alpha=0.6)
        ax0.set_title(
            f"{name}  (dx_true = {true_dx*1e3:.1f} mm)\n"
            f"mon − base (noisy)  |  crop centred @ x={x_apex_base*100:.1f} cm",
            fontsize=9
        )
        ax0.set_xlabel('x [m]'); ax0.set_ylabel('z [m]')
        ax0.legend(fontsize=7, loc='lower right')

        # Col 1: cross-spectrum phase (should be a kx-slope = dx)
        ax1 = axes[row, 1]
        im1 = ax1.pcolormesh(kx_s, kz_s, phi_show,
                             cmap='RdBu_r', vmin=-180, vmax=180, shading='auto')
        for sgn in [-1, 1]:
            ax1.axhline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.5)
            ax1.axvline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.5)
        ax1.set_title("XS phase [deg]\n(kx tilt = dx_est)", fontsize=9)
        ax1.set_xlabel('kx [rad/m]'); ax1.set_ylabel('kz [rad/m]')
        ax1.set_xlim(-klim * 2.5, klim * 2.5); ax1.set_ylim(-klim * 2.5, klim * 2.5)
        plt.colorbar(im1, ax=ax1, fraction=0.046)

        # Col 2: cross-spectrum energy
        ax2 = axes[row, 2]
        im2 = ax2.pcolormesh(kx_s, kz_s, energy, cmap='inferno', shading='auto')
        for sgn in [-1, 1]:
            ax2.axhline(sgn * klim, color='w', lw=0.8, ls='--', alpha=0.5)
            ax2.axvline(sgn * klim, color='w', lw=0.8, ls='--', alpha=0.5)
        ax2.set_title("XS energy |XS|", fontsize=9)
        ax2.set_xlabel('kx [rad/m]'); ax2.set_ylabel('kz [rad/m]')
        ax2.set_xlim(-klim * 2.5, klim * 2.5); ax2.set_ylim(-klim * 2.5, klim * 2.5)
        plt.colorbar(im2, ax=ax2, fraction=0.046)

        # Col 3: numerical summary
        ax3 = axes[row, 3]
        ax3.axis('off')
        txt = (
            f"True Front Δx: {true_dx * 1e3:+7.3f} mm\n\n"
            f"Raw Est Δx:    {dx_est  * 1e3:+7.3f} mm (Centroid)\n"
            f"Inferred Δx:   {dx_front_inferred * 1e3:+7.3f} mm ({ff_scale:.0f} × Est)\n\n"
            f"Inference Err: {(dx_front_inferred - true_dx) * 1e3:+.4f} mm\n"
            f"Vertical Drift: {dz_est  * 1e3:+.4f} mm (true 0)\n\n"
            f"x₀ (base)   @ {x_apex_base*100:.2f} cm\n"
            f"Front (inf) @ {x_front_predicted*100:.2f} cm"
        )
        ax3.text(0.05, 0.95, txt, transform=ax3.transAxes,
                 fontsize=10, va='top', family='monospace',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.tight_layout()
    plt.show()

## Section 3b — Error Summary Tables

Pulls the per-scenario shift estimates already shown in Sections 2/2c/2e/2g/2i/2k/3/3c
into one compact table per movement type, instead of reading them off each
scenario's individual text-summary panel. Each cell below re-runs the *same*
apex-finding + crop + phase-plane-fit pipeline as its corresponding section
(not a separate analysis) — just without the per-scenario plotting — so the
numbers match exactly what's shown in those figures.

- **Error = estimated shift − true shift (mm)**, i.e. predicted change minus
  actual change. Diverging colour scale per table: white ≈ 0, red = positive,
  blue = negative.
- **Lateral / Vertical / Diagonal / FluidFlow** tables include both the clean
  and noisy (Laplace-noise) results side by side, to show how much noise
  degrades each method.
- Lateral table reports Δx error only (Δz is identically 0 by construction for
  that movement type); Vertical reports Δz only (Δx is identically 0); Diagonal
  reports both; FluidFlow reports its own front-displacement inference error
  (Section 3's 2× centroid correction), not a raw point-scatterer Δz/Δx.
- FluidFlow (Noisy)'s Gazdag column shows the same structural noisy-migration
  artifact documented in Section 2b/2c — its estimates collapse to ≈0, giving
  an error that exactly equals −true_dx for every scenario.

In [ ]:
import pandas as pd


def compute_shift_errors(base_stack, scenario_labels, true_dz_arr, true_dx_arr,
                          x_traces, z_img, dz_mig, dx_mig, kz_c,
                          estimator_fn=estimate_shift_2d, shared_apex=False,
                          crop_z=False, search_lam=3.0, crop_hw_lam=2.5):
    """
    Re-runs the apex-finding + crop + phase-plane-fit pipeline used throughout
    Section 2/2c/2e/2g/2i (per-scenario apex by default; shared_apex=True
    replicates Section 2c/2i's median-stacked-consensus apex), returning the
    per-scenario (dz_est, dx_est, true_dz, true_dx) instead of plotting it —
    used to build the error-summary tables below.
    """
    if base_stack is None:
        return None

    base_img = np.nan_to_num(base_stack[0])
    env_base = np.abs(sp_hilbert(base_img, axis=0))
    mon_imgs = [np.nan_to_num(base_stack[i]) for i in range(1, len(scenario_labels))]

    ix_d_shared = None
    if shared_apex:
        diff_env_stack = np.stack([
            np.abs(np.abs(sp_hilbert(m, axis=0)) - env_base) for m in mon_imgs
        ], axis=0)
        diff_env_consensus = np.median(diff_env_stack, axis=0)
        _, ix_d_shared = np.unravel_index(np.argmax(diff_env_consensus), diff_env_consensus.shape)

    results = []
    for i, mon_img in enumerate(mon_imgs, start=1):
        if shared_apex:
            x_rough = x_traces[ix_d_shared]
        else:
            env_mon  = np.abs(sp_hilbert(mon_img, axis=0))
            diff_env = np.abs(env_mon - env_base)
            _, ix_d  = np.unravel_index(np.argmax(diff_env), diff_env.shape)
            x_rough  = x_traces[ix_d]

        hw_search      = search_lam * lam
        ix_lo_s        = np.searchsorted(x_traces, x_rough - hw_search)
        ix_hi_s        = np.searchsorted(x_traces, x_rough + hw_search)
        iz_loc, ix_loc = np.unravel_index(
            np.argmax(env_base[:, ix_lo_s:ix_hi_s]),
            env_base[:, ix_lo_s:ix_hi_s].shape
        )
        x_apex = x_traces[ix_lo_s + ix_loc]
        z_apex = z_img[iz_loc]

        x_crop_hw = crop_hw_lam * lam
        ix_lo = np.searchsorted(x_traces, x_apex - x_crop_hw)
        ix_hi = np.searchsorted(x_traces, x_apex + x_crop_hw)

        if crop_z:
            z_crop_hw = crop_hw_lam * lam
            iz_lo = np.searchsorted(z_img, z_apex - z_crop_hw)
            iz_hi = np.searchsorted(z_img, z_apex + z_crop_hw)
            base_crop = base_img[iz_lo:iz_hi, ix_lo:ix_hi]
            mon_crop  = mon_img[iz_lo:iz_hi, ix_lo:ix_hi]
        else:
            base_crop = base_img[:, ix_lo:ix_hi]
            mon_crop  = mon_img[:, ix_lo:ix_hi]

        dz_est, dx_est, *_ = estimator_fn(base_crop, mon_crop, dz_mig, dx_mig, kz_c)

        results.append(dict(scenario=scenario_labels[i],
                             dz_est=dz_est, dx_est=dx_est,
                             true_dz=true_dz_arr[i], true_dx=true_dx_arr[i]))
    return results


def compute_fluidflow_errors(base_stack, scenario_labels, true_dx_arr,
                              x_traces, z_img, dz_mig, dx_mig, kz_c,
                              scale=2.0):
    """Replicates Section 3's baseline-front-centroid + scale correction.
    scale=2.0 for Kirchhoff/Gazdag (centroid of the smeared wavelet sits at
    half the true front displacement); scale=1.0 for back-propagation (the
    focused image directly resolves the reflector, no upscaling needed)."""
    if base_stack is None:
        return None

    base_img = np.nan_to_num(base_stack[0])
    env_base = np.abs(sp_hilbert(base_img, axis=0))

    env_ref      = np.abs(sp_hilbert(np.nan_to_num(base_stack[-1]), axis=0))
    diff_env_ref = np.abs(env_ref - env_base)
    _, ix_ref    = np.unravel_index(np.argmax(diff_env_ref), diff_env_ref.shape)
    x_apex_base  = x_traces[ix_ref]

    x_crop_hw = 2.5 * lam
    ix_lo = np.searchsorted(x_traces, x_apex_base - x_crop_hw)
    ix_hi = np.searchsorted(x_traces, x_apex_base + x_crop_hw)
    base_crop = base_img[:, ix_lo:ix_hi]

    results = []
    for i in range(1, len(scenario_labels)):
        mon_crop = np.nan_to_num(base_stack[i])[:, ix_lo:ix_hi]
        dz_est, dx_est, *_ = estimate_shift_2d(base_crop, mon_crop, dz_mig, dx_mig, kz_c)
        dx_front_inferred = scale * dx_est
        results.append(dict(scenario=scenario_labels[i],
                             dx_front_inferred=dx_front_inferred,
                             true_dx=true_dx_arr[i]))
    return results


def style_error_table(df, decimals=2):
    """Symmetric diverging colour scale (white=0) shared by the error tables below."""
    vmax = np.nanmax(np.abs(df.to_numpy(dtype=float)))
    vmax = vmax if vmax > 0 else 1.0
    return (df.style
              .format(f'{{:+.{decimals}f}}', na_rep='—')
              .background_gradient(cmap='RdBu_r', vmin=-vmax, vmax=vmax, axis=None))

def style_pct_table(df, decimals=1):
    """Same diverging colour scale; values shown as % of true displacement."""
    vmax = np.nanmax(np.abs(df.to_numpy(dtype=float)))
    vmax = vmax if vmax > 0 else 1.0
    return (df.style
              .format(f'{{:+.{decimals}f}} %', na_rep='—')
              .background_gradient(cmap='RdBu_r', vmin=-vmax, vmax=vmax, axis=None))

In [ ]:
# -- Table 1: Lateral movement -- Δx error (mm), clean vs noisy -----------------
lateral_methods = [
    ('Kirchhoff', kirchhoff, noisy_kirchhoff),
    ('Gazdag',    gazdag,    noisy_gazdag),
    ('Back-prop', backprop,  dn['backprop'] if 'backprop' in dn.files else None),
]
assert scenarios[1:] == noisy_scenarios[1:], "lateral clean/noisy scenario order mismatch"

lat_true_dz, lat_true_dx     = np.zeros(len(scenarios)), fdtd_true(separation_lambda * lam)
lat_true_dz_n, lat_true_dx_n = np.zeros(len(noisy_scenarios)), fdtd_true(noisy_separation_lambda * lam)
lat_rows = scenarios[1:]

lat_data = {}
for name, clean_stack, noisy_stack in lateral_methods:
    res_clean = compute_shift_errors(
        clean_stack, scenarios, lat_true_dz, lat_true_dx,
        x_traces, z_img, dz_mig, dx_mig, kz_c,
        estimator_fn=estimate_shift_2d, shared_apex=False, crop_z=False
    )
    res_noisy = compute_shift_errors(
        noisy_stack, noisy_scenarios, lat_true_dz_n, lat_true_dx_n,
        noisy_x_traces, noisy_z_img, noisy_dz_mig, noisy_dx_mig, noisy_kz_c,
        estimator_fn=estimate_shift_2d_cleaning, shared_apex=True, crop_z=True
    )
    lat_data[(name, 'Clean')] = [r['dx_est'] - r['true_dx'] for r in res_clean] if res_clean else [np.nan] * len(lat_rows)
    lat_data[(name, 'Noisy')] = [r['dx_est'] - r['true_dx'] for r in res_noisy] if res_noisy else [np.nan] * len(lat_rows)

lateral_table = pd.DataFrame(lat_data, index=lat_rows) * 1e3   # mm
lateral_table.columns.names = ['Method', 'Condition']
lateral_table.index.name = 'Scenario'

print('Table 1 -- Lateral movement: Δx error (estimate - true), mm')
style_error_table(lateral_table)

In [ ]:
# -- Table 1 (%): Lateral -- Δx error as % of true displacement ---------------
lat_true_mm = fdtd_true(separation_lambda * lam)[1:] * 1e3   # mm, one value per row
lat_pct = lateral_table.divide(pd.Series(lat_true_mm, index=lat_rows), axis=0) * 100

print('Table 1 (%) -- Lateral: Δx error relative to true displacement')
style_pct_table(lat_pct)

In [ ]:
# -- Table 2: Vertical movement -- Δz error (mm), clean vs noisy -----------------
vertical_methods = [
    ('Kirchhoff', vt_kirchhoff, vtn_kirchhoff),
    ('Gazdag',    vt_gazdag,    vtn_gazdag),
    ('Back-prop', vt_backprop,  vtn_backprop),
]
vt_true_dz_full,  vt_true_dx_full  = fdtd_true(vt_shift_lambda * lam),  np.zeros(len(vt_scenarios))
vtn_true_dz_full, vtn_true_dx_full = fdtd_true(vtn_shift_lambda * lam), np.zeros(len(vtn_scenarios))
vt_rows = vt_scenarios[1:]
assert vt_scenarios[1:] == vtn_scenarios[1:], "vertical clean/noisy scenario order mismatch"

vt_data = {}
for name, clean_stack, noisy_stack in vertical_methods:
    res_clean = compute_shift_errors(
        clean_stack, vt_scenarios, vt_true_dz_full, vt_true_dx_full,
        vt_x_traces, vt_z_img, vt_dz_mig, vt_dx_mig, vt_kz_c,
        estimator_fn=estimate_shift_2d, shared_apex=False, crop_z=False
    )
    res_noisy = compute_shift_errors(
        noisy_stack, vtn_scenarios, vtn_true_dz_full, vtn_true_dx_full,
        vtn_x_traces, vtn_z_img, vtn_dz_mig, vtn_dx_mig, vtn_kz_c,
        estimator_fn=estimate_shift_2d, shared_apex=False, crop_z=False
    )
    vt_data[(name, 'Clean')] = [r['dz_est'] - r['true_dz'] for r in res_clean] if res_clean else [np.nan] * len(vt_rows)
    vt_data[(name, 'Noisy')] = [r['dz_est'] - r['true_dz'] for r in res_noisy] if res_noisy else [np.nan] * len(vt_rows)

vertical_table = pd.DataFrame(vt_data, index=vt_rows) * 1e3
vertical_table.columns.names = ['Method', 'Condition']
vertical_table.index.name = 'Scenario'

print('Table 2 -- Vertical movement: Δz error (estimate - true), mm')
style_error_table(vertical_table)


In [ ]:
# -- Table 2 (%): Vertical -- Δz error as % of true displacement ---------------
vt_true_mm = fdtd_true(vt_shift_lambda * lam)[1:] * 1e3
vt_pct = vertical_table.divide(pd.Series(vt_true_mm, index=vt_rows), axis=0) * 100

print('Table 2 (%) -- Vertical: Δz error relative to true displacement')
style_pct_table(vt_pct)

In [ ]:
# -- Table 3: Diagonal movement -- Δz/Δx error (mm), clean vs noisy --------------
diagonal_methods = [
    ('Kirchhoff', dg_kirchhoff, dgn_kirchhoff),
    ('Gazdag',    dg_gazdag,    dgn_gazdag),
    ('Back-prop', dg_backprop,  dgn_backprop),
]
dg_true_dz_full,  dg_true_dx_full  = fdtd_true(dg_shift_lambda_y * lam),  fdtd_true(dg_shift_lambda_x * lam)
dgn_true_dz_full, dgn_true_dx_full = fdtd_true(dgn_shift_lambda_y * lam), fdtd_true(dgn_shift_lambda_x * lam)

# Shorten 'Scenario N (1λx, ½λy)' -> '1λx, ½λy' for compact row labels
dg_rows  = [s.split('(', 1)[1].rstrip(')') for s in dg_scenarios[1:]]
dgn_rows = [s.split('(', 1)[1].rstrip(')') for s in dgn_scenarios[1:]]
assert dg_rows == dgn_rows, "diagonal clean/noisy scenario order mismatch"

dg_data = {}
for name, clean_stack, noisy_stack in diagonal_methods:
    res_clean = compute_shift_errors(
        clean_stack, dg_scenarios, dg_true_dz_full, dg_true_dx_full,
        dg_x_traces, dg_z_img, dg_dz_mig, dg_dx_mig, dg_kz_c,
        estimator_fn=estimate_shift_2d, shared_apex=False, crop_z=False
    )
    res_noisy = compute_shift_errors(
        noisy_stack, dgn_scenarios, dgn_true_dz_full, dgn_true_dx_full,
        dgn_x_traces, dgn_z_img, dgn_dz_mig, dgn_dx_mig, dgn_kz_c,
        estimator_fn=estimate_shift_2d, shared_apex=False, crop_z=False
    )
    for cond, res in [('Clean', res_clean), ('Noisy', res_noisy)]:
        dg_data[(name, cond, 'Δz')] = [r['dz_est'] - r['true_dz'] for r in res] if res else [np.nan] * len(dg_rows)
        dg_data[(name, cond, 'Δx')] = [r['dx_est'] - r['true_dx'] for r in res] if res else [np.nan] * len(dg_rows)

diagonal_table = pd.DataFrame(dg_data, index=dg_rows) * 1e3
diagonal_table.columns.names = ['Method', 'Condition', 'Axis']
diagonal_table.index.name = 'Scenario'

print('Table 3 -- Diagonal movement: Δz / Δx error (estimate - true), mm')
style_error_table(diagonal_table)

In [ ]:
# -- Table 3 (%): Diagonal -- Δz/Δx errors as % of true displacement per axis --
# Each row has a different true_dz (y) and true_dx (x), so divide each column family
# separately by its own axis's true displacement.
dg_true_dz_mm = dg_true_dz_full[1:] * 1e3   # mm, one value per scenario row
dg_true_dx_mm = dg_true_dx_full[1:] * 1e3

dg_pct = diagonal_table.copy().astype(float)
for col in dg_pct.columns:
    _, _, axis = col   # (Method, Condition, 'Δz' or 'Δx')
    divisor = dg_true_dz_mm if axis == 'Δz' else dg_true_dx_mm
    dg_pct[col] = diagonal_table[col].to_numpy() / divisor * 100

print('Table 3 (%) -- Diagonal: errors as % of true displacement (Δz÷true_dz, Δx÷true_dx)')
style_pct_table(dg_pct)

In [ ]:
# -- Table 4: FluidFlow -- inferred front Δx error (mm), clean vs noisy ---------
fluidflow_methods = [
    ('Kirchhoff', ff_kirchhoff, ff_noisy_kirchhoff),
    ('Gazdag',    ff_gazdag,    ff_noisy_gazdag),
    ('Back-prop', ff_backprop,  ff_noisy_backprop),
]
assert ff_scenarios == ff_noisy_scenarios, "FluidFlow clean/noisy scenario order mismatch"

ff_true_dx_full = fdtd_true(ff_separation_lambda * lam)
ff_rows = ff_scenarios[1:]

ff_data = {}
for name, clean_stack, noisy_stack in fluidflow_methods:
    scale = 1.0 if name == 'Back-prop' else 2.0
    res_clean = compute_fluidflow_errors(
        clean_stack, ff_scenarios, ff_true_dx_full, ff_x_traces, ff_z_img, ff_dz_mig, ff_dx_mig, ff_kz_c,
        scale=scale
    )
    res_noisy = compute_fluidflow_errors(
        noisy_stack, ff_noisy_scenarios, ff_true_dx_full, ff_x_traces, ff_z_img, ff_dz_mig, ff_dx_mig, ff_kz_c,
        scale=scale
    )
    ff_data[(name, 'Clean')] = [r['dx_front_inferred'] - r['true_dx'] for r in res_clean] if res_clean else [np.nan] * len(ff_rows)
    ff_data[(name, 'Noisy')] = [r['dx_front_inferred'] - r['true_dx'] for r in res_noisy] if res_noisy else [np.nan] * len(ff_rows)

fluidflow_table = pd.DataFrame(ff_data, index=ff_rows) * 1e3
fluidflow_table.columns.names = ['Method', 'Condition']
fluidflow_table.index.name = 'Scenario'

print('Table 4 -- FluidFlow: inferred front Δx error (2x centroid estimate - true), mm')
print('Note: Gazdag (Noisy) shows the same structural migration artifact documented')
print('in Section 2b/2c -- its estimates collapse to ~0, giving an error that')
print('exactly equals -true_dx for every scenario.')
style_error_table(fluidflow_table)

In [ ]:
# -- Table 4 (%): FluidFlow -- front Δx error as % of true displacement ---------
ff_true_mm = fdtd_true(ff_separation_lambda * lam)[1:] * 1e3
ff_pct = fluidflow_table.divide(pd.Series(ff_true_mm, index=ff_rows), axis=0) * 100

print('Table 4 (%) -- FluidFlow: front Δx error relative to true displacement')
style_pct_table(ff_pct)

## Section 3c — Comparison of Migration Methods

### Summary of estimation errors across all experiments

The tables above (Section 3b) quantify estimation errors for Kirchhoff delay-and-sum,
Gazdag phase-shift, and back-propagation (time-reversal FDTD) migration across four
movement types.  The phase-plane estimator is only reliable up to roughly **½λ**
(≈ 56 mm): beyond that the cross-spectrum wraps and all three methods diverge.
The comparisons below therefore focus on the **¼λ–¹⁄₃₂λ range** (28 mm down to 4 mm).

---

### Clean synthetic data

| Method | Lateral Δx (MAE, ≤½λ) | Vertical Δz (MAE, ≤½λ) | Fluid-flow Δx (MAE) |
|---|---|---|---|
| Kirchhoff | 0.03 mm | 16.8 mm* | 32.5 mm* |
| Gazdag | **0.00 mm** | 16.1 mm* | 32.3 mm* |
| Back-prop | 18.9 mm† | 11.9 mm* | 46.5 mm* |

\* MAE dominated by 1λ and 2λ phase-wrap failures shared by all methods.  
† Single ½λ outlier (−94 mm); ¼λ–¹⁄₃₂λ errors are 0.06–0.32 mm.

**Gazdag** achieves essentially machine-precision errors on clean lateral data
(< 0.01 mm for ¼λ through ¹⁄₃₂λ).  Kirchhoff is nearly as accurate (~0.01–0.05 mm).
Back-propagation matches both methods for ¼λ and smaller, but shows an unexplained
large error at ½λ lateral that does not appear in the other movement types.

---

### Noisy synthetic data (Laplace noise, σ ≈ 10 % of signal std)

| Method | Lateral Δx (¼λ–¹⁄₃₂λ) | Vertical Δz (¼λ–¹⁄₃₂λ) | Fluid-flow (¼λ–¹⁄₃₂λ) |
|---|---|---|---|
| Kirchhoff | 0.1–24 mm (erratic) | 0.7–4.0 mm | **< 0.3 mm** |
| Gazdag | 2–29 mm (collapses) | 1–25 mm (collapses) | Structural failure (≈ −true Δx) |
| Back-prop | **< 0.2 mm** | **< 0.8 mm** | 0.5–1.2 mm |

**Gazdag collapses under noise** across all movement types.  The laterally-invariant
interface reflection dominates the cross-spectrum at kx = 0, suppressing the
scattered-wave signal and driving the estimated shift to zero.  This is a structural
limitation of the method, not a tuning issue.

**Back-propagation is the most robust to noise** for point-scatterer targets
(lateral and vertical studies): once the shift is ≤ ¼λ, errors remain below 0.8 mm
regardless of noise level — significantly better than Kirchhoff, which produces
coherent wave-like migration artefacts from the noise that corrupt the cross-spectrum.

**Kirchhoff is the most robust for fluid flow under noise**, where the extended
reflector geometry reduces sensitivity to the spatially-correlated artefacts it
creates.

---

### Overall recommendation

No single method is universally superior, but the ranking is clear in practice:

1. **Gazdag** — use for clean or pre-processed data where its zero-bias phase-shift
   extrapolation gives the highest accuracy.  Do not apply to raw, noise-contaminated
   B-scans without aggressive pre-filtering.

2. **Kirchhoff** — best practical choice for fluid-flow monitoring and noisy data in
   general; slightly biased on clean data but consistently useful across all scenarios.

3. **Back-propagation** — most noise-robust for detecting small point-scatterer
   displacements (¼λ and below); computationally expensive and sensitive to the ½λ
   regime, but the sign-bit excitation makes it viable even without amplitude
   normalisation.

## Section 4 — Spectral Phase Lines ΔΦ(kz)

Trace-by-trace **vertical windowed FFT** at the fracture depth to reveal three distinct
lateral physical regimes in the base-vs-monitor comparison.

For a single lateral trace $x$, a Hanning-windowed segment of width $\approx\lambda$ is
extracted around $z_\text{top}$ from both images.  The 1-D cross-spectrum phase reads:

$$\Delta\Phi(k_z)
  = \angle\!\Bigl[\mathcal{F}(B_\text{win})\cdot\mathcal{F}(M_\text{win})^*\Bigr]
  = -k_z\,\Delta z_\text{apparent} + \Delta\theta$$

where $\Delta z_\text{apparent}$ is the **slope** (apparent velocity pull-down) and
$\Delta\theta$ is the **intercept** (pure material-substitution phase).

| Zone | Location | Expected $\Delta\Phi(k_z)$ |
|---|---|---|
| **Air Zone** | $x > x_0+\Delta x$ — ahead of front, unchanged | flat $\approx 0°$ |
| **Front Zone** | $x \approx x_0+\Delta x$ — at advancing front | slope or offset emerging |
| **Water Zone** | $x_0 < x < x_0+\Delta x$ — newly flooded | flat non-zero $\Delta\theta$ |


## Section 5 — ΔΦ(f): Temporal-Frequency Spectral Line

The most direct verification of the governing equation in the **time domain**:

$$\Delta\Phi(f) = \underbrace{-2\pi f\,\Delta t}_{\text{slope: mechanical shift}} + \underbrace{\Delta\theta}_{\text{intercept: material change}}$$

The depth-domain migrated trace maps to TWT via $t = 2z/v_\text{ice}$, so taking the
windowed FFT of a migrated column is equivalent to computing the STFT of the raw A-scan
at the two-way time of the fracture reflection.  The wavenumber axis converts to
temporal frequency as:

$$f\;\text{[GHz]} = \frac{v_\text{ice}\;\text{[m/ns]}\;\cdot\;k_z\;\text{[rad/m]}}{2\pi}$$

A weighted linear regression $\Delta\Phi = A\cdot f + B$ then cleanly decouples:

| Fitted parameter | Physical quantity | Pure mechanical | Pure fluid |
|---|---|---|---|
| $A$ [°/GHz] | slope | $A = -360\,\Delta t$ [ns] $\neq 0$ | $A \approx 0$ |
| $B$ [°] | intercept $=\Delta\theta$ | $B \approx 0$ | $B \neq 0$ |

Three lateral zones are evaluated at the fracture depth: **Air** (unchanged), **Front**
(transition), and **Water** (newly flooded).


## Section 6 — Instantaneous Phase Analysis

Adapted from CWT_playground cells 32 & 33, applied to the imported Section 1 migrated
images.  Raw B-scans are not available, so synthesis and migration are skipped; rows 0
and 1 of the playground figure are omitted.

**2-D Riesz transform in z** gives the z-quadrature component, accounting for 2-D
spatial structure (unlike a 1-D Hilbert in z which ignores lateral wavenumber):

$$H_z\{W\}:\quad \hat{R}(k_z, k_x) = \frac{-i\,k_z}{|k|}\,\hat{W}(k_z, k_x)$$

Instantaneous amplitude and phase follow as:

$$A = \sqrt{W^2 + H_z^2}, \qquad \varphi = \arctan\!\left(\frac{H_z}{W}\right)$$

The **wrapped phase difference** uses the complex exponential to avoid wrap artefacts:

$$\Delta\varphi = \angle\!\left[e^{i(\varphi_\mathrm{mon} - \varphi_\mathrm{base})}\right] \in (-\pi, \pi]$$

An amplitude mask (threshold 8 % of peak) suppresses noise-dominated pixels.
Three figures are produced per (method, scenario):

1. **Phase images** — unmasked and masked $\varphi_\mathrm{base}$, $\varphi_\mathrm{mon}$, $\Delta\varphi$ (zoomed around scatterer)
2. **Cross-section at $z_\mathrm{scatterer}$** — amplitude profile + phase profiles (faded unmasked + solid masked)
3. **Zoomed cross-section** — ±½λ window around the two scatterer positions with Δφ zero-crossing detection and tangent slope


In [ ]:
# ─── Section 6: Instantaneous Phase Analysis ─────────────────────────────────
# Adapted from CWT_playground cells 32 & 33 (B-scan rows omitted).

def riesz_z(W):
    """
    2-D Riesz transform in z: H_z{W}(kz,kx) = -i kz/|k| * W(kz,kx).
    Accounts for 2-D spatial structure; reduces to the Hilbert transform in z
    only when kx = 0 (pure vertical structure).
    """
    nz, nx  = W.shape
    KX, KZ  = np.meshgrid(np.fft.fftfreq(nx), np.fft.fftfreq(nz))
    K       = np.sqrt(KX**2 + KZ**2)
    K[0, 0] = 1.0          # avoid division by zero at DC (DC contribution is zero anyway)
    return np.real(np.fft.ifft2((-1j * KZ / K) * np.fft.fft2(W)))


thresh_frac = 0.08    # amplitude mask: suppress pixels below 8 % of the peak

for method_name, base_stack in [
        ('Kirchhoff', kirchhoff),
        ('Gazdag',    gazdag),
        # ('Back-prop', backprop),
]:

    # ── Baseline phase — computed once per method ─────────────────────────────
    base_img = base_stack[0]
    R_base   = riesz_z(np.nan_to_num(base_img))
    A_base   = np.sqrt(base_img**2 + R_base**2)
    phi_base = np.arctan2(R_base, base_img)          # [-π, π]

    # x_s1[0] = baseline scatterer position (same index as cell 5 uses for i=0)
    x_sc0 = float(x_s1[0])
    ext   = [x_traces[0], x_traces[-1], z_img[-1], z_img[0]]

    hsv_cmap = plt.get_cmap('hsv').copy()
    hsv_cmap.set_bad(color='lightgrey')    # masked pixels → grey
    slope_results = []                     # accumulate per-scenario slope data

    for i in range(1, len(scenarios)):
        mon_img  = base_stack[i]
        name     = str(scenarios[i])
        true_dx  = float(separation_lambda[i]) * lam
        # x_s1[i] = monitor scatterer position for scenario i
        # (same as cell 5: ax.axvline(x_s1[i]) for panel i)
        x_sc_mon = float(x_s1[i])

        R_mon     = riesz_z(np.nan_to_num(mon_img))
        A_mon     = np.sqrt(mon_img**2 + R_mon**2)
        phi_mon   = np.arctan2(R_mon, mon_img)
        # Wrapped difference: avoids wrap artefacts at ±π boundary
        phi_delta = np.angle(np.exp(1j * (phi_mon - phi_base)))

        # Amplitude masks (global threshold — used for display)
        mask_base  = A_base < thresh_frac * A_base.max()
        mask_mon   = A_mon  < thresh_frac * A_mon.max()
        mask_delta = mask_base | mask_mon

        phi_base_msk  = np.where(mask_base,  np.nan, phi_base)
        phi_mon_msk   = np.where(mask_mon,   np.nan, phi_mon)
        phi_delta_msk = np.where(mask_delta, np.nan, phi_delta)

        # ── Zoom window: ±2λ around the scatterer pair ───────────────────────
        # Tighter than ±5λ-around-midpoint; for the 2λ scenario the old x_hi
        # was 2.335 m, which let the PSF left lobe of the static reference
        # scatterer (x_s2=2.338 m, PSF starts at ~2.226 m) bleed into Fig 2.
        x_cs_lo = min(x_sc0, x_sc_mon) - 2.0 * lam
        x_cs_hi = max(x_sc0, x_sc_mon) + 2.0 * lam
        z_lo    = z_scatterer - 0.08
        z_hi    = z_scatterer + 0.08

        def mig_zoom(ax):
            """
            Zoom + marker overlay matching cell 5 style:
              axvline(x_s1[0])  →  limegreen dotted  (baseline scatterer)
              axvline(x_s1[i])  →  tomato dotted     (monitor scatterer)
              axhline(z_scatterer) →  white dashed   (scatterer depth)
            """
            ax.set_xlim(x_cs_lo, x_cs_hi)
            ax.set_ylim(z_hi, z_lo)          # inverted: deeper at bottom
            ax.axvline(x_sc0,       color='limegreen', ls=':', lw=1.5, alpha=0.9,
                       label=f'base  x={x_sc0:.3f} m  (x_s1[0])')
            ax.axvline(x_sc_mon,    color='tomato',    ls=':', lw=1.5, alpha=0.9,
                       label=f'mon  x={x_sc_mon:.3f} m  (x_s1[{i}], {name})')
            ax.axhline(z_scatterer, color='white',     ls='--', lw=0.8, alpha=0.65,
                       label=f'z_sc = {z_scatterer*1e3:.0f} mm')
            if x_cs_lo < x_s2 < x_cs_hi:
                ax.axvline(x_s2, color='cyan', ls=':', lw=1.2, alpha=0.8,
                           label=f'ref  x_s2={x_s2:.3f} m')

        # ═══════════════════════════════════════════════════════════════════════
        # Figure 1: Instantaneous phase images (unmasked + masked)  [cell 32]
        # ═══════════════════════════════════════════════════════════════════════
        fig1, axes1 = plt.subplots(2, 3, figsize=(18, 10))
        fig1.suptitle(
            f"Instantaneous Phase — {method_name}  |  Baseline vs {name}"
            f"   (Δx = {true_dx*1e3:.1f} mm = {true_dx/lam:.4f}λ)",
            fontsize=12, fontweight='bold'
        )

        row0_data = [
            (phi_base,  r"Inst. phase  $\varphi_\mathrm{base}$  [unmasked]"),
            (phi_mon,   r"Inst. phase  $\varphi_\mathrm{mon}$  [unmasked]"),
            (phi_delta, r"Phase diff  $\Delta\varphi$  [wrapped, unmasked]"),
        ]
        row1_data = [
            (phi_base_msk,
             rf"$\varphi_\mathrm{{base}}$ masked  (|A| < {thresh_frac*100:.0f}% peak)"),
            (phi_mon_msk,
             rf"$\varphi_\mathrm{{mon}}$ masked  (|A| < {thresh_frac*100:.0f}% peak)"),
            (phi_delta_msk,
             rf"$\Delta\varphi$ masked  (either |A| < {thresh_frac*100:.0f}% peak)"),
        ]

        for ax, (data, title) in zip(axes1[0], row0_data):
            im = ax.imshow(data, aspect='auto', origin='upper', extent=ext,
                           cmap='hsv', vmin=-np.pi, vmax=np.pi,
                           interpolation='bilinear')
            ax.set_title(title, fontsize=10)
            ax.set_xlabel('x [m]');  ax.set_ylabel('z [m]')
            mig_zoom(ax);  ax.legend(fontsize=7, loc='lower right')
            fig1.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='[rad]')

        for ax, (data, title) in zip(axes1[1], row1_data):
            im = ax.imshow(data, aspect='auto', origin='upper', extent=ext,
                           cmap=hsv_cmap, vmin=-np.pi, vmax=np.pi,
                           interpolation='none')
            ax.set_title(title, fontsize=10)
            ax.set_xlabel('x [m]');  ax.set_ylabel('z [m]')
            mig_zoom(ax);  ax.legend(fontsize=7, loc='lower right')
            fig1.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='[rad]')

        plt.tight_layout()
        plt.show()

        # ═══════════════════════════════════════════════════════════════════════
        # Figure 2: Phase cross-section at z_scatterer  [cell 33, full range]
        # ═══════════════════════════════════════════════════════════════════════
        iz_sc = int(np.argmin(np.abs(z_img - z_scatterer)))

        p_A_base      = A_base[iz_sc, :]
        p_A_mon       = A_mon[iz_sc, :]
        p_phi_base    = phi_base[iz_sc, :]
        p_phi_mon     = phi_mon[iz_sc, :]
        p_phi_delta   = phi_delta[iz_sc, :]
        p_phi_b_msk   = phi_base_msk[iz_sc, :]
        p_phi_m_msk   = phi_mon_msk[iz_sc, :]
        p_phi_d_msk   = phi_delta_msk[iz_sc, :]

        # Local amplitude threshold for zero-crossing detection.
        # A_base.max() is dominated by the static reference scatterer at
        # x_s2=2.338 m; 8% of that global max silences the PSF overlap zone
        # between the two moving scatterers for the 1λ and 2λ cases.
        # Fix: use 5% of the LOCAL peak within the ±2λ scatterer-pair window.
        _ix_lo_sc = np.searchsorted(x_traces, x_cs_lo)
        _ix_hi_sc = np.searchsorted(x_traces, x_cs_hi)
        _A_loc    = max(float(A_base[iz_sc, _ix_lo_sc:_ix_hi_sc].max()),
                        float(A_mon[iz_sc,  _ix_lo_sc:_ix_hi_sc].max()))
        _thr_loc  = 0.05 * _A_loc if _A_loc > 0 else np.inf
        mask_cross    = (A_base[iz_sc, :] < _thr_loc) | (A_mon[iz_sc, :] < _thr_loc)
        p_phi_d_cross = np.where(mask_cross, np.nan, p_phi_delta)

        fig2, ax2 = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
        fig2.suptitle(
            f"Phase cross-section  z = {z_scatterer*1e3:.0f} mm  —  {method_name}  |  {name}"
            f"  (Δx = {true_dx*1e3:.1f} mm = {true_dx/lam:.4f}λ)",
            fontsize=12, fontweight='bold'
        )

        # Panel 0: Instantaneous amplitude
        ax2[0].plot(x_traces, p_A_base, 'b-',  lw=1.8, label='$A_\\mathrm{base}$')
        ax2[0].plot(x_traces, p_A_mon,  'r--', lw=1.8, label=f'$A_\\mathrm{{mon}}$ ({name})')
        ax2[0].axhline(thresh_frac * A_base.max(), color='b', ls=':', lw=1.0, alpha=0.7,
                       label=f'base thresh ({thresh_frac*100:.0f}% global peak)')
        ax2[0].axhline(thresh_frac * A_mon.max(),  color='r', ls=':', lw=1.0, alpha=0.7,
                       label=f'mon thresh ({thresh_frac*100:.0f}% global peak)')
        ax2[0].axhline(_thr_loc, color='purple', ls=':', lw=1.2, alpha=0.8,
                       label=f'local thresh (5% local peak = {_thr_loc:.3f})')
        ax2[0].axvline(x_sc0,    color='limegreen', ls=':', lw=1.5,
                       label=f'base  x={x_sc0:.3f} m')
        ax2[0].axvline(x_sc_mon, color='tomato',    ls=':', lw=1.5,
                       label=f'mon   x={x_sc_mon:.3f} m')
        ax2[0].set_xlim(x_cs_lo, x_cs_hi)
        ax2[0].set_ylabel('Instantaneous amplitude [a.u.]', fontsize=10)
        ax2[0].legend(fontsize=8, ncol=2, loc='upper right')
        ax2[0].grid(True, alpha=0.3)

        # Panel 1: Phase profiles (faded unmasked + solid masked)
        for arr, col, ls in [
            (p_phi_base, 'b', '-'), (p_phi_mon, 'r', '--'), (p_phi_delta, 'k', '-')
        ]:
            ax2[1].plot(x_traces, arr, color=col, ls=ls, lw=0.9, alpha=0.25)
        ax2[1].plot(x_traces, p_phi_b_msk,  'b-',  lw=2.2,
                    label=r'$\varphi_\mathrm{base}$')
        ax2[1].plot(x_traces, p_phi_m_msk,  'r--', lw=2.2,
                    label=rf'$\varphi_\mathrm{{mon}}$ ({name})')
        ax2[1].plot(x_traces, p_phi_d_msk,  'k-',  lw=2.2,
                    label=r'$\Delta\varphi$ (wrapped, 8% global)')
        ax2[1].axvline(x_sc0,    color='limegreen', ls=':', lw=1.5)
        ax2[1].axvline(x_sc_mon, color='tomato',    ls=':', lw=1.5)
        ax2[1].axhline(0, color='k', lw=0.5, alpha=0.5)
        ax2[1].set_ylim(-np.pi - 0.3, np.pi + 0.3)
        ax2[1].set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
        ax2[1].set_yticklabels([r'$-\pi$', r'$-\pi/2$', '0', r'$\pi/2$', r'$\pi$'],
                               fontsize=10)
        ax2[1].set_ylabel('Instantaneous phase [rad]', fontsize=10)
        ax2[1].set_xlabel('x [m]', fontsize=10)
        ax2[1].legend(fontsize=9, loc='upper right')
        ax2[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        # ═══════════════════════════════════════════════════════════════════════
        # Figure 3: Zoomed cross-section + Δφ zero-crossing  [cell 33, zoomed]
        # ═══════════════════════════════════════════════════════════════════════
        x_win_lo = min(x_sc0, x_sc_mon) - 0.5 * lam
        x_win_hi = max(x_sc0, x_sc_mon) + 0.5 * lam

        fig3, ax3 = plt.subplots(2, 1, figsize=(10, 10), sharex=True)
        fig3.suptitle(
            f"Phase cross-section — zoomed  (±½λ around scatterers)\n"
            f"{method_name}  |  {name}  (Δx = {true_dx*1e3:.1f} mm = {true_dx/lam:.4f}λ)",
            fontsize=12, fontweight='bold'
        )

        ax3[0].plot(x_traces, p_A_base, 'b-',  lw=1.8, label='$A_\\mathrm{base}$')
        ax3[0].plot(x_traces, p_A_mon,  'r--', lw=1.8,
                    label=f'$A_\\mathrm{{mon}}$ ({name})')
        ax3[0].axhline(thresh_frac * A_base.max(), color='b', ls=':', lw=1.0, alpha=0.7,
                       label='8% global thresh')
        ax3[0].axhline(thresh_frac * A_mon.max(),  color='r', ls=':', lw=1.0, alpha=0.7)
        ax3[0].axhline(_thr_loc, color='purple', ls=':', lw=1.2, alpha=0.8,
                       label=f'5% local thresh = {_thr_loc:.3f}')
        ax3[0].axvline(x_sc0,    color='limegreen', ls=':', lw=1.5,
                       label=f'base  x={x_sc0:.3f} m  (x_s1[0])')
        ax3[0].axvline(x_sc_mon, color='tomato',    ls=':', lw=1.5,
                       label=f'mon   x={x_sc_mon:.3f} m  (x_s1[{i}])')
        ax3[0].set_xlim(x_win_lo, x_win_hi)
        ax3[0].set_ylabel('Instantaneous amplitude [a.u.]', fontsize=10)
        ax3[0].legend(fontsize=9)
        ax3[0].grid(True, alpha=0.3)

        for arr, col, ls in [
            (p_phi_base,  'b', '-'), (p_phi_mon, 'r', '--'), (p_phi_delta, 'k', '-')
        ]:
            ax3[1].plot(x_traces, arr, color=col, ls=ls, lw=0.9, alpha=0.25)
        # Grey: locally masked Δφ (5% local threshold) — used for crossing detection
        ax3[1].plot(x_traces, p_phi_d_cross, color='grey', ls='-', lw=1.2, alpha=0.6,
                    zorder=2, label=r'$\Delta\varphi$ 5% local mask (crossing)')
        ax3[1].plot(x_traces, p_phi_b_msk, 'b-',  lw=2.5,
                    label=r'$\varphi_\mathrm{base}$')
        ax3[1].plot(x_traces, p_phi_m_msk, 'r--', lw=2.5,
                    label=rf'$\varphi_\mathrm{{mon}}$ ({name})')
        ax3[1].plot(x_traces, p_phi_d_msk, 'k-',  lw=2.5,
                    label=r'$\Delta\varphi$ 8% global mask')

        # Annotate phase values at each scatterer position
        for ix_pt, x_pt, col in [
            (int(np.argmin(np.abs(x_traces - x_sc0))),    x_sc0,    'limegreen'),
            (int(np.argmin(np.abs(x_traces - x_sc_mon))), x_sc_mon, 'tomato'),
        ]:
            for phi_arr, mk in [(p_phi_b_msk, 'o'), (p_phi_m_msk, 's')]:
                val = phi_arr[ix_pt]
                if not np.isnan(val):
                    ax3[1].plot(x_pt, val, mk, color=col, ms=8, zorder=6)
                    ax3[1].annotate(
                        f'{val:.2f} rad',
                        xy=(x_pt, val), xytext=(6, 6), textcoords='offset points',
                        fontsize=8, color=col,
                        arrowprops=dict(arrowstyle='->', color=col, lw=0.8),
                    )

        # Zero-crossing detection using the locally masked Δφ (p_phi_d_cross).
        # The grey curve uses a 5%-of-local-peak threshold that keeps the PSF
        # overlap zone between the two scatterers unmasked, even when the global
        # A.max() (dominated by the static x_s2 scatterer) would blank it out.
        win_mask  = (x_traces >= x_win_lo) & (x_traces <= x_win_hi)
        x_win_arr = x_traces[win_mask]
        d_win     = p_phi_d_cross[win_mask]
        x_zc, slope_zc = [], []
        for k in range(len(d_win) - 1):
            a, b = d_win[k], d_win[k + 1]
            if np.isnan(a) or np.isnan(b):
                continue
            if a * b < 0:                          # sign change → zero crossing
                t = a / (a - b)
                x_zc.append(x_win_arr[k] + t * (x_win_arr[k + 1] - x_win_arr[k]))
                slope_zc.append((b - a) / (x_win_arr[k + 1] - x_win_arr[k]))

        if x_zc:
            for xz, sl in zip(x_zc, slope_zc):
                ax3[0].axvline(xz, color='purple', ls='-', lw=2.0, alpha=0.8,
                               label=f'Δφ=0  x={xz:.4f} m')
                ax3[1].axvline(xz, color='purple', ls='-', lw=2.0, alpha=0.8)
                x_tan = np.array([xz - lam, xz + lam])
                ax3[1].plot(x_tan, sl * (x_tan - xz), color='orange', lw=1.8,
                            ls='--', label=f'slope = {sl:.1f} rad/m')
                ax3[1].annotate(
                    f'slope = {sl:.1f} rad/m',
                    xy=(xz, 0), xytext=(8, 18), textcoords='offset points',
                    fontsize=9, color='darkorange', fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color='darkorange', lw=1.0),
                )
            ax3[0].legend(fontsize=9)

        ax3[1].axvline(x_sc0,    color='limegreen', ls=':', lw=1.5)
        ax3[1].axvline(x_sc_mon, color='tomato',    ls=':', lw=1.5)
        ax3[1].axhline(0, color='k', lw=0.5, alpha=0.5)
        ax3[1].set_ylim(-np.pi - 0.3, np.pi + 0.3)
        ax3[1].set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
        ax3[1].set_yticklabels([r'$-\pi$', r'$-\pi/2$', '0', r'$\pi/2$', r'$\pi$'],
                               fontsize=10)
        ax3[1].set_ylabel('Instantaneous phase [rad]', fontsize=10)
        ax3[1].set_xlabel('x [m]', fontsize=10)
        ax3[1].legend(fontsize=9, loc='upper right')
        ax3[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        # Collect slope data for the summary plot in the next cell
        slope_results.append({
            'sep_lam': float(separation_lambda[i]),
            'name': name,
            'crossings': list(zip(x_zc, slope_zc))
        })

        # Numeric summary
        ix_b = int(np.argmin(np.abs(x_traces - x_sc0)))
        ix_m = int(np.argmin(np.abs(x_traces - x_sc_mon)))
        print(f"\n{method_name}  |  {name}  (Δx = {true_dx*1e3:.2f} mm = {true_dx/lam:.4f}λ)")
        print(f"  x_s1[0]  = {x_sc0:.4f} m  (baseline scatterer, same as cell 5 i=0)")
        print(f"  x_s1[{i}]  = {x_sc_mon:.4f} m  (monitor scatterer, same as cell 5 i={i})")
        print(f"  At base x={x_sc0:.4f} m: "
              f"phi_base={p_phi_base[ix_b]:.3f}  phi_mon={p_phi_mon[ix_b]:.3f}"
              f"  Dphi={p_phi_delta[ix_b]:.3f} rad")
        print(f"  At mon  x={x_sc_mon:.4f} m: "
              f"phi_base={p_phi_base[ix_m]:.3f}  phi_mon={p_phi_mon[ix_m]:.3f}"
              f"  Dphi={p_phi_delta[ix_m]:.3f} rad")
        for xz, sl in zip(x_zc, slope_zc):
            print(f"  Dphi=0 crossing: x={xz:.4f} m  slope={sl:.2f} rad/m"
                  f"  ({sl*lam:.3f} rad/lambda)")
        print(f"  local thresh = {_thr_loc:.4f}  (5% of local max {_A_loc:.4f})")
        print(f"  mask: base={mask_base.mean()*100:.1f}%  "
              f"mon={mask_mon.mean()*100:.1f}%  delta={mask_delta.mean()*100:.1f}%")

In [ ]:
# ─── Section 6 summary: Δφ zero-crossing slope vs. scatterer separation ──────
sr_sorted = sorted(slope_results, key=lambda r: r['sep_lam'])

sep_vals = np.array([r['sep_lam'] for r in sr_sorted])
names    = [r['name']             for r in sr_sorted]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Gazdag — Δφ zero-crossing slope vs. scatterer separation',
             fontsize=12, fontweight='bold')

_ref_rad_m   = 2 * np.pi / lam
_ref_rad_lam = 2 * np.pi

for ax, (ylabel, scale_fn, ref_val, ref_lbl, col) in zip(axes, [
    ('dΔφ/dx  [rad/m]',
     lambda sl: sl,
     _ref_rad_m,
     f'2π/λ = {_ref_rad_m:.1f} rad/m',
     'steelblue'),
    ('dΔφ/dx  [rad/λ]',
     lambda sl: sl * lam,
     _ref_rad_lam,
     '2π rad/λ',
     'darkorange'),
]):
    ax.axhline(ref_val, color='grey', ls='--', lw=1.2, alpha=0.7, label=ref_lbl)

    no_cross_added = False
    for r in sr_sorted:
        s = r['sep_lam']
        if r['crossings']:
            for xz, sl in r['crossings']:
                val = scale_fn(sl)
                ax.plot(s, val, 'o', ms=9, color=col, zorder=5)
                ax.annotate(f'{val:.2f}',
                            xy=(s, val), xytext=(5, 5),
                            textcoords='offset points', fontsize=8)
        else:
            lbl = 'no crossing' if not no_cross_added else None
            no_cross_added = True
            ax.plot(s, 0, 'x', ms=11, color='tomato', mew=2, zorder=5, label=lbl)
            ax.annotate(f'{r["name"]}\n(no crossing)',
                        xy=(s, 0), xytext=(0, 14), textcoords='offset points',
                        ha='center', fontsize=7.5, color='tomato')

    ax.set_xscale('log')
    ax.set_xticks(sep_vals)
    ax.set_xticklabels(names, fontsize=9)
    ax.set_xlabel('Scatterer separation', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nSlope summary (Gazdag):")
for r in sr_sorted:
    if r['crossings']:
        for xz, sl in r['crossings']:
            print(f"  {r['name']:8s}  ({r['sep_lam']:.5f}λ):  "
                  f"slope = {sl:+7.2f} rad/m  = {sl*lam:+.3f} rad/λ  @ x={xz:.4f} m")
    else:
        print(f"  {r['name']:8s}  ({r['sep_lam']:.5f}λ):  no zero-crossing detected")

## Section 6b — VerticalTimeLapse: Instantaneous Phase Analysis

Mirrors Section 6, with the cross-section axis swapped: here the scatterer moves in
**depth** ($\Delta z$) at a fixed lateral position, so phase is sampled along $z$ at
$x=x_\mathrm{scatterer}$ instead of along $x$ at $z=z_\mathrm{scatterer}$. The
2-D Riesz transform `riesz_z` and the $\Delta\varphi$ wrapping convention from Section 6
are reused unchanged — only the cross-section direction differs.

Three figures per (method, scenario), as in Section 6:

1. **Phase images** — unmasked and masked $\varphi_\mathrm{base}$, $\varphi_\mathrm{mon}$, $\Delta\varphi$ (zoomed around the scatterer column)
2. **Cross-section at $x=x_\mathrm{scatterer}$** — amplitude profile + phase profiles vs depth $z$
3. **Zoomed cross-section** — ±½λ window around the two scatterer depths with Δφ zero-crossing detection and tangent slope $d\Delta\varphi/dz$

In [ ]:
# ─── Section 6b: Instantaneous Phase Analysis — VerticalTimeLapse ────────────
# Mirrors Section 6; riesz_z(...) is reused as-is from that section.
#
# Note: unlike Section 6 (cross-section orthogonal to the depth axis that
# riesz_z builds its analytic signal in, giving a smooth Δφ ramp along x),
# here the cross-section runs ALONG that same depth axis — so φ(z) carries
# the fast carrier oscillation itself, and a Δφ=0 crossing/slope is not a
# stable quantity (mirrors CWT_playground cell 42, where the zero-crossing
# code is commented out for the vertical case). We report the mean Δφ
# between the two scatterer depths instead, as that playground cell does.

thresh_frac_vt = 0.08    # amplitude mask: suppress pixels below 8 % of the peak

for method_name, base_stack in [
        # ('Kirchhoff', vt_kirchhoff),
        ('Gazdag',    vt_gazdag),
        # ('Back-prop', vt_backprop),
]:

    # ── Baseline phase — computed once per method ─────────────────────────────
    base_img = base_stack[0]
    R_base   = riesz_z(np.nan_to_num(base_img))
    A_base   = np.sqrt(base_img**2 + R_base**2)
    phi_base = np.arctan2(R_base, base_img)          # [-π, π]

    # Lateral position is fixed across all scenarios in this study
    x_sc  = float(vt_x_scatterer)
    z_sc0 = float(vt_z_depths[0])          # baseline scatterer depth
    ext   = [vt_x_traces[0], vt_x_traces[-1], vt_z_img[-1], vt_z_img[0]]
    ix_sc = int(np.argmin(np.abs(vt_x_traces - x_sc)))

    hsv_cmap = plt.get_cmap('hsv').copy()
    hsv_cmap.set_bad(color='lightgrey')    # masked pixels → grey
    vt_dphi_results = []                   # accumulate per-scenario mean-Δφ data

    for i in range(1, len(vt_scenarios)):
        mon_img  = base_stack[i]
        name     = str(vt_scenarios[i])
        true_dz  = float(vt_shift_lambda[i]) * lam
        z_sc_mon = float(vt_z_depths[i])    # monitor scatterer depth for scenario i

        R_mon     = riesz_z(np.nan_to_num(mon_img))
        A_mon     = np.sqrt(mon_img**2 + R_mon**2)
        phi_mon   = np.arctan2(R_mon, mon_img)
        # Wrapped difference: avoids wrap artefacts at ±π boundary
        phi_delta = np.angle(np.exp(1j * (phi_mon - phi_base)))

        # Amplitude masks (global threshold — used for display)
        mask_base  = A_base < thresh_frac_vt * A_base.max()
        mask_mon   = A_mon  < thresh_frac_vt * A_mon.max()
        mask_delta = mask_base | mask_mon

        phi_base_msk  = np.where(mask_base,  np.nan, phi_base)
        phi_mon_msk   = np.where(mask_mon,   np.nan, phi_mon)
        phi_delta_msk = np.where(mask_delta, np.nan, phi_delta)

        # Mean Δφ between the two scatterer depths (robust analogue of the
        # zero-crossing slope used in Section 6 — see note above).
        mask_btw   = (vt_z_img >= min(z_sc0, z_sc_mon)) & (vt_z_img <= max(z_sc0, z_sc_mon))
        mean_dphi  = float(np.nanmean(phi_delta_msk[mask_btw, ix_sc]))

        # ── Zoom window: ±2λ in z around the two scatterer depths, narrow in x ──
        z_cs_lo = min(z_sc0, z_sc_mon) - 2.0 * lam
        z_cs_hi = max(z_sc0, z_sc_mon) + 2.0 * lam
        x_lo    = x_sc - 2.5 * lam
        x_hi    = x_sc + 2.5 * lam

        def mig_zoom(ax):
            """
            Zoom + marker overlay (vertical analogue of Section 6's mig_zoom):
              axhline(z_depths[0])  →  limegreen dotted  (baseline scatterer depth)
              axhline(z_depths[i])  →  tomato dotted     (monitor scatterer depth)
              axvline(x_scatterer)  →  white dashed      (fixed lateral position)
            """
            ax.set_xlim(x_lo, x_hi)
            ax.set_ylim(z_cs_hi, z_cs_lo)     # inverted: deeper at bottom
            ax.axhline(z_sc0,    color='limegreen', ls=':', lw=1.5, alpha=0.9,
                       label=f'base  z={z_sc0*1e3:.1f} mm  (z_depths[0])')
            ax.axhline(z_sc_mon, color='tomato',    ls=':', lw=1.5, alpha=0.9,
                       label=f'mon  z={z_sc_mon*1e3:.1f} mm  (z_depths[{i}], {name})')
            ax.axvline(x_sc,     color='white',     ls='--', lw=0.8, alpha=0.65,
                       label=f'x_sc = {x_sc*1e3:.0f} mm')

        # ═══════════════════════════════════════════════════════════════════════
        # Figure 1: Instantaneous phase images (unmasked + masked)
        # ═══════════════════════════════════════════════════════════════════════
        fig1, axes1 = plt.subplots(2, 3, figsize=(18, 10))
        fig1.suptitle(
            f"VerticalTimeLapse — Instantaneous Phase — {method_name}  |  Baseline vs {name}"
            f"   (Δz = {true_dz*1e3:.1f} mm = {true_dz/lam:.4f}λ)",
            fontsize=12, fontweight='bold'
        )

        row0_data = [
            (phi_base,  r"Inst. phase  $\varphi_\mathrm{base}$  [unmasked]"),
            (phi_mon,   r"Inst. phase  $\varphi_\mathrm{mon}$  [unmasked]"),
            (phi_delta, r"Phase diff  $\Delta\varphi$  [wrapped, unmasked]"),
        ]
        row1_data = [
            (phi_base_msk,
             rf"$\varphi_\mathrm{{base}}$ masked  (|A| < {thresh_frac_vt*100:.0f}% peak)"),
            (phi_mon_msk,
             rf"$\varphi_\mathrm{{mon}}$ masked  (|A| < {thresh_frac_vt*100:.0f}% peak)"),
            (phi_delta_msk,
             rf"$\Delta\varphi$ masked  (either |A| < {thresh_frac_vt*100:.0f}% peak)"),
        ]

        for ax, (data, title) in zip(axes1[0], row0_data):
            im = ax.imshow(data, aspect='auto', origin='upper', extent=ext,
                           cmap='hsv', vmin=-np.pi, vmax=np.pi,
                           interpolation='bilinear')
            ax.set_title(title, fontsize=10)
            ax.set_xlabel('x [m]');  ax.set_ylabel('z [m]')
            mig_zoom(ax);  ax.legend(fontsize=7, loc='lower right')
            fig1.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='[rad]')

        for ax, (data, title) in zip(axes1[1], row1_data):
            im = ax.imshow(data, aspect='auto', origin='upper', extent=ext,
                           cmap=hsv_cmap, vmin=-np.pi, vmax=np.pi,
                           interpolation='none')
            ax.set_title(title, fontsize=10)
            ax.set_xlabel('x [m]');  ax.set_ylabel('z [m]')
            mig_zoom(ax);  ax.legend(fontsize=7, loc='lower right')
            fig1.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='[rad]')

        plt.tight_layout()
        plt.show()

        # ═══════════════════════════════════════════════════════════════════════
        # Figure 2: Phase cross-section at x = x_scatterer (wide z window)
        # ═══════════════════════════════════════════════════════════════════════
        p_A_base      = A_base[:, ix_sc]
        p_A_mon       = A_mon[:, ix_sc]
        p_phi_base    = phi_base[:, ix_sc]
        p_phi_mon     = phi_mon[:, ix_sc]
        p_phi_delta   = phi_delta[:, ix_sc]
        p_phi_b_msk   = phi_base_msk[:, ix_sc]
        p_phi_m_msk   = phi_mon_msk[:, ix_sc]
        p_phi_d_msk   = phi_delta_msk[:, ix_sc]

        # Local amplitude threshold for display, using the LOCAL peak within
        # the ±2λ depth window (mirrors Section 6's local-vs-global threshold).
        _iz_lo_sc = np.searchsorted(vt_z_img, z_cs_lo)
        _iz_hi_sc = np.searchsorted(vt_z_img, z_cs_hi)
        _A_loc    = max(float(A_base[_iz_lo_sc:_iz_hi_sc, ix_sc].max()),
                        float(A_mon[_iz_lo_sc:_iz_hi_sc, ix_sc].max()))
        _thr_loc  = 0.05 * _A_loc if _A_loc > 0 else np.inf
        mask_cross    = (A_base[:, ix_sc] < _thr_loc) | (A_mon[:, ix_sc] < _thr_loc)
        p_phi_d_cross = np.where(mask_cross, np.nan, p_phi_delta)

        fig2, ax2 = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
        fig2.suptitle(
            f"Phase cross-section  x = {x_sc*1e3:.0f} mm  —  {method_name}  |  {name}"
            f"  (Δz = {true_dz*1e3:.1f} mm = {true_dz/lam:.4f}λ)",
            fontsize=12, fontweight='bold'
        )

        # Panel 0: Instantaneous amplitude
        ax2[0].plot(vt_z_img, p_A_base, 'b-',  lw=1.8, label='$A_\\mathrm{base}$')
        ax2[0].plot(vt_z_img, p_A_mon,  'r--', lw=1.8, label=f'$A_\\mathrm{{mon}}$ ({name})')
        ax2[0].axhline(thresh_frac_vt * A_base.max(), color='b', ls=':', lw=1.0, alpha=0.7,
                       label=f'base thresh ({thresh_frac_vt*100:.0f}% global peak)')
        ax2[0].axhline(thresh_frac_vt * A_mon.max(),  color='r', ls=':', lw=1.0, alpha=0.7,
                       label=f'mon thresh ({thresh_frac_vt*100:.0f}% global peak)')
        ax2[0].axhline(_thr_loc, color='purple', ls=':', lw=1.2, alpha=0.8,
                       label=f'local thresh (5% local peak = {_thr_loc:.3f})')
        ax2[0].axvline(z_sc0,    color='limegreen', ls=':', lw=1.5,
                       label=f'base  z={z_sc0*1e3:.1f} mm')
        ax2[0].axvline(z_sc_mon, color='tomato',    ls=':', lw=1.5,
                       label=f'mon   z={z_sc_mon*1e3:.1f} mm')
        ax2[0].set_xlim(z_cs_lo, z_cs_hi)
        ax2[0].set_ylabel('Instantaneous amplitude [a.u.]', fontsize=10)
        ax2[0].legend(fontsize=8, ncol=2, loc='upper right')
        ax2[0].grid(True, alpha=0.3)

        # Panel 1: Phase profiles (faded unmasked + solid masked)
        for arr, col, ls in [
            (p_phi_base, 'b', '-'), (p_phi_mon, 'r', '--'), (p_phi_delta, 'k', '-')
        ]:
            ax2[1].plot(vt_z_img, arr, color=col, ls=ls, lw=0.9, alpha=0.25)
        ax2[1].plot(vt_z_img, p_phi_b_msk,  'b-',  lw=2.2,
                    label=r'$\varphi_\mathrm{base}$')
        ax2[1].plot(vt_z_img, p_phi_m_msk,  'r--', lw=2.2,
                    label=rf'$\varphi_\mathrm{{mon}}$ ({name})')
        ax2[1].plot(vt_z_img, p_phi_d_msk,  'k-',  lw=2.2,
                    label=r'$\Delta\varphi$ (wrapped, 8% global)')
        ax2[1].axvline(z_sc0,    color='limegreen', ls=':', lw=1.5)
        ax2[1].axvline(z_sc_mon, color='tomato',    ls=':', lw=1.5)
        ax2[1].axhline(0, color='k', lw=0.5, alpha=0.5)
        ax2[1].axhline(mean_dphi, color='purple', ls='--', lw=1.2, alpha=0.8,
                       label=rf'$\langle\Delta\varphi\rangle$ = {mean_dphi:.3f} rad')
        ax2[1].set_ylim(-np.pi - 0.3, np.pi + 0.3)
        ax2[1].set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
        ax2[1].set_yticklabels([r'$-\pi$', r'$-\pi/2$', '0', r'$\pi/2$', r'$\pi$'],
                               fontsize=10)
        ax2[1].set_ylabel('Instantaneous phase [rad]', fontsize=10)
        ax2[1].set_xlabel('z [m]', fontsize=10)
        ax2[1].legend(fontsize=9, loc='upper right')
        ax2[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        # ═══════════════════════════════════════════════════════════════════════
        # Figure 3: Zoomed cross-section (±½λ around the two scatterer depths)
        # ═══════════════════════════════════════════════════════════════════════
        z_win_lo = min(z_sc0, z_sc_mon) - 0.5 * lam
        z_win_hi = max(z_sc0, z_sc_mon) + 0.5 * lam

        fig3, ax3 = plt.subplots(2, 1, figsize=(10, 10), sharex=True)
        fig3.suptitle(
            f"Phase cross-section — zoomed  (±½λ around scatterer depths)\n"
            f"{method_name}  |  {name}  (Δz = {true_dz*1e3:.1f} mm = {true_dz/lam:.4f}λ)",
            fontsize=12, fontweight='bold'
        )

        ax3[0].plot(vt_z_img, p_A_base, 'b-',  lw=1.8, label='$A_\\mathrm{base}$')
        ax3[0].plot(vt_z_img, p_A_mon,  'r--', lw=1.8,
                    label=f'$A_\\mathrm{{mon}}$ ({name})')
        ax3[0].axhline(thresh_frac_vt * A_base.max(), color='b', ls=':', lw=1.0, alpha=0.7,
                       label='8% global thresh')
        ax3[0].axhline(thresh_frac_vt * A_mon.max(),  color='r', ls=':', lw=1.0, alpha=0.7)
        ax3[0].axhline(_thr_loc, color='purple', ls=':', lw=1.2, alpha=0.8,
                       label=f'5% local thresh = {_thr_loc:.3f}')
        ax3[0].axvline(z_sc0,    color='limegreen', ls=':', lw=1.5,
                       label=f'base  z={z_sc0*1e3:.1f} mm  (z_depths[0])')
        ax3[0].axvline(z_sc_mon, color='tomato',    ls=':', lw=1.5,
                       label=f'mon   z={z_sc_mon*1e3:.1f} mm  (z_depths[{i}])')
        ax3[0].set_xlim(z_win_lo, z_win_hi)
        ax3[0].set_ylabel('Instantaneous amplitude [a.u.]', fontsize=10)
        ax3[0].legend(fontsize=9)
        ax3[0].grid(True, alpha=0.3)

        for arr, col, ls in [
            (p_phi_base,  'b', '-'), (p_phi_mon, 'r', '--'), (p_phi_delta, 'k', '-')
        ]:
            ax3[1].plot(vt_z_img, arr, color=col, ls=ls, lw=0.9, alpha=0.25)
        # Grey: locally masked Δφ (5% local threshold)
        ax3[1].plot(vt_z_img, p_phi_d_cross, color='grey', ls='-', lw=1.2, alpha=0.6,
                    zorder=2, label=r'$\Delta\varphi$ 5% local mask')
        ax3[1].plot(vt_z_img, p_phi_b_msk, 'b-',  lw=2.5,
                    label=r'$\varphi_\mathrm{base}$')
        ax3[1].plot(vt_z_img, p_phi_m_msk, 'r--', lw=2.5,
                    label=rf'$\varphi_\mathrm{{mon}}$ ({name})')
        ax3[1].plot(vt_z_img, p_phi_d_msk, 'k-',  lw=2.5,
                    label=r'$\Delta\varphi$ 8% global mask')

        # Annotate phase values at each scatterer depth
        for iz_pt, z_pt, col in [
            (int(np.argmin(np.abs(vt_z_img - z_sc0))),    z_sc0,    'limegreen'),
            (int(np.argmin(np.abs(vt_z_img - z_sc_mon))), z_sc_mon, 'tomato'),
        ]:
            for phi_arr, mk in [(p_phi_b_msk, 'o'), (p_phi_m_msk, 's')]:
                val = phi_arr[iz_pt]
                if not np.isnan(val):
                    ax3[1].plot(z_pt, val, mk, color=col, ms=8, zorder=6)
                    ax3[1].annotate(
                        f'{val:.2f} rad',
                        xy=(z_pt, val), xytext=(6, 6), textcoords='offset points',
                        fontsize=8, color=col,
                        arrowprops=dict(arrowstyle='->', color=col, lw=0.8),
                    )

        ax3[1].axhline(mean_dphi, color='purple', ls='--', lw=1.2, alpha=0.8,
                       label=rf'$\langle\Delta\varphi\rangle$ = {mean_dphi:.3f} rad')
        ax3[1].axvline(z_sc0,    color='limegreen', ls=':', lw=1.5)
        ax3[1].axvline(z_sc_mon, color='tomato',    ls=':', lw=1.5)
        ax3[1].axhline(0, color='k', lw=0.5, alpha=0.5)
        ax3[1].set_ylim(-np.pi - 0.3, np.pi + 0.3)
        ax3[1].set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
        ax3[1].set_yticklabels([r'$-\pi$', r'$-\pi/2$', '0', r'$\pi/2$', r'$\pi$'],
                               fontsize=10)
        ax3[1].set_ylabel('Instantaneous phase [rad]', fontsize=10)
        ax3[1].set_xlabel('z [m]', fontsize=10)
        ax3[1].legend(fontsize=9, loc='upper right')
        ax3[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        # Collect mean-Δφ data for the summary plot in the next cell
        vt_dphi_results.append({
            'shift_lam': float(vt_shift_lambda[i]),
            'name': name,
            'mean_dphi': mean_dphi,
        })

        # Numeric summary
        iz_b = int(np.argmin(np.abs(vt_z_img - z_sc0)))
        iz_m = int(np.argmin(np.abs(vt_z_img - z_sc_mon)))
        print(f"\n{method_name}  |  {name}  (Δz = {true_dz*1e3:.2f} mm = {true_dz/lam:.4f}λ)")
        print(f"  z_depths[0]  = {z_sc0:.4f} m  (baseline scatterer depth)")
        print(f"  z_depths[{i}]  = {z_sc_mon:.4f} m  (monitor scatterer depth)")
        print(f"  At base z={z_sc0:.4f} m: "
              f"phi_base={p_phi_base[iz_b]:.3f}  phi_mon={p_phi_mon[iz_b]:.3f}"
              f"  Dphi={p_phi_delta[iz_b]:.3f} rad")
        print(f"  At mon  z={z_sc_mon:.4f} m: "
              f"phi_base={p_phi_base[iz_m]:.3f}  phi_mon={p_phi_mon[iz_m]:.3f}"
              f"  Dphi={p_phi_delta[iz_m]:.3f} rad")
        print(f"  Mean Δφ between z_depths[0] and z_depths[{i}]: {mean_dphi:.4f} rad"
              f"  ({mean_dphi/(2*np.pi):.4f} cycles)")
        print(f"  local thresh = {_thr_loc:.4f}  (5% of local max {_A_loc:.4f})")
        print(f"  mask: base={mask_base.mean()*100:.1f}%  "
              f"mon={mask_mon.mean()*100:.1f}%  delta={mask_delta.mean()*100:.1f}%")

In [ ]:
# ─── Section 6b summary: mean Δφ vs. vertical shift ──────────────────────────
vt_sr_sorted = sorted(vt_dphi_results, key=lambda r: r['shift_lam'])

shift_vals = np.array([r['shift_lam']  for r in vt_sr_sorted])
mean_dphi_vals = np.array([r['mean_dphi'] for r in vt_sr_sorted])
vt_names   = [r['name']                for r in vt_sr_sorted]

# Ideal (wrapped) phase shift for a pure depth shift dz: kz_c * dz, wrapped to (-π, π]
shift_dense  = np.linspace(0, shift_vals.max(), 400)
ideal_dense  = np.angle(np.exp(1j * 2 * np.pi * shift_dense))
ideal_points = np.angle(np.exp(1j * 2 * np.pi * shift_vals))

fig, ax = plt.subplots(figsize=(7, 5))
fig.suptitle('VerticalTimeLapse — Gazdag — Mean Δφ vs. scatterer vertical shift',
             fontsize=12, fontweight='bold')

ax.plot(shift_dense, ideal_dense, color='grey', ls='--', lw=1.2, alpha=0.7,
        label=r'ideal: $\angle e^{i\,2\pi\,\mathrm{shift}/\lambda}$')
ax.plot(shift_vals, mean_dphi_vals, 'o-', ms=9, color='steelblue', zorder=5,
        label=r'measured $\langle\Delta\varphi\rangle$')
for s, v in zip(shift_vals, mean_dphi_vals):
    ax.annotate(f'{v:.2f}', xy=(s, v), xytext=(5, 5),
                textcoords='offset points', fontsize=8)

ax.set_xscale('log')
ax.set_xticks(shift_vals)
ax.set_xticklabels(vt_names, fontsize=9)
ax.set_xlabel('Scatterer vertical shift', fontsize=11)
ax.set_ylabel(r'$\langle\Delta\varphi\rangle$  [rad]', fontsize=11)
ax.set_ylim(-np.pi - 0.3, np.pi + 0.3)
ax.set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
ax.set_yticklabels([r'$-\pi$', r'$-\pi/2$', '0', r'$\pi/2$', r'$\pi$'])
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nMean Δφ summary (Gazdag, vertical shift):")
for r, ideal in zip(vt_sr_sorted, ideal_points):
    print(f"  {r['name']:8s}  ({r['shift_lam']:.5f}λ):  "
          f"measured = {r['mean_dphi']:+.4f} rad   ideal = {ideal:+.4f} rad   "
          f"err = {r['mean_dphi'] - ideal:+.4f} rad")

## Section 6c — NoisyTimeLapse: Instantaneous Phase Analysis

Mirrors Section 6 (riesz_z(...) reused as-is), applied to the noisy migrated images
loaded in Section 2b (`noisy_kirchhoff`, `noisy_gazdag`).

**Kirchhoff only.** As in Section 2c, Gazdag is excluded for the noisy dataset — its
noisy migrated baseline carries a systematic structural artifact near x≈3.04m
(unrelated to the moving scatterer) that survives even amplitude masking, so its
instantaneous-phase maps would be analysing an artifact rather than noise robustness.
That issue needs to be fixed in the Gazdag migration itself (see Section 2c markdown),
not patched around here.


In [ ]:
# ─── Section 6c: Instantaneous Phase Analysis — NoisyTimeLapse ───────────────
# Mirrors Section 6; riesz_z(...) is reused as-is from that section.

thresh_frac_noisy = 0.08    # amplitude mask: suppress pixels below 8 % of the peak

for method_name, base_stack in [
        ('Kirchhoff', noisy_kirchhoff),
        # ('Gazdag', noisy_gazdag),  # excluded — structural migration artifact, see Section 2c
]:

    # ── Baseline phase — computed once per method ─────────────────────────────
    base_img = base_stack[0]
    R_base   = riesz_z(np.nan_to_num(base_img))
    A_base   = np.sqrt(base_img**2 + R_base**2)
    phi_base = np.arctan2(R_base, base_img)          # [-π, π]

    x_sc0 = float(noisy_x_s1[0])
    ext   = [noisy_x_traces[0], noisy_x_traces[-1], noisy_z_img[-1], noisy_z_img[0]]

    hsv_cmap = plt.get_cmap('hsv').copy()
    hsv_cmap.set_bad(color='lightgrey')    # masked pixels → grey
    slope_results_noisy = []               # accumulate per-scenario slope data

    for i in range(1, len(noisy_scenarios)):
        mon_img  = base_stack[i]
        name     = str(noisy_scenarios[i])
        true_dx  = float(noisy_separation_lambda[i]) * lam
        x_sc_mon = float(noisy_x_s1[i])

        R_mon     = riesz_z(np.nan_to_num(mon_img))
        A_mon     = np.sqrt(mon_img**2 + R_mon**2)
        phi_mon   = np.arctan2(R_mon, mon_img)
        # Wrapped difference: avoids wrap artefacts at ±π boundary
        phi_delta = np.angle(np.exp(1j * (phi_mon - phi_base)))

        # Amplitude masks (global threshold — used for display)
        mask_base  = A_base < thresh_frac_noisy * A_base.max()
        mask_mon   = A_mon  < thresh_frac_noisy * A_mon.max()
        mask_delta = mask_base | mask_mon

        phi_base_msk  = np.where(mask_base,  np.nan, phi_base)
        phi_mon_msk   = np.where(mask_mon,   np.nan, phi_mon)
        phi_delta_msk = np.where(mask_delta, np.nan, phi_delta)

        # ── Zoom window: ±2λ around the scatterer pair ───────────────────────
        x_cs_lo = min(x_sc0, x_sc_mon) - 2.0 * lam
        x_cs_hi = max(x_sc0, x_sc_mon) + 2.0 * lam
        z_lo    = noisy_z_scatterer - 0.08
        z_hi    = noisy_z_scatterer + 0.08

        def mig_zoom(ax):
            ax.set_xlim(x_cs_lo, x_cs_hi)
            ax.set_ylim(z_hi, z_lo)          # inverted: deeper at bottom
            ax.axvline(x_sc0,       color='limegreen', ls=':', lw=1.5, alpha=0.9,
                       label=f'base  x={x_sc0:.3f} m  (noisy_x_s1[0])')
            ax.axvline(x_sc_mon,    color='tomato',    ls=':', lw=1.5, alpha=0.9,
                       label=f'mon  x={x_sc_mon:.3f} m  (noisy_x_s1[{i}], {name})')
            ax.axhline(noisy_z_scatterer, color='white', ls='--', lw=0.8, alpha=0.65,
                       label=f'z_sc = {noisy_z_scatterer*1e3:.0f} mm')
            if x_cs_lo < noisy_x_s2 < x_cs_hi:
                ax.axvline(noisy_x_s2, color='cyan', ls=':', lw=1.2, alpha=0.8,
                           label=f'ref  x_s2={noisy_x_s2:.3f} m')

        # ═══════════════════════════════════════════════════════════════════════
        # Figure 1: Instantaneous phase images (unmasked + masked)
        # ═══════════════════════════════════════════════════════════════════════
        fig1, axes1 = plt.subplots(2, 3, figsize=(18, 10))
        fig1.suptitle(
            f"NoisyTimeLapse — Instantaneous Phase — {method_name}  |  Baseline vs {name}"
            f"   (Δx = {true_dx*1e3:.1f} mm = {true_dx/lam:.4f}λ, noise_level={noisy_noise_level})",
            fontsize=12, fontweight='bold'
        )

        row0_data = [
            (phi_base,  r"Inst. phase  $\varphi_\mathrm{base}$  [unmasked]"),
            (phi_mon,   r"Inst. phase  $\varphi_\mathrm{mon}$  [unmasked]"),
            (phi_delta, r"Phase diff  $\Delta\varphi$  [wrapped, unmasked]"),
        ]
        row1_data = [
            (phi_base_msk,
             rf"$\varphi_\mathrm{{base}}$ masked  (|A| < {thresh_frac_noisy*100:.0f}% peak)"),
            (phi_mon_msk,
             rf"$\varphi_\mathrm{{mon}}$ masked  (|A| < {thresh_frac_noisy*100:.0f}% peak)"),
            (phi_delta_msk,
             rf"$\Delta\varphi$ masked  (either |A| < {thresh_frac_noisy*100:.0f}% peak)"),
        ]

        for ax, (data, title) in zip(axes1[0], row0_data):
            im = ax.imshow(data, aspect='auto', origin='upper', extent=ext,
                           cmap='hsv', vmin=-np.pi, vmax=np.pi,
                           interpolation='bilinear')
            ax.set_title(title, fontsize=10)
            ax.set_xlabel('x [m]');  ax.set_ylabel('z [m]')
            mig_zoom(ax);  ax.legend(fontsize=7, loc='lower right')
            fig1.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='[rad]')

        for ax, (data, title) in zip(axes1[1], row1_data):
            im = ax.imshow(data, aspect='auto', origin='upper', extent=ext,
                           cmap=hsv_cmap, vmin=-np.pi, vmax=np.pi,
                           interpolation='none')
            ax.set_title(title, fontsize=10)
            ax.set_xlabel('x [m]');  ax.set_ylabel('z [m]')
            mig_zoom(ax);  ax.legend(fontsize=7, loc='lower right')
            fig1.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='[rad]')

        plt.tight_layout()
        plt.show()

        # ═══════════════════════════════════════════════════════════════════════
        # Figure 2: Phase cross-section at z_scatterer
        # ═══════════════════════════════════════════════════════════════════════
        iz_sc = int(np.argmin(np.abs(noisy_z_img - noisy_z_scatterer)))

        p_A_base      = A_base[iz_sc, :]
        p_A_mon       = A_mon[iz_sc, :]
        p_phi_base    = phi_base[iz_sc, :]
        p_phi_mon     = phi_mon[iz_sc, :]
        p_phi_delta   = phi_delta[iz_sc, :]
        p_phi_b_msk   = phi_base_msk[iz_sc, :]
        p_phi_m_msk   = phi_mon_msk[iz_sc, :]
        p_phi_d_msk   = phi_delta_msk[iz_sc, :]

        # Local amplitude threshold (5% of local peak within ±2λ window) — see
        # Section 6 for why the global 8% threshold is too aggressive here.
        _ix_lo_sc = np.searchsorted(noisy_x_traces, x_cs_lo)
        _ix_hi_sc = np.searchsorted(noisy_x_traces, x_cs_hi)
        _A_loc    = max(float(A_base[iz_sc, _ix_lo_sc:_ix_hi_sc].max()),
                        float(A_mon[iz_sc,  _ix_lo_sc:_ix_hi_sc].max()))
        _thr_loc  = 0.05 * _A_loc if _A_loc > 0 else np.inf
        mask_cross    = (A_base[iz_sc, :] < _thr_loc) | (A_mon[iz_sc, :] < _thr_loc)
        p_phi_d_cross = np.where(mask_cross, np.nan, p_phi_delta)

        fig2, ax2 = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
        fig2.suptitle(
            f"NoisyTimeLapse — Phase cross-section  z = {noisy_z_scatterer*1e3:.0f} mm  —  {method_name}  |  {name}"
            f"  (Δx = {true_dx*1e3:.1f} mm = {true_dx/lam:.4f}λ)",
            fontsize=12, fontweight='bold'
        )

        ax2[0].plot(noisy_x_traces, p_A_base, 'b-',  lw=1.8, label='$A_\\mathrm{base}$')
        ax2[0].plot(noisy_x_traces, p_A_mon,  'r--', lw=1.8, label=f'$A_\\mathrm{{mon}}$ ({name})')
        ax2[0].axhline(thresh_frac_noisy * A_base.max(), color='b', ls=':', lw=1.0, alpha=0.7,
                       label=f'base thresh ({thresh_frac_noisy*100:.0f}% global peak)')
        ax2[0].axhline(thresh_frac_noisy * A_mon.max(),  color='r', ls=':', lw=1.0, alpha=0.7,
                       label=f'mon thresh ({thresh_frac_noisy*100:.0f}% global peak)')
        ax2[0].axhline(_thr_loc, color='purple', ls=':', lw=1.2, alpha=0.8,
                       label=f'local thresh (5% local peak = {_thr_loc:.3f})')
        ax2[0].axvline(x_sc0,    color='limegreen', ls=':', lw=1.5,
                       label=f'base  x={x_sc0:.3f} m')
        ax2[0].axvline(x_sc_mon, color='tomato',    ls=':', lw=1.5,
                       label=f'mon   x={x_sc_mon:.3f} m')
        ax2[0].set_xlim(x_cs_lo, x_cs_hi)
        ax2[0].set_ylabel('Instantaneous amplitude [a.u.]', fontsize=10)
        ax2[0].legend(fontsize=8, ncol=2, loc='upper right')
        ax2[0].grid(True, alpha=0.3)

        for arr, col, ls in [
            (p_phi_base, 'b', '-'), (p_phi_mon, 'r', '--'), (p_phi_delta, 'k', '-')
        ]:
            ax2[1].plot(noisy_x_traces, arr, color=col, ls=ls, lw=0.9, alpha=0.25)
        ax2[1].plot(noisy_x_traces, p_phi_b_msk,  'b-',  lw=2.2,
                    label=r'$\varphi_\mathrm{base}$')
        ax2[1].plot(noisy_x_traces, p_phi_m_msk,  'r--', lw=2.2,
                    label=rf'$\varphi_\mathrm{{mon}}$ ({name})')
        ax2[1].plot(noisy_x_traces, p_phi_d_msk,  'k-',  lw=2.2,
                    label=r'$\Delta\varphi$ (wrapped, 8% global)')
        ax2[1].axvline(x_sc0,    color='limegreen', ls=':', lw=1.5)
        ax2[1].axvline(x_sc_mon, color='tomato',    ls=':', lw=1.5)
        ax2[1].axhline(0, color='k', lw=0.5, alpha=0.5)
        ax2[1].set_ylim(-np.pi - 0.3, np.pi + 0.3)
        ax2[1].set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
        ax2[1].set_yticklabels([r'$-\pi$', r'$-\pi/2$', '0', r'$\pi/2$', r'$\pi$'],
                               fontsize=10)
        ax2[1].set_ylabel('Instantaneous phase [rad]', fontsize=10)
        ax2[1].set_xlabel('x [m]', fontsize=10)
        ax2[1].legend(fontsize=9, loc='upper right')
        ax2[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        # ═══════════════════════════════════════════════════════════════════════
        # Figure 3: Zoomed cross-section + Δφ zero-crossing
        # ═══════════════════════════════════════════════════════════════════════
        x_win_lo = min(x_sc0, x_sc_mon) - 0.5 * lam
        x_win_hi = max(x_sc0, x_sc_mon) + 0.5 * lam

        fig3, ax3 = plt.subplots(2, 1, figsize=(10, 10), sharex=True)
        fig3.suptitle(
            f"NoisyTimeLapse — Phase cross-section — zoomed  (±½λ around scatterers)\n"
            f"{method_name}  |  {name}  (Δx = {true_dx*1e3:.1f} mm = {true_dx/lam:.4f}λ)",
            fontsize=12, fontweight='bold'
        )

        ax3[0].plot(noisy_x_traces, p_A_base, 'b-',  lw=1.8, label='$A_\\mathrm{base}$')
        ax3[0].plot(noisy_x_traces, p_A_mon,  'r--', lw=1.8,
                    label=f'$A_\\mathrm{{mon}}$ ({name})')
        ax3[0].axhline(thresh_frac_noisy * A_base.max(), color='b', ls=':', lw=1.0, alpha=0.7,
                       label='8% global thresh')
        ax3[0].axhline(thresh_frac_noisy * A_mon.max(),  color='r', ls=':', lw=1.0, alpha=0.7)
        ax3[0].axhline(_thr_loc, color='purple', ls=':', lw=1.2, alpha=0.8,
                       label=f'5% local thresh = {_thr_loc:.3f}')
        ax3[0].axvline(x_sc0,    color='limegreen', ls=':', lw=1.5,
                       label=f'base  x={x_sc0:.3f} m  (noisy_x_s1[0])')
        ax3[0].axvline(x_sc_mon, color='tomato',    ls=':', lw=1.5,
                       label=f'mon   x={x_sc_mon:.3f} m  (noisy_x_s1[{i}])')
        ax3[0].set_xlim(x_win_lo, x_win_hi)
        ax3[0].set_ylabel('Instantaneous amplitude [a.u.]', fontsize=10)
        ax3[0].legend(fontsize=9)
        ax3[0].grid(True, alpha=0.3)

        for arr, col, ls in [
            (p_phi_base,  'b', '-'), (p_phi_mon, 'r', '--'), (p_phi_delta, 'k', '-')
        ]:
            ax3[1].plot(noisy_x_traces, arr, color=col, ls=ls, lw=0.9, alpha=0.25)
        ax3[1].plot(noisy_x_traces, p_phi_d_cross, color='grey', ls='-', lw=1.2, alpha=0.6,
                    zorder=2, label=r'$\Delta\varphi$ 5% local mask (crossing)')
        ax3[1].plot(noisy_x_traces, p_phi_b_msk, 'b-',  lw=2.5,
                    label=r'$\varphi_\mathrm{base}$')
        ax3[1].plot(noisy_x_traces, p_phi_m_msk, 'r--', lw=2.5,
                    label=rf'$\varphi_\mathrm{{mon}}$ ({name})')
        ax3[1].plot(noisy_x_traces, p_phi_d_msk, 'k-',  lw=2.5,
                    label=r'$\Delta\varphi$ 8% global mask')

        for ix_pt, x_pt, col in [
            (int(np.argmin(np.abs(noisy_x_traces - x_sc0))),    x_sc0,    'limegreen'),
            (int(np.argmin(np.abs(noisy_x_traces - x_sc_mon))), x_sc_mon, 'tomato'),
        ]:
            for phi_arr, mk_ in [(p_phi_b_msk, 'o'), (p_phi_m_msk, 's')]:
                val = phi_arr[ix_pt]
                if not np.isnan(val):
                    ax3[1].plot(x_pt, val, mk_, color=col, ms=8, zorder=6)
                    ax3[1].annotate(
                        f'{val:.2f} rad',
                        xy=(x_pt, val), xytext=(6, 6), textcoords='offset points',
                        fontsize=8, color=col,
                        arrowprops=dict(arrowstyle='->', color=col, lw=0.8),
                    )

        win_mask  = (noisy_x_traces >= x_win_lo) & (noisy_x_traces <= x_win_hi)
        x_win_arr = noisy_x_traces[win_mask]
        d_win     = p_phi_d_cross[win_mask]
        x_zc, slope_zc = [], []
        for k in range(len(d_win) - 1):
            a, b = d_win[k], d_win[k + 1]
            if np.isnan(a) or np.isnan(b):
                continue
            if a * b < 0:                          # sign change → zero crossing
                t = a / (a - b)
                x_zc.append(x_win_arr[k] + t * (x_win_arr[k + 1] - x_win_arr[k]))
                slope_zc.append((b - a) / (x_win_arr[k + 1] - x_win_arr[k]))

        if x_zc:
            for xz, sl in zip(x_zc, slope_zc):
                ax3[0].axvline(xz, color='purple', ls='-', lw=2.0, alpha=0.8,
                               label=f'Δφ=0  x={xz:.4f} m')
                ax3[1].axvline(xz, color='purple', ls='-', lw=2.0, alpha=0.8)
                x_tan = np.array([xz - lam, xz + lam])
                ax3[1].plot(x_tan, sl * (x_tan - xz), color='orange', lw=1.8,
                            ls='--', label=f'slope = {sl:.1f} rad/m')
                ax3[1].annotate(
                    f'slope = {sl:.1f} rad/m',
                    xy=(xz, 0), xytext=(8, 18), textcoords='offset points',
                    fontsize=9, color='darkorange', fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color='darkorange', lw=1.0),
                )
            ax3[0].legend(fontsize=9)

        ax3[1].axvline(x_sc0,    color='limegreen', ls=':', lw=1.5)
        ax3[1].axvline(x_sc_mon, color='tomato',    ls=':', lw=1.5)
        ax3[1].axhline(0, color='k', lw=0.5, alpha=0.5)
        ax3[1].set_ylim(-np.pi - 0.3, np.pi + 0.3)
        ax3[1].set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
        ax3[1].set_yticklabels([r'$-\pi$', r'$-\pi/2$', '0', r'$\pi/2$', r'$\pi$'],
                               fontsize=10)
        ax3[1].set_ylabel('Instantaneous phase [rad]', fontsize=10)
        ax3[1].set_xlabel('x [m]', fontsize=10)
        ax3[1].legend(fontsize=9, loc='upper right')
        ax3[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        slope_results_noisy.append({
            'sep_lam': float(noisy_separation_lambda[i]),
            'name': name,
            'crossings': list(zip(x_zc, slope_zc))
        })

        ix_b = int(np.argmin(np.abs(noisy_x_traces - x_sc0)))
        ix_m = int(np.argmin(np.abs(noisy_x_traces - x_sc_mon)))
        print(f"\n{method_name}  |  {name}  (Δx = {true_dx*1e3:.2f} mm = {true_dx/lam:.4f}λ)")
        print(f"  noisy_x_s1[0]  = {x_sc0:.4f} m  (baseline scatterer)")
        print(f"  noisy_x_s1[{i}]  = {x_sc_mon:.4f} m  (monitor scatterer)")
        print(f"  At base x={x_sc0:.4f} m: "
              f"phi_base={p_phi_base[ix_b]:.3f}  phi_mon={p_phi_mon[ix_b]:.3f}"
              f"  Dphi={p_phi_delta[ix_b]:.3f} rad")
        print(f"  At mon  x={x_sc_mon:.4f} m: "
              f"phi_base={p_phi_base[ix_m]:.3f}  phi_mon={p_phi_mon[ix_m]:.3f}"
              f"  Dphi={p_phi_delta[ix_m]:.3f} rad")
        for xz, sl in zip(x_zc, slope_zc):
            print(f"  Dphi=0 crossing: x={xz:.4f} m  slope={sl:.2f} rad/m"
                  f"  ({sl*lam:.3f} rad/lambda)")
        print(f"  local thresh = {_thr_loc:.4f}  (5% of local max {_A_loc:.4f})")
        print(f"  mask: base={mask_base.mean()*100:.1f}%  "
              f"mon={mask_mon.mean()*100:.1f}%  delta={mask_delta.mean()*100:.1f}%")


In [ ]:
# ─── Section 6c summary: Δφ zero-crossing slope vs. scatterer separation ─────
sr_sorted_noisy = sorted(slope_results_noisy, key=lambda r: r['sep_lam'])

sep_vals_noisy = np.array([r['sep_lam'] for r in sr_sorted_noisy])
names_noisy    = [r['name']             for r in sr_sorted_noisy]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('NoisyTimeLapse (Kirchhoff) — Δφ zero-crossing slope vs. scatterer separation',
             fontsize=12, fontweight='bold')

_ref_rad_m   = 2 * np.pi / lam
_ref_rad_lam = 2 * np.pi

for ax, (ylabel, scale_fn, ref_val, ref_lbl, col) in zip(axes, [
    ('dΔφ/dx  [rad/m]',
     lambda sl: sl,
     _ref_rad_m,
     f'2π/λ = {_ref_rad_m:.1f} rad/m',
     'steelblue'),
    ('dΔφ/dx  [rad/λ]',
     lambda sl: sl * lam,
     _ref_rad_lam,
     '2π rad/λ',
     'darkorange'),
]):
    ax.axhline(ref_val, color='grey', ls='--', lw=1.2, alpha=0.7, label=ref_lbl)

    no_cross_added = False
    for r in sr_sorted_noisy:
        s = r['sep_lam']
        if r['crossings']:
            for xz, sl in r['crossings']:
                val = scale_fn(sl)
                ax.plot(s, val, 'o', ms=9, color=col, zorder=5)
                ax.annotate(f'{val:.2f}',
                            xy=(s, val), xytext=(5, 5),
                            textcoords='offset points', fontsize=8)
        else:
            lbl = 'no crossing' if not no_cross_added else None
            no_cross_added = True
            ax.plot(s, 0, 'x', ms=11, color='tomato', mew=2, zorder=5, label=lbl)
            ax.annotate(f'{r["name"]}\n(no crossing)',
                        xy=(s, 0), xytext=(0, 14), textcoords='offset points',
                        ha='center', fontsize=7.5, color='tomato')

    ax.set_xscale('log')
    ax.set_xticks(sep_vals_noisy)
    ax.set_xticklabels(names_noisy, fontsize=9)
    ax.set_xlabel('Scatterer separation', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nSlope summary (NoisyTimeLapse, Kirchhoff):")
for r in sr_sorted_noisy:
    if r['crossings']:
        for xz, sl in r['crossings']:
            print(f"  {r['name']:8s}  ({r['sep_lam']:.5f}λ):  "
                  f"slope = {sl:+7.2f} rad/m  = {sl*lam:+.3f} rad/λ  @ x={xz:.4f} m")
    else:
        print(f"  {r['name']:8s}  ({r['sep_lam']:.5f}λ):  no zero-crossing detected")


## Section 7 — The "Spectral Line" Plot (Frequency vs. Phase Difference)

For each monitor scenario a single migrated column trace is extracted at two lateral positions:

| Row | x-position | Baseline trace | Monitor trace |
|---|---|---|---|
| **0** | x = x_sc0 (baseline pos) | scatterer **present** | scatterer **moved away** |
| **1** | x = x_sc_mon (monitor pos) | scatterer **absent** | scatterer **present** |

CLSSA ([`phase_decomposition.py`](phase_decomposition.py)) is applied to the trace that **contains the scatterer** in each row (baseline at x_sc0, monitor at x_sc_mon), giving:

- **Col 0** — amplitude wiggle trace (baseline=blue, monitor=red) vs TWT
- **Col 1** — CLSSA amplitude spectrum: Frequency [GHz] × TWT, colourscale = amplitude [dB]
- **Col 2** — CLSSA phase gather S'(θ, t): Phase [°] × TWT, colourscale = amplitude

In [ ]:
# # ─── Section 7: CLSSA Spectral Analysis ──────────────────────────────────────
# # 4 rows × 5 columns per scenario.
# # Rows 0–1: baseline antenna position (x_sc0)
# # Rows 2–3: monitor  antenna position (x_sc_mon)
# # Within each pair:
# #   Even row (0, 2): baseline survey — BEFORE scatterer moves
# #   Odd  row (1, 3): monitor  survey — AFTER  scatterer moves
# #
# # Col 0: wiggle trace (primary survey solid, other survey faint grey)
# # Col 1: CLSSA amplitude  A(f,τ)      [dB]
# # Col 2: CLSSA phase      φ(f,τ)      [°, raw amplitude-masked]
# # Col 3: CLSSA phase gather S'(θ,τ)  — carrier-detrended
# # Col 4 (even row): ΔΦ(f,τ) = φ_mon − φ_base  [2-D heatmap, joint-amp mask]
# # Col 4 (odd  row): ΔΦ(f) at τ_sc  +  theoretical −2πfΔt  [1-D slice]

# import sys as _sys
# _sys.path.insert(0, r'C:\Users\Administrator\OneDrive\Thesis\TimeLapse_Notebooks')
# from phase_decomposition import clssa_phase_decomposition

# dt_twt = float(2 * dz_mig / v_ice)
# twt_ns = (2 * z_img / v_ice).astype(float)
# twt_sc = float(2 * z_scatterer / v_ice)
# y_lo   = float(twt_ns[-1])
# y_hi   = float(twt_ns[0])

# # Dominant-phase log: (scenario, row label, theta_dom [deg]) — filled below,
# # printed once after all scenarios are processed so values can be compared.
# _dominant_theta_log  = []
# _ROW_LABELS_SHORT    = ['Baseline-Before', 'Baseline-After',
#                          'Monitor-Before',  'Monitor-After']

# for i in range(1, len(scenarios)):
#     base_img = gazdag[0]
#     mon_img  = gazdag[i]
#     name     = str(scenarios[i])
#     true_dx  = float(separation_lambda[i]) * lam
#     x_sc0    = float(x_s1[0])
#     x_sc_mon = float(x_s1[i])

#     ix0 = int(np.argmin(np.abs(x_traces - x_sc0)))
#     ixm = int(np.argmin(np.abs(x_traces - x_sc_mon)))

#     base_col0 = base_img[:, ix0].astype(float)
#     mon_col0  = mon_img[:,  ix0].astype(float)
#     base_colm = base_img[:, ixm].astype(float)
#     mon_colm  = mon_img[:,  ixm].astype(float)

#     # ── CLSSA for all four traces ─────────────────────────────────────────
#     _kw = dict(f_min_GHz=0.3, f_max_GHz=4.5, win_ns=2.0)
#     A_b0, ph_b0, pg_b0, freqs_c, t_ax, theta_ax = clssa_phase_decomposition(base_col0, dt_twt, **_kw)
#     A_m0, ph_m0, pg_m0, freqs_c, _,    theta_ax = clssa_phase_decomposition(mon_col0,  dt_twt, **_kw)
#     A_bm, ph_bm, pg_bm, freqs_c, _,    theta_ax = clssa_phase_decomposition(base_colm, dt_twt, **_kw)
#     A_mm, ph_mm, pg_mm, freqs_c, _,    theta_ax = clssa_phase_decomposition(mon_colm,  dt_twt, **_kw)

#     it_sc = int(np.argmin(np.abs(t_ax - twt_sc)))

#     # ── 1-D ΔΦ slices at τ_sc ────────────────────────────────────────────
#     def _dphi_1d(ph_mon, ph_base, it):
#         return np.rad2deg(np.angle(np.exp(1j * np.deg2rad(
#             ph_mon[:, it] - ph_base[:, it]))))

#     dphi_1d_0 = _dphi_1d(ph_m0, ph_b0, it_sc)
#     dphi_1d_m = _dphi_1d(ph_mm, ph_bm, it_sc)

#     # ── 2-D ΔΦ(f,τ) heatmaps, masked where joint amplitude < 5 % ─────────
#     def _dphi_2d(ph_mon, ph_base, A_mon, A_base, thr=0.05):
#         raw = np.rad2deg(np.angle(np.exp(1j * np.deg2rad(ph_mon - ph_base))))
#         A_j = np.minimum(A_mon, A_base)
#         return np.where(A_j / (A_j.max() + 1e-30) < thr, np.nan, raw)

#     dphi_2d_0 = _dphi_2d(ph_m0, ph_b0, A_m0, A_b0)
#     dphi_2d_m = _dphi_2d(ph_mm, ph_bm, A_mm, A_bm)

#     # ── Theoretical Δt [ns] per antenna position ──────────────────────────
#     _hyp   = float(np.sqrt(true_dx**2 + z_scatterer**2))
#     dt_th0 = 2.0 * (_hyp - z_scatterer) / v_ice   # x_sc0:    scatterer moved away
#     dt_thm = 2.0 * (z_scatterer - _hyp) / v_ice   # x_sc_mon: scatterer moved to

#     # ── Carrier-phase detrending for the phase gather ─────────────────────
#     _d_theta_pg = float(theta_ax[1] - theta_ax[0])
#     _n_theta_pg = len(theta_ax)
#     _shift_bins = (np.round(360.0 * f_c * t_ax / _d_theta_pg).astype(int)
#                    % _n_theta_pg)

#     def _detrend_gather(pg):
#         return np.stack(
#             [np.roll(pg[:, _it], -_shift_bins[_it]) for _it in range(pg.shape[1])],
#             axis=1
#         )

#     ext_tf = [freqs_c[0], freqs_c[-1], y_lo, y_hi]
#     ext_pg = [theta_ax[0], theta_ax[-1], y_lo, y_hi]

#     # ── Row specifications ────────────────────────────────────────────────
#     # Each tuple:
#     #   (title, trace_primary, trace_ref, A, ph, pg,
#     #    col4_type, dt_th, dphi_1d, dphi_2d, A_b_sc, A_m_sc)
#     _ROW_SPECS = [
#         (
#             f'Baseline pos  x={x_sc0:.3f} m  —  BEFORE  (baseline survey)',
#             base_col0, mon_col0,
#             A_b0, ph_b0, pg_b0,
#             '2d', dt_th0, dphi_1d_0, dphi_2d_0,
#             A_b0[:, it_sc], A_m0[:, it_sc],
#         ),
#         (
#             f'Baseline pos  x={x_sc0:.3f} m  —  AFTER   (monitor survey)',
#             mon_col0, base_col0,
#             A_m0, ph_m0, pg_m0,
#             '1d', dt_th0, dphi_1d_0, dphi_2d_0,
#             A_b0[:, it_sc], A_m0[:, it_sc],
#         ),
#         (
#             f'Monitor pos   x={x_sc_mon:.3f} m  —  BEFORE  (baseline survey)',
#             base_colm, mon_colm,
#             A_bm, ph_bm, pg_bm,
#             '2d', dt_thm, dphi_1d_m, dphi_2d_m,
#             A_bm[:, it_sc], A_mm[:, it_sc],
#         ),
#         (
#             f'Monitor pos   x={x_sc_mon:.3f} m  —  AFTER   (monitor survey)',
#             mon_colm, base_colm,
#             A_mm, ph_mm, pg_mm,
#             '1d', dt_thm, dphi_1d_m, dphi_2d_m,
#             A_bm[:, it_sc], A_mm[:, it_sc],
#         ),
#     ]
 
#     # ── Figure ─────────────────────────────────────────────────────────────
#     fig, axes = plt.subplots(4, 5, figsize=(28, 20))
#     fig.suptitle(
#         f'Spectral Line (CLSSA)  —  Gazdag  |  {name}'
#         f'   (Δx = {true_dx*1e3:.1f} mm = {true_dx/lam:.4f}λ)',
#         fontsize=12, fontweight='bold',
#     )
 
#     # Faint background: blue tint for baseline-pos rows, orange tint for monitor-pos rows
#     for _ri in range(4):
#         _bg = '#eef2ff' if _ri < 2 else '#fff3ee'
#         for _ci in range(5):
#             axes[_ri, _ci].set_facecolor(_bg)

#     for row_idx, (
#         row_title, trace_p, trace_r,
#         A_cl, ph_cl, pg_cl, col4_type, dt_th,
#         dphi_1d, dphi_2d, A_b_sc, A_m_sc,
#     ) in enumerate(_ROW_SPECS):

#         _clr_p = 'steelblue' if row_idx % 2 == 0 else 'tomato'

#         # ── Col 0: wiggle trace ─────────────────────────────────────────────
#         ax = axes[row_idx, 0]
#         ax.plot(trace_r, twt_ns, color='grey',  lw=0.8, alpha=0.28, label='other survey')
#         ax.plot(trace_p, twt_ns, color=_clr_p,  lw=1.5, label='this survey')
#         ax.fill_betweenx(twt_ns, trace_p, 0, where=(trace_p > 0),
#                          color=_clr_p, alpha=0.18)
#         ax.axhline(twt_sc, color='grey', ls='--', lw=0.9, alpha=0.7,
#                    label=f'z_sc ({twt_sc:.3f} ns)')
#         ax.axvline(0, color='k', lw=0.5, alpha=0.4)
#         ax.set_ylim(y_lo, y_hi)
#         ax.set_ylabel('TWT [ns]', fontsize=9)
#         ax.set_xlabel('Amplitude [a.u.]', fontsize=9)
#         ax.set_title(row_title, fontsize=8.5)
#         ax.legend(fontsize=7.5, loc='lower right')
#         ax.grid(True, alpha=0.25)

#         # ── Col 1: CLSSA amplitude spectrum ────────────────────────────────
#         ax = axes[row_idx, 1]
#         A_dB = np.clip(20 * np.log10(A_cl / (A_cl.max() + 1e-30) + 1e-30), -40, 0)
#         im1 = ax.imshow(A_dB.T, aspect='auto', origin='upper', extent=ext_tf,
#                         cmap='jet', vmin=-40, vmax=0, interpolation='bilinear')
#         ax.axhline(twt_sc, color='white', ls='--', lw=0.9, alpha=0.8,
#                    label=f'z_sc ({twt_sc:.3f} ns)')
#         ax.set_xlabel('Frequency [GHz]', fontsize=9)
#         ax.set_ylabel('TWT [ns]', fontsize=9)
#         ax.set_title('CLSSA Amplitude [dB]', fontsize=9)
#         ax.legend(fontsize=7.5)
#         fig.colorbar(im1, ax=ax, fraction=0.046, pad=0.04, label='[dB]')

#         # ── Col 2: CLSSA phase spectrum φ(f,τ) — raw, amplitude-masked ──────────────
#         ax = axes[row_idx, 2]
#         ph_msk = np.where(A_cl / (A_cl.max() + 1e-30) > 0.05, ph_cl, np.nan)
#         im2 = ax.imshow(ph_msk.T, aspect='auto', origin='upper', extent=ext_tf,
#                         cmap='hsv', vmin=-180, vmax=180, interpolation='bilinear')
#         ax.axhline(twt_sc, color='white', ls='--', lw=0.9, alpha=0.8,
#                    label=f'z_sc ({twt_sc:.3f} ns)')
#         ax.set_xlabel('Frequency [GHz]', fontsize=9)
#         ax.set_ylabel('TWT [ns]', fontsize=9)
#         ax.set_title(r'CLSSA Phase  $\phi(f,\tau)$  [°]', fontsize=9)
#         ax.legend(fontsize=7.5)
#         fig.colorbar(im2, ax=ax, fraction=0.046, pad=0.04, label='[°]')

#         # ── Col 3: CLSSA phase gather — carrier-detrended ──────────────────
#         ax = axes[row_idx, 3]
#         pg_dt   = _detrend_gather(pg_cl)
#         pg_clip = float(np.percentile(np.abs(pg_dt), 99)) or 1.0
#         im3 = ax.imshow(pg_dt.T, aspect='auto', origin='upper', extent=ext_pg,
#                         cmap='RdBu', vmin=-pg_clip, vmax=pg_clip,
#                         interpolation='bilinear')
#         ax.axhline(twt_sc, color='grey', ls='--', lw=0.9, alpha=0.7)
#         for ph_line in (-90, 90):
#             ax.axvline(ph_line, color='lime', lw=1.0, ls='--', alpha=0.85)
#         ax.axvline(0, color='white', lw=0.6, ls=':', alpha=0.5)

#         # Dominant phase: energy-weighted (integrated over all TWT) peak of
#         # the detrended gather, so it is not tied to a single TWT sample.
#         _theta_energy = np.nansum(np.abs(pg_dt), axis=1)   # (n_theta,)
#         _idx_dom      = int(np.argmax(_theta_energy))
#         theta_dom     = float(theta_ax[_idx_dom])
#         ax.axvline(theta_dom, color='black', lw=1.4, ls='-', alpha=0.9, zorder=5)
#         ax.text(0.02, 0.04, f'θ_dom = {theta_dom:.1f}°', transform=ax.transAxes,
#                 fontsize=8, color='black', ha='left', va='bottom',
#                 bbox=dict(boxstyle='round', fc='white', alpha=0.75, ec='none'))
#         _dominant_theta_log.append((name, _ROW_LABELS_SHORT[row_idx], theta_dom))

#         ax.set_xticks([-180, -90, 0, 90, 180])
#         ax.set_xlabel('Phase [°]', fontsize=9)
#         ax.set_ylabel('TWT [ns]', fontsize=9)
#         ax.set_title("Phase Gather S'(θ,t) — detrended", fontsize=9)
#         fig.colorbar(im3, ax=ax, fraction=0.046, pad=0.04, label='Amplitude')

#         # ── Col 4 ──────────────────────────────────────────────────────────
#         ax = axes[row_idx, 4]

#         if col4_type == '2d':
#             # 2-D ΔΦ(f,τ) heatmap: φ_mon − φ_base, masked by joint amplitude
#             im4 = ax.imshow(dphi_2d.T, aspect='auto', origin='upper',
#                             extent=ext_tf, cmap='hsv',
#                             vmin=-180, vmax=180, interpolation='bilinear')
#             ax.axhline(twt_sc, color='white', ls='--', lw=0.9, alpha=0.8,
#                        label=f'z_sc ({twt_sc:.3f} ns)')
#             ax.set_xlabel('Frequency [GHz]', fontsize=9)
#             ax.set_ylabel('TWT [ns]', fontsize=9)
#             ax.set_title(
#                 r'$\Delta\Phi(f,\tau) = \phi_\mathrm{mon} - \phi_\mathrm{base}$  [°]',
#                 fontsize=9,
#             )
#             ax.legend(fontsize=7.5)
#             fig.colorbar(im4, ax=ax, fraction=0.046, pad=0.04, label='[°]')

#         else:
#             # Group delay: Δt(f) = -(1/2π) ∂ΔΦ/∂f
#             # Unwrap the masked ΔΦ(f) (deg -> rad) to remove ±180° wraps, then
#             # differentiate wrt frequency. This recovers the frequency-dependent
#             # arrival-time shift directly, which should converge to the constant
#             # ray-theory Δt at high frequency and curve away from it at low
#             # frequency (geometric dispersion of the Gazdag migration operator).
#             A_joint  = np.minimum(A_b_sc, A_m_sc)
#             thr_amp  = 0.05 * A_joint.max() if A_joint.max() > 0 else np.inf
#             dphi_msk = np.where(A_joint >= thr_amp, dphi_1d, np.nan)

#             _valid = ~np.isnan(dphi_msk)
#             dt_f = np.full_like(freqs_c, np.nan)
#             if np.count_nonzero(_valid) > 2:
#                 f_v   = freqs_c[_valid]
#                 phi_v = np.unwrap(np.deg2rad(dphi_msk[_valid]))
#                 dt_f[_valid] = -1.0 / (2 * np.pi) * np.gradient(phi_v, f_v)

#             ax_amp = ax.twinx()
#             ax_amp.fill_between(freqs_c, A_b_sc, alpha=0.14, color='steelblue',
#                                 label='A_base(f)')
#             ax_amp.fill_between(freqs_c, A_m_sc, alpha=0.14, color='tomato',
#                                 label='A_mon(f)')
#             ax_amp.set_ylabel('Amplitude [a.u.]', fontsize=8, color='grey')
#             ax_amp.tick_params(axis='y', labelcolor='grey', labelsize=7)
#             ax_amp.set_ylim(bottom=0)
#             ax_amp.legend(fontsize=7, loc='upper left')

#             ax.plot(freqs_c, dt_f, color='darkorange', lw=2.2, zorder=3,
#                     label=r'$\Delta t(f)=-\frac{1}{2\pi}\partial\Delta\Phi/\partial f$')
#             ax.axhline(dt_th, color='limegreen', lw=1.8, ls='--', zorder=4,
#                        label=f'ray theory  (Δt={dt_th:.6f} ns)')
#             ax.axhline(0, color='k', lw=0.7, alpha=0.5, ls=':')

#             _valid_dt = dt_f[~np.isnan(dt_f)]
#             _dmax = float(np.max(np.abs(_valid_dt))) if len(_valid_dt) > 0 else abs(dt_th)
#             _dmax = max(_dmax, abs(dt_th), 1e-6)
#             _pad  = _dmax * 0.3
#             ax.set_ylim(-(_dmax + _pad), _dmax + _pad)

#             ax.set_xlabel('Frequency [GHz]', fontsize=9)
#             ax.set_ylabel('Δt(f)  [ns]', fontsize=9)
#             ax.set_title(
#                 f'Group Delay  Δt(f)   (z_sc={z_scatterer*1e3:.0f} mm)',
#                 fontsize=9,
#             )
#             ax.legend(fontsize=7.5, loc='upper right')
#             ax.grid(True, alpha=0.25, zorder=0)

#     plt.tight_layout()
#     plt.show()

# # ── Cross-scenario comparison of the dominant phase-gather angle ────────────
# print()
# print("Dominant phase angle theta_dom [deg]  (energy-weighted peak of the "
#       "detrended phase gather S'(theta,t), integrated over all TWT):")
# print(f'{"Scenario":<10}{"Row":<18}{"theta_dom [deg]":>16}')
# for _name, _row_lbl, _th in _dominant_theta_log:
#     print(f'{_name:<10}{_row_lbl:<18}{_th:>16.2f}')


In [ ]:
# # ─── Section 7b: CLSSA Spectral Analysis — Unmigrated (Raw) Data ─────────────
# # Same analysis as Section 7, applied to the raw background-subtracted
# # B-scans (data_static, Section 0) instead of the Gazdag-migrated images.
# # The raw trace is already recorded in two-way travel time, so no
# # depth->TWT conversion via v_ice is needed for the vertical axis.
# # 4 rows × 5 columns per scenario.
# # Rows 0–1: baseline antenna position (x_sc0)
# # Rows 2–3: monitor  antenna position (x_sc_mon)
# # Within each pair:
# #   Even row (0, 2): baseline survey — BEFORE scatterer moves
# #   Odd  row (1, 3): monitor  survey — AFTER  scatterer moves
# #
# # Col 0: wiggle trace (primary survey solid, other survey faint grey)
# # Col 1: CLSSA amplitude  A(f,τ)      [dB]
# # Col 2: CLSSA phase      φ(f,τ)      [°, raw amplitude-masked]
# # Col 3: CLSSA phase gather S'(θ,τ)  — carrier-detrended
# # Col 4 (even row): ΔΦ(f,τ) = φ_mon − φ_base  [2-D heatmap, joint-amp mask]
# # Col 4 (odd  row): ΔΦ(f) at τ_sc  +  theoretical −2πfΔt  [1-D slice]

# import sys as _sys
# _sys.path.insert(0, r'C:\Users\Administrator\OneDrive\Thesis\TimeLapse_Notebooks')
# from phase_decomposition import clssa_phase_decomposition

# dt_twt = float(dt * 1e9)          # raw sample interval [ns]
# twt_ns = time_ns.astype(float)    # raw data already in two-way travel time
# twt_sc = float(2 * z_scatterer / v_ice)
# y_lo   = float(twt_ns[-1])
# y_hi   = float(twt_ns[0])

# # Dominant-phase log: (scenario, row label, theta_dom [deg]) — filled below,
# # printed once after all scenarios are processed so values can be compared.
# _dominant_theta_log  = []
# _ROW_LABELS_SHORT    = ['Baseline-Before', 'Baseline-After',
#                          'Monitor-Before',  'Monitor-After']

# for i in range(1, len(scenarios)):
#     base_img = data_static[0]
#     mon_img  = data_static[i]
#     name     = str(scenarios[i])
#     true_dx  = float(separation_lambda[i]) * lam
#     x_sc0    = float(x_s1[0])
#     x_sc_mon = float(x_s1[i])

#     ix0 = int(np.argmin(np.abs(x_traces - x_sc0)))
#     ixm = int(np.argmin(np.abs(x_traces - x_sc_mon)))

#     base_col0 = base_img[:, ix0].astype(float)
#     mon_col0  = mon_img[:,  ix0].astype(float)
#     base_colm = base_img[:, ixm].astype(float)
#     mon_colm  = mon_img[:,  ixm].astype(float)

#     # ── CLSSA for all four traces ─────────────────────────────────────────
#     _kw = dict(f_min_GHz=0.3, f_max_GHz=4.5, win_ns=2.0)
#     A_b0, ph_b0, pg_b0, freqs_c, t_ax, theta_ax = clssa_phase_decomposition(base_col0, dt_twt, **_kw)
#     A_m0, ph_m0, pg_m0, freqs_c, _,    theta_ax = clssa_phase_decomposition(mon_col0,  dt_twt, **_kw)
#     A_bm, ph_bm, pg_bm, freqs_c, _,    theta_ax = clssa_phase_decomposition(base_colm, dt_twt, **_kw)
#     A_mm, ph_mm, pg_mm, freqs_c, _,    theta_ax = clssa_phase_decomposition(mon_colm,  dt_twt, **_kw)

#     it_sc = int(np.argmin(np.abs(t_ax - twt_sc)))

#     # ── 1-D ΔΦ slices at τ_sc ────────────────────────────────────────────
#     def _dphi_1d(ph_mon, ph_base, it):
#         return np.rad2deg(np.angle(np.exp(1j * np.deg2rad(
#             ph_mon[:, it] - ph_base[:, it]))))

#     dphi_1d_0 = _dphi_1d(ph_m0, ph_b0, it_sc)
#     dphi_1d_m = _dphi_1d(ph_mm, ph_bm, it_sc)

#     # ── 2-D ΔΦ(f,τ) heatmaps, masked where joint amplitude < 5 % ─────────
#     def _dphi_2d(ph_mon, ph_base, A_mon, A_base, thr=0.05):
#         raw = np.rad2deg(np.angle(np.exp(1j * np.deg2rad(ph_mon - ph_base))))
#         A_j = np.minimum(A_mon, A_base)
#         return np.where(A_j / (A_j.max() + 1e-30) < thr, np.nan, raw)

#     dphi_2d_0 = _dphi_2d(ph_m0, ph_b0, A_m0, A_b0)
#     dphi_2d_m = _dphi_2d(ph_mm, ph_bm, A_mm, A_bm)

#     # ── Theoretical Δt [ns] per antenna position ──────────────────────────
#     _hyp   = float(np.sqrt(true_dx**2 + z_scatterer**2))
#     dt_th0 = 2.0 * (_hyp - z_scatterer) / v_ice   # x_sc0:    scatterer moved away
#     dt_thm = 2.0 * (z_scatterer - _hyp) / v_ice   # x_sc_mon: scatterer moved to

#     # ── Carrier-phase detrending for the phase gather ─────────────────────
#     _d_theta_pg = float(theta_ax[1] - theta_ax[0])
#     _n_theta_pg = len(theta_ax)
#     _shift_bins = (np.round(360.0 * f_c * t_ax / _d_theta_pg).astype(int)
#                    % _n_theta_pg)

#     def _detrend_gather(pg):
#         return np.stack(
#             [np.roll(pg[:, _it], -_shift_bins[_it]) for _it in range(pg.shape[1])],
#             axis=1
#         )

#     ext_tf = [freqs_c[0], freqs_c[-1], y_lo, y_hi]
#     ext_pg = [theta_ax[0], theta_ax[-1], y_lo, y_hi]

#     # ── Row specifications ────────────────────────────────────────────────
#     # Each tuple:
#     #   (title, trace_primary, trace_ref, A, ph, pg,
#     #    col4_type, dt_th, dphi_1d, dphi_2d, A_b_sc, A_m_sc)
#     _ROW_SPECS = [
#         (
#             f'Baseline pos  x={x_sc0:.3f} m  —  BEFORE  (baseline survey)',
#             base_col0, mon_col0,
#             A_b0, ph_b0, pg_b0,
#             '2d', dt_th0, dphi_1d_0, dphi_2d_0,
#             A_b0[:, it_sc], A_m0[:, it_sc],
#         ),
#         (
#             f'Baseline pos  x={x_sc0:.3f} m  —  AFTER   (monitor survey)',
#             mon_col0, base_col0,
#             A_m0, ph_m0, pg_m0,
#             '1d', dt_th0, dphi_1d_0, dphi_2d_0,
#             A_b0[:, it_sc], A_m0[:, it_sc],
#         ),
#         (
#             f'Monitor pos   x={x_sc_mon:.3f} m  —  BEFORE  (baseline survey)',
#             base_colm, mon_colm,
#             A_bm, ph_bm, pg_bm,
#             '2d', dt_thm, dphi_1d_m, dphi_2d_m,
#             A_bm[:, it_sc], A_mm[:, it_sc],
#         ),
#         (
#             f'Monitor pos   x={x_sc_mon:.3f} m  —  AFTER   (monitor survey)',
#             mon_colm, base_colm,
#             A_mm, ph_mm, pg_mm,
#             '1d', dt_thm, dphi_1d_m, dphi_2d_m,
#             A_bm[:, it_sc], A_mm[:, it_sc],
#         ),
#     ]

#     # ── Figure ─────────────────────────────────────────────────────────────
#     fig, axes = plt.subplots(4, 5, figsize=(28, 20))
#     fig.suptitle(
#         f'Spectral Line (CLSSA)  —  Raw (Unmigrated)  |  {name}'
#         f'   (Δx = {true_dx*1e3:.1f} mm = {true_dx/lam:.4f}λ)',
#         fontsize=12, fontweight='bold',
#     )

#     # Faint background: blue tint for baseline-pos rows, orange tint for monitor-pos rows
#     for _ri in range(4):
#         _bg = '#eef2ff' if _ri < 2 else '#fff3ee'
#         for _ci in range(5):
#             axes[_ri, _ci].set_facecolor(_bg)

#     for row_idx, (
#         row_title, trace_p, trace_r,
#         A_cl, ph_cl, pg_cl, col4_type, dt_th,
#         dphi_1d, dphi_2d, A_b_sc, A_m_sc,
#     ) in enumerate(_ROW_SPECS):

#         _clr_p = 'steelblue' if row_idx % 2 == 0 else 'tomato'

#         # ── Col 0: wiggle trace ─────────────────────────────────────────────
#         ax = axes[row_idx, 0]
#         ax.plot(trace_r, twt_ns, color='grey',  lw=0.8, alpha=0.28, label='other survey')
#         ax.plot(trace_p, twt_ns, color=_clr_p,  lw=1.5, label='this survey')
#         ax.fill_betweenx(twt_ns, trace_p, 0, where=(trace_p > 0),
#                          color=_clr_p, alpha=0.18)
#         ax.axhline(twt_sc, color='grey', ls='--', lw=0.9, alpha=0.7,
#                    label=f'z_sc ({twt_sc:.3f} ns)')
#         ax.axvline(0, color='k', lw=0.5, alpha=0.4)
#         ax.set_ylim(y_lo, y_hi)
#         ax.set_ylabel('TWT [ns]', fontsize=9)
#         ax.set_xlabel('Amplitude [a.u.]', fontsize=9)
#         ax.set_title(row_title, fontsize=8.5)
#         ax.legend(fontsize=7.5, loc='lower right')
#         ax.grid(True, alpha=0.25)

#         # ── Col 1: CLSSA amplitude spectrum ────────────────────────────────
#         ax = axes[row_idx, 1]
#         A_dB = np.clip(20 * np.log10(A_cl / (A_cl.max() + 1e-30) + 1e-30), -40, 0)
#         im1 = ax.imshow(A_dB.T, aspect='auto', origin='upper', extent=ext_tf,
#                         cmap='jet', vmin=-40, vmax=0, interpolation='bilinear')
#         ax.axhline(twt_sc, color='white', ls='--', lw=0.9, alpha=0.8,
#                    label=f'z_sc ({twt_sc:.3f} ns)')
#         ax.set_xlabel('Frequency [GHz]', fontsize=9)
#         ax.set_ylabel('TWT [ns]', fontsize=9)
#         ax.set_title('CLSSA Amplitude [dB]', fontsize=9)
#         ax.legend(fontsize=7.5)
#         fig.colorbar(im1, ax=ax, fraction=0.046, pad=0.04, label='[dB]')

#         # ── Col 2: CLSSA phase spectrum φ(f,τ) — raw, amplitude-masked ──────────────
#         ax = axes[row_idx, 2]
#         ph_msk = np.where(A_cl / (A_cl.max() + 1e-30) > 0.05, ph_cl, np.nan)
#         im2 = ax.imshow(ph_msk.T, aspect='auto', origin='upper', extent=ext_tf,
#                         cmap='hsv', vmin=-180, vmax=180, interpolation='bilinear')
#         ax.axhline(twt_sc, color='white', ls='--', lw=0.9, alpha=0.8,
#                    label=f'z_sc ({twt_sc:.3f} ns)')
#         ax.set_xlabel('Frequency [GHz]', fontsize=9)
#         ax.set_ylabel('TWT [ns]', fontsize=9)
#         ax.set_title(r'CLSSA Phase  $\phi(f,\tau)$  [°]', fontsize=9)
#         ax.legend(fontsize=7.5)
#         fig.colorbar(im2, ax=ax, fraction=0.046, pad=0.04, label='[°]')

#         # ── Col 3: CLSSA phase gather — carrier-detrended ──────────────────
#         ax = axes[row_idx, 3]
#         pg_dt   = _detrend_gather(pg_cl)
#         pg_clip = float(np.percentile(np.abs(pg_dt), 99)) or 1.0
#         im3 = ax.imshow(pg_dt.T, aspect='auto', origin='upper', extent=ext_pg,
#                         cmap='RdBu', vmin=-pg_clip, vmax=pg_clip,
#                         interpolation='bilinear')
#         ax.axhline(twt_sc, color='grey', ls='--', lw=0.9, alpha=0.7)
#         for ph_line in (-90, 90):
#             ax.axvline(ph_line, color='lime', lw=1.0, ls='--', alpha=0.85)
#         ax.axvline(0, color='white', lw=0.6, ls=':', alpha=0.5)

#         # Dominant phase: energy-weighted (integrated over all TWT) peak of
#         # the detrended gather, so it is not tied to a single TWT sample.
#         _theta_energy = np.nansum(np.abs(pg_dt), axis=1)   # (n_theta,)
#         _idx_dom      = int(np.argmax(_theta_energy))
#         theta_dom     = float(theta_ax[_idx_dom])
#         ax.axvline(theta_dom, color='black', lw=1.4, ls='-', alpha=0.9, zorder=5)
#         ax.text(0.02, 0.04, f'θ_dom = {theta_dom:.1f}°', transform=ax.transAxes,
#                 fontsize=8, color='black', ha='left', va='bottom',
#                 bbox=dict(boxstyle='round', fc='white', alpha=0.75, ec='none'))
#         _dominant_theta_log.append((name, _ROW_LABELS_SHORT[row_idx], theta_dom))

#         ax.set_xticks([-180, -90, 0, 90, 180])
#         ax.set_xlabel('Phase [°]', fontsize=9)
#         ax.set_ylabel('TWT [ns]', fontsize=9)
#         ax.set_title("Phase Gather S'(θ,t) — detrended", fontsize=9)
#         fig.colorbar(im3, ax=ax, fraction=0.046, pad=0.04, label='Amplitude')

#         # ── Col 4 ──────────────────────────────────────────────────────────
#         ax = axes[row_idx, 4]

#         if col4_type == '2d':
#             # 2-D ΔΦ(f,τ) heatmap: φ_mon − φ_base, masked by joint amplitude
#             im4 = ax.imshow(dphi_2d.T, aspect='auto', origin='upper',
#                             extent=ext_tf, cmap='hsv',
#                             vmin=-180, vmax=180, interpolation='bilinear')
#             ax.axhline(twt_sc, color='white', ls='--', lw=0.9, alpha=0.8,
#                        label=f'z_sc ({twt_sc:.3f} ns)')
#             ax.set_xlabel('Frequency [GHz]', fontsize=9)
#             ax.set_ylabel('TWT [ns]', fontsize=9)
#             ax.set_title(
#                 r'$\Delta\Phi(f,\tau) = \phi_\mathrm{mon} - \phi_\mathrm{base}$  [°]',
#                 fontsize=9,
#             )
#             ax.legend(fontsize=7.5)
#             fig.colorbar(im4, ax=ax, fraction=0.046, pad=0.04, label='[°]')

#         else:
#             # Group delay: Δt(f) = -(1/2π) ∂ΔΦ/∂f
#             # Unwrap the masked ΔΦ(f) (deg -> rad) to remove ±180° wraps, then
#             # differentiate wrt frequency. This recovers the frequency-dependent
#             # arrival-time shift directly, which should converge to the constant
#             # ray-theory Δt at high frequency and curve away from it at low
#             # frequency (wavelet/antenna dispersion in the raw recorded signal).
#             A_joint  = np.minimum(A_b_sc, A_m_sc)
#             thr_amp  = 0.05 * A_joint.max() if A_joint.max() > 0 else np.inf
#             dphi_msk = np.where(A_joint >= thr_amp, dphi_1d, np.nan)

#             _valid = ~np.isnan(dphi_msk)
#             dt_f = np.full_like(freqs_c, np.nan)
#             if np.count_nonzero(_valid) > 2:
#                 f_v   = freqs_c[_valid]
#                 phi_v = np.unwrap(np.deg2rad(dphi_msk[_valid]))
#                 dt_f[_valid] = -1.0 / (2 * np.pi) * np.gradient(phi_v, f_v)

#             ax_amp = ax.twinx()
#             ax_amp.fill_between(freqs_c, A_b_sc, alpha=0.14, color='steelblue',
#                                 label='A_base(f)')
#             ax_amp.fill_between(freqs_c, A_m_sc, alpha=0.14, color='tomato',
#                                 label='A_mon(f)')
#             ax_amp.set_ylabel('Amplitude [a.u.]', fontsize=8, color='grey')
#             ax_amp.tick_params(axis='y', labelcolor='grey', labelsize=7)
#             ax_amp.set_ylim(bottom=0)
#             ax_amp.legend(fontsize=7, loc='upper left')

#             ax.plot(freqs_c, dt_f, color='darkorange', lw=2.2, zorder=3,
#                     label=r'$\Delta t(f)=-\frac{1}{2\pi}\partial\Delta\Phi/\partial f$')
#             ax.axhline(dt_th, color='limegreen', lw=1.8, ls='--', zorder=4,
#                        label=f'ray theory  (Δt={dt_th:.6f} ns)')
#             ax.axhline(0, color='k', lw=0.7, alpha=0.5, ls=':')

#             _valid_dt = dt_f[~np.isnan(dt_f)]
#             _dmax = float(np.max(np.abs(_valid_dt))) if len(_valid_dt) > 0 else abs(dt_th)
#             _dmax = max(_dmax, abs(dt_th), 1e-6)
#             _pad  = _dmax * 0.3
#             ax.set_ylim(-(_dmax + _pad), _dmax + _pad)

#             ax.set_xlabel('Frequency [GHz]', fontsize=9)
#             ax.set_ylabel('Δt(f)  [ns]', fontsize=9)
#             ax.set_title(
#                 f'Group Delay  Δt(f)   (z_sc={z_scatterer*1e3:.0f} mm)',
#                 fontsize=9,
#             )
#             ax.legend(fontsize=7.5, loc='upper right')
#             ax.grid(True, alpha=0.25, zorder=0)

#     plt.tight_layout()
#     plt.show()

# # ── Cross-scenario comparison of the dominant phase-gather angle ────────────
# print()
# print("Dominant phase angle theta_dom [deg]  (energy-weighted peak of the "
#       "detrended phase gather S'(theta,t), integrated over all TWT):")
# print(f'{"Scenario":<10}{"Row":<18}{"theta_dom [deg]":>16}')
# for _name, _row_lbl, _th in _dominant_theta_log:
#     print(f'{_name:<10}{_row_lbl:<18}{_th:>16.2f}')


## Section 8 — Cross-Phase Spectrogram (TWT vs. Frequency)

For the central trace (x = x_sc0, the scatterer's baseline position — the fixed
reference column shared by every monitor scenario), use CLSSA (the same narrowband
decomposition as Section 7) to get the baseline and monitor phase spectra
$\phi_\text{base}(f,\tau)$ and $\phi_\text{mon}(f,\tau)$, and form the cross-phase

$$\Delta\Phi(\tau, f) = \angle\Big[\exp\big(i\,(\phi_\text{base} - \phi_\text{mon})\big)\Big]
= \angle\,XS(\tau,f), \qquad XS = S_\text{base}\cdot S_\text{mon}^{*}$$

without ever forming the complex STFT explicitly — CLSSA samples frequency directly, so
unlike a generic STFT it isn't limited to a handful of coarse FFT bins regardless of
window length.

Plotting $\Delta\Phi(\tau,f)$ as a 2-D heatmap (frequency on the x-axis, TWT on the
y-axis, phase in $[-\pi, +\pi]$ on the colorbar) tests whether the measured phase shift
is tied to the actual target rather than to background noise:

- **Outside** the target's reflection time, the background should look like chaotic,
  pixelated salt-and-pepper noise — randomly wrapped phase from incoherent background
  scattering.
- **Exactly at** the TWT of the scatterer, a clean, coherent window of structured phase
  should appear across the antenna's frequency band.
- If the target **moved**, that window shows a smooth vertical colour gradient
  (a fringe pattern) — the phase ramps with frequency, consistent with the $-2\pi f\Delta t$
  Fourier-shift theorem.
- If a diffuse change occurred instead (e.g. fluid filling a pore space), the window would
  instead show a solid, near-uniform block of a single colour.

For separations **below ¼λ** the phase change at the scatterer is very subtle, so those
scenarios additionally zoom the TWT axis in on a window around the scatterer depth and
clip the colorbar to a much smaller range, so the subtle contrast is still visible.


In [ ]:
# # ─── Section 8: Cross-Phase Spectrogram (TWT vs. Frequency) ──────────────────
# # For the central trace (x = x_sc0, the scatterer's baseline position — the
# # fixed reference column shared by every monitor scenario), use CLSSA (the
# # same narrowband decomposition as Section 7) to get the baseline and monitor
# # phase spectra φ_base(f,τ) and φ_mon(f,τ), then form the cross-phase
# #     ΔΦ(τ,f) = angle[ exp(i·(φ_base − φ_mon)) ]   (radians, wrapped to ±π)
# # which is the phase of the complex cross-spectrum XS = S_base · S_mon* without
# # ever forming S explicitly. CLSSA samples frequency directly (no FFT
# # bin-spacing limit like the earlier STFT version had), so the frequency axis
# # is smooth regardless of analysis window length.
# #
# # No amplitude masking — the whole point is to see the noise-vs-coherence
# # contrast across the full image, not just the signal-bearing band.
# #
# # For separations below 1/4 lambda the phase change at the scatterer is very
# # subtle, so those scenarios additionally:
# #   - zoom the TWT axis in on a window around the scatterer depth
# #   - clip the colorbar to a much smaller range, to make the subtle phase
# #     contrast visible

# import sys as _sys
# _sys.path.insert(0, r'C:\Users\Administrator\OneDrive\Thesis\TimeLapse_Notebooks')
# from phase_decomposition import clssa_phase_decomposition

# dt_twt = float(2 * dz_mig / v_ice)
# twt_sc = float(2 * z_scatterer / v_ice)

# _clssa_kw = dict(f_min_GHz=0.3, f_max_GHz=4.5, win_ns=2.0)

# _SMALL_SEP_THRESH   = 0.25       # lambda — scenarios below this get zoom + clip
# _ZOOM_HALF_WIDTH_NS = 1.5        # ns around twt_sc to zoom in on
# _CLIP_SMALL         = np.pi / 10  # rad (~30°) colorbar clip for small-sep cases
# _CLIP_FULL          = np.pi      # rad (180°) colorbar clip otherwise

# for i in range(1, len(scenarios)):
#     base_img = gazdag[0]
#     mon_img  = gazdag[i]
#     name     = str(scenarios[i])
#     true_dx  = float(separation_lambda[i]) * lam
#     x_sc0    = float(x_s1[0])

#     ix0 = int(np.argmin(np.abs(x_traces - x_sc0)))
#     base_trace = base_img[:, ix0].astype(float)
#     mon_trace  = mon_img[:,  ix0].astype(float)

#     A_b, ph_b, _, freqs_c, t_ax, _ = clssa_phase_decomposition(base_trace, dt_twt, **_clssa_kw)
#     A_m, ph_m, _, freqs_c, _,    _ = clssa_phase_decomposition(mon_trace,  dt_twt, **_clssa_kw)

#     # Cross-phase ΔΦ(f,τ) = φ_base − φ_mon, wrapped to (−π, +π]
#     phi_xs = np.angle(np.exp(1j * np.deg2rad(ph_b - ph_m)))   # (n_f, n_t) radians

#     is_small_sep = float(separation_lambda[i]) < _SMALL_SEP_THRESH
#     clip = _CLIP_SMALL if is_small_sep else _CLIP_FULL

#     ext = [freqs_c[0], freqs_c[-1], t_ax[-1], t_ax[0]]

#     fig, ax = plt.subplots(figsize=(8, 6))
#     im = ax.imshow(phi_xs.T, aspect='auto', origin='upper', extent=ext,
#                    cmap='hsv', vmin=-clip, vmax=clip, interpolation='nearest')
#     ax.axhline(twt_sc, color='white', ls='--', lw=1.1, alpha=0.85,
#                label=f'τ_sc ({twt_sc:.3f} ns)')

#     _zoom_note = ''
#     if is_small_sep:
#         ax.set_ylim(twt_sc + _ZOOM_HALF_WIDTH_NS, twt_sc - _ZOOM_HALF_WIDTH_NS)
#         _zoom_note = f'  [zoomed ±{_ZOOM_HALF_WIDTH_NS:.1f} ns, clip=±{np.degrees(clip):.0f}°]'

#     ax.set_xlabel('Frequency [GHz]', fontsize=10)
#     ax.set_ylabel('TWT [ns]', fontsize=10)
#     ax.set_title(
#         f'Cross-Phase Spectrogram (CLSSA)  ΔΦ(τ,f)  —  Gazdag  |  {name}\n'
#         f'x = {x_sc0:.3f} m   (Δx = {true_dx*1e3:.1f} mm = {true_dx/lam:.4f}λ){_zoom_note}',
#         fontsize=9.5, fontweight='bold'
#     )
#     ax.legend(fontsize=8)
#     cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='ΔΦ  [rad]')
#     if is_small_sep:
#         cbar.set_ticks([-clip, -clip / 2, 0, clip / 2, clip])
#         cbar.set_ticklabels([f'{np.degrees(-clip):.0f}°', f'{np.degrees(-clip / 2):.0f}°',
#                               '0', f'{np.degrees(clip / 2):.0f}°', f'{np.degrees(clip):.0f}°'])
#     else:
#         cbar.set_ticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi])
#         cbar.set_ticklabels(['-π', '-π/2', '0', 'π/2', 'π'])

#     plt.tight_layout()
#     plt.show()


## Section 9 — Localized Fourier Shift (STFT Phase Decomposition)

### 1. The Mathematical Bridge

Let $s_1(t)$ be the extracted 1-D baseline trace, where $t$ is Two-Way Time (TWT).
Applying a sliding Gaussian window $w(t)$ gives the **Gabor / Short-Time Fourier Transform**:

$$S_1(\tau, \omega) = \int_{-\infty}^{\infty} s_1(t)\, w(t - \tau)\, e^{-j\omega t}\, dt$$

This complex matrix decomposes into:
- **Amplitude** $A_1(\tau, f) = |S_1(\tau, f)|$
- **Phase** $\phi_1(\tau, f) = \angle S_1(\tau, f)$

A physical sub-wavelength shift $\Delta t$ (mechanical) plus a material rotation $\Delta\theta$ transform the monitor trace $s_2(t) = s_1(t - \Delta t)\, e^{j\Delta\theta}$ into:

$$S_2(\tau, \omega) \approx S_1(\tau, \omega)\cdot e^{-j\omega \Delta t}\cdot e^{j\Delta\theta} \qquad \text{(Localized Fourier Shift Theorem, valid when } \Delta t \ll \sigma_w\text{)}$$

### 2. The Fundamental Phase-Difference Equation

The **local cross-spectrum** $\text{XS}(\tau,\omega) = S_2 \cdot S_1^*$ cancels baseline amplitude and phase, leaving:

$$\boxed{\Delta\Phi(\tau, f) = \angle\,\text{XS}(\tau, f) \approx -2\pi f\,\Delta t + \Delta\theta}$$

At the scatterer depth $\tau_0$, plotting $\Delta\Phi$ versus $f$ is a **straight line** whose slope $= -2\pi\Delta t$ and intercept $= \Delta\theta$.

### 3. The Three Panels

| Panel | Axes | Colour | Physical meaning |
|---|---|---|---|
| **A** $A(\tau,f)$ | TWT × Frequency | dB amplitude | Where pulse energy lives; equivalent to the WLS weight matrix |
| **B** $\Delta\Phi(\tau,f)$ | TWT × Frequency | Phase [°] | Full 2-D view of shift; slice at $\tau_0$ gives the spectral line |
| **C** $A(\tau,\theta)$ | TWT × Phase angle | Amplitude | Phase "hotspot" shifts by $\Delta\theta$ between baseline and monitor |

In [ ]:
# # ─── Section 9: Localized Fourier Shift — STFT Phase Decomposition ───────────
# # Implements the Gabor/STFT framework described in the section header.
# # The Gaussian-windowed STFT S(τ,ω) decomposes each column trace into a
# # time-frequency plane; the cross-spectrum XS = S_mon · S_base* then gives the
# # full 2-D phase-difference surface ΔΦ(τ,f) without reducing to a single depth.
# #
# # Four panels per row (2 rows = baseline position / monitor position):
# #   Col 0  A(τ,f)         — amplitude [dB]  of the signal-bearing trace
# #   Col 1  ΔΦ(τ,f)        — XS phase [°],  τ_sc marked with white dashed line
# #   Col 2  A(τ,θ)         — amplitude-weighted phase gather
# #   Col 3  ΔΦ(τ_sc,f)     — 1-D spectral line at the scatterer depth
# #                             + theoretical −2πfΔt overlay

# from scipy.signal import stft as _stft
# from scipy.signal.windows import gaussian as _gauss_win

# # Window: 2 periods of the centre frequency (rounded up to the next even int)
# _n_win  = int(np.ceil(1.0 / (f_c * dt_twt)))
# _n_win += _n_win % 2
# _gauss  = _gauss_win(_n_win, std=_n_win / 6.0)   # σ ≈ 1/3 of window width
# _fs_ns  = 1.0 / dt_twt                            # sampling rate [GHz = 1/ns]

# _n_theta  = 360
# _theta_pg = np.linspace(-180, 180, _n_theta + 1)          # bin edges [°]
# _theta_cx = 0.5 * (_theta_pg[:-1] + _theta_pg[1:])        # bin centres

# def _do_stft(trace):
#     """Gaussian-windowed STFT. Returns f [GHz], t [ns], Z (n_f × n_t) complex."""
#     f_ax, t_ax, Z = _stft(
#         trace, fs=_fs_ns, window=_gauss,
#         nperseg=_n_win, noverlap=_n_win - 1,
#         return_onesided=True
#     )
#     return f_ax, t_ax, Z

# def _amp_phase_gather(Z):
#     """A(τ,θ): amplitude-weighted histogram over instantaneous phase.
#     Returns array of shape (n_theta, n_t)."""
#     A_tf  = np.abs(Z)
#     ph_tf = np.rad2deg(np.angle(Z))
#     n_t   = Z.shape[1]
#     PG    = np.zeros((_n_theta, n_t))
#     for it in range(n_t):
#         PG[:, it], _ = np.histogram(
#             ph_tf[:, it], bins=_theta_pg, weights=A_tf[:, it]
#         )
#     return PG

# for i in range(1, len(scenarios)):
#     base_img = gazdag[0]
#     mon_img  = gazdag[i]
#     name     = str(scenarios[i])
#     true_dx  = float(separation_lambda[i]) * lam
#     x_sc0    = float(x_s1[0])
#     x_sc_mon = float(x_s1[i])

#     ix0 = int(np.argmin(np.abs(x_traces - x_sc0)))
#     ixm = int(np.argmin(np.abs(x_traces - x_sc_mon)))

#     # STFT for all four column traces
#     f_stft, t_stft, Z_b0 = _do_stft(base_img[:, ix0].astype(float))
#     _,       _,     Z_m0 = _do_stft(mon_img[:,  ix0].astype(float))
#     _,       _,     Z_bm = _do_stft(base_img[:, ixm].astype(float))
#     _,       _,     Z_mm = _do_stft(mon_img[:,  ixm].astype(float))

#     # Restrict frequency axis to GPR bandwidth
#     f_msk  = (f_stft >= 0.3) & (f_stft <= 4.5)
#     f_plot = f_stft[f_msk]

#     # STFT time index nearest the scatterer depth
#     it_sc_s = int(np.argmin(np.abs(t_stft - twt_sc)))

#     # Theoretical Δt [ns] per row (same geometry as Section 7)
#     _hyp   = float(np.sqrt(true_dx**2 + z_scatterer**2))
#     dt_row = [
#         2.0 * (_hyp - z_scatterer) / v_ice,    # Row 0: Δt > 0
#         2.0 * (z_scatterer - _hyp)  / v_ice,   # Row 1: Δt < 0
#     ]

#     # imshow extents: frequency on x, TWT on y (increasing downward)
#     ext_tf = [f_plot[0], f_plot[-1], t_stft[-1], t_stft[0]]
#     ext_pg = [_theta_cx[0], _theta_cx[-1], t_stft[-1], t_stft[0]]

#     fig, axes = plt.subplots(2, 4, figsize=(22, 10))
#     fig.suptitle(
#         f'Localized Fourier Shift (STFT, Gaussian win={_n_win} smp)  —  Gazdag  |  {name}'
#         f'   (Δx = {true_dx*1e3:.1f} mm = {true_dx/lam:.4f}λ)',
#         fontsize=11, fontweight='bold'
#     )

#     for row_idx, (row_label, Z_sig, Z_b, Z_m, dt_th) in enumerate([
#         (f'Baseline pos  x = {x_sc0:.3f} m  (scatterer moved away in monitor)',
#          Z_b0, Z_b0, Z_m0, dt_row[0]),
#         (f'Monitor pos   x = {x_sc_mon:.3f} m  (scatterer moved to in monitor)',
#          Z_mm, Z_bm, Z_mm, dt_row[1]),
#     ]):
#         # Cross-spectrum: ΔΦ(τ,f) = angle[ S_mon · S_base* ]
#         XS      = Z_m * np.conj(Z_b)
#         A_joint = np.minimum(np.abs(Z_b), np.abs(Z_m))
#         dphi_2d = np.where(
#             A_joint > 0.05 * A_joint.max(),
#             np.rad2deg(np.angle(XS)), np.nan
#         )   # (n_f, n_t)

#         # ── Col 0: A(τ,f) amplitude heatmap ────────────────────────────────
#         ax   = axes[row_idx, 0]
#         A_tf = np.abs(Z_sig)[f_msk, :]        # (n_f_plot, n_t)
#         A_dB = np.clip(20*np.log10(A_tf / (A_tf.max()+1e-30) + 1e-30), -40, 0)
#         im0  = ax.imshow(A_dB.T, aspect='auto', origin='upper', extent=ext_tf,
#                          cmap='jet', vmin=-40, vmax=0, interpolation='bilinear')
#         ax.axhline(twt_sc, color='white', ls='--', lw=0.9, alpha=0.8,
#                    label=f'τ_sc ({twt_sc:.3f} ns)')
#         ax.set_xlabel('Frequency [GHz]', fontsize=9)
#         ax.set_ylabel('TWT [ns]', fontsize=9)
#         ax.set_title(f'A(τ,f)  —  {row_label}', fontsize=8.2)
#         ax.legend(fontsize=7.5)
#         fig.colorbar(im0, ax=ax, fraction=0.046, pad=0.04, label='[dB]')

#         # ── Col 1: ΔΦ(τ,f) heatmap ─────────────────────────────────────────
#         ax      = axes[row_idx, 1]
#         dp_plot = dphi_2d[f_msk, :]           # (n_f_plot, n_t)
#         im1     = ax.imshow(dp_plot.T, aspect='auto', origin='upper', extent=ext_tf,
#                             cmap='RdBu', vmin=-180, vmax=180, interpolation='bilinear')
#         ax.axhline(twt_sc, color='white', ls='--', lw=0.9, alpha=0.9,
#                    label=f'τ_sc ({twt_sc:.3f} ns)')
#         ax.set_xlabel('Frequency [GHz]', fontsize=9)
#         ax.set_ylabel('TWT [ns]', fontsize=9)
#         ax.set_title(r'$\Delta\Phi(\tau,f)$  [°]  — XS phase', fontsize=9)
#         ax.legend(fontsize=7.5)
#         fig.colorbar(im1, ax=ax, fraction=0.046, pad=0.04, label='[°]')

#         # ── Col 2: A(τ,θ) phase gather ─────────────────────────────────────
#         ax     = axes[row_idx, 2]
#         PG     = _amp_phase_gather(Z_sig)      # (n_theta, n_t)
#         pg_max = float(np.percentile(PG, 99)) or 1.0
#         im2    = ax.imshow(PG.T, aspect='auto', origin='upper', extent=ext_pg,
#                            cmap='inferno', vmin=0, vmax=pg_max,
#                            interpolation='bilinear')
#         ax.axhline(twt_sc, color='white', ls='--', lw=0.9, alpha=0.8)
#         for ref in (-90, 0, 90):
#             ax.axvline(ref, color='grey', lw=0.5, ls=':', alpha=0.5)
#         ax.set_xticks([-180, -90, 0, 90, 180])
#         ax.set_xlabel('Phase θ [°]', fontsize=9)
#         ax.set_ylabel('TWT [ns]', fontsize=9)
#         ax.set_title('A(τ,θ)  phase gather', fontsize=9)
#         fig.colorbar(im2, ax=ax, fraction=0.046, pad=0.04, label='Amplitude')

#         # ── Col 3: ΔΦ(τ_sc, f) spectral line + theoretical ─────────────────
#         ax         = axes[row_idx, 3]
#         dphi_slice = dp_plot[:, it_sc_s]       # 1-D slice at τ_sc
#         dphi_th    = np.rad2deg(np.angle(
#             np.exp(-1j * 2 * np.pi * f_plot * dt_th)  # f[GHz] × Δt[ns] = cycles
#         ))

#         A_slice = A_tf[:, it_sc_s]
#         ax_amp  = ax.twinx()
#         ax_amp.fill_between(f_plot, A_slice, alpha=0.15, color='steelblue',
#                             label='A(τ_sc, f)')
#         ax_amp.set_ylim(bottom=0)
#         ax_amp.set_ylabel('Amplitude [a.u.]', fontsize=8, color='grey')
#         ax_amp.tick_params(axis='y', labelcolor='grey', labelsize=7)
#         ax_amp.legend(fontsize=7, loc='upper left')

#         ax.plot(f_plot, dphi_slice, color='darkorange', lw=2.0, zorder=3,
#                 label='ΔΦ(τ_sc, f)  [STFT]')
#         ax.plot(f_plot, dphi_th,    color='limegreen',  lw=1.8, ls='--', zorder=4,
#                 label=rf'$-2\pi f\,\Delta t$  (Δt = {dt_th:.5f} ns)')
#         ax.axhline(0, color='k', lw=0.5, ls=':', alpha=0.5)

#         _valid = dphi_slice[~np.isnan(dphi_slice)]
#         _dmax  = float(np.abs(_valid).max()) if len(_valid) > 0 else 180.0
#         _dmax  = max(_dmax, float(np.abs(dphi_th).max()))
#         _pad   = max(_dmax * 0.25, 10.0)
#         _ylo   = max(-195.0, -(_dmax + _pad))
#         _yhi   = min( 195.0,   _dmax + _pad)
#         ax.set_ylim(_ylo, _yhi)
#         ax.set_yticks([t for t in (-180, -90, 0, 90, 180) if _ylo <= t <= _yhi])

#         ax.set_xlabel('Frequency [GHz]', fontsize=9)
#         ax.set_ylabel('ΔΦ  [°]', fontsize=9)
#         ax.set_title(
#             f'ΔΦ(τ = {t_stft[it_sc_s]:.3f} ns, f)  [STFT]',
#             fontsize=9
#         )
#         ax.legend(fontsize=7.5, loc='upper right')
#         ax.grid(True, alpha=0.25, zorder=0)

#     plt.tight_layout()
#     plt.show()